# GLM-5.3-Flash on a free Kaggle TPU v5e-8

This notebook serves **GLM-5.3-Flash**, a 320B-parameter mixture-of-experts model (18B active), on Kaggle's free
TPU and gives you a public endpoint that speaks both the OpenAI and the Anthropic API. Use it from your laptop,
Claude Code, Codex CLI, opencode, or plain `curl`.

As far as we know this is the first inference engine to run GLM-5.3-Flash on a TPU, and it runs on the free one.
We wrote it in JAX for this model: the routed experts sit in HBM as 3-bit codebook weights and custom TPU kernels read
them straight into the matrix multiply. Measured on this setup: about **64 tok/s** for one stream, **~90 tok/s** aggregate over three streams, **~1,600 tok/s** prefill, the
model's full **262,144-token context**, prefix caching that makes agent sessions resume in about a second, and
image inputs.

## Before you run — three clicks in the right sidebar
1. **Accelerator → TPU VM v5e-8** (Session options)
2. **Internet → ON** (Session options; needed for the tunnel)
3. **Add Input** → search and attach these three datasets:
   - `rahim3/glm53-flash-iq3xxs-1` and `rahim3/glm53-flash-iq3xxs-2` — the routed experts (Unsloth's UD-IQ3_XXS GGUF)
   - `rahim3/glm53-flash-serve` — the rest of the weights, the vision tower, the tokenizer and the compiled programs

Then run the cells top to bottom. The last cell keeps running on purpose: that is your server. The endpoint URL
and API key appear in a `READY` banner in its output, about 16 minutes after you start.

(Without the serve dataset the script can still build from `rahim3/glm53-flash-fp8-1` … `-4`, the FP8 checkpoint;
that takes about 22 minutes.)

If the last cell stops in its first minute, its message says why: no TPU in this session (Kaggle does that
sometimes; stop and start the session again), Internet off, or a dataset not attached.


In [ ]:
%%writefile serve_config.json
{
  "streams": 4,
  "max_len": 262144,
  "reasoning_effort_default": "low",
  "vision": true,
  "keepalive_min": 90
}


### Configuration
The defaults above are what we serve. Things you might change:
- `streams`: how many requests decode together. Three fit in HBM at the full 262k context; the fourth waits for a
  free slot. Each extra stream slows the others (one stream ~65 tok/s, two ~40 each, three ~30 each).
- `reasoning_effort_default`: `"low"`, `"medium"` or `"high"`; clients can still set it per request.
- `vision`: `false` skips the vision tower (saves a minute and a little HBM; image inputs then error out).
- `keepalive_min`: the server shuts itself down after this long. Raise it (up to about 480) when you want a
  long-lived endpoint; the default keeps a test run cheap on your TPU quota.
- `api_key`: set your own; otherwise one is generated and printed in the banner.
- `ntfy_topic`: optional; the script posts its progress to `ntfy.sh/<topic>` so you can follow it from your phone.


### The engine
One cell that writes the engine package next to the script. It is generated from the repo, so there is nothing to edit here; scroll past it.


In [ ]:
# The engine: the `glm53` package (15 files), generated from the repo's glm53-flash/engine/glm53/ — nothing to edit here.
import os
os.makedirs("glm53", exist_ok=True)
FILES = {}
FILES["glm53/__init__.py"] = r'''"""GLM-5.3-Flash (Glm5Next) in JAX for TPU v5e-8 with host-resident experts."""
'''
FILES["glm53/basestore.py"] = r'''"""Base store: everything a serving build needs besides the routed experts — the non-expert parameters exactly as
the engine holds them (int8 nodes included), the vision tower and the tokenizer / config files — written once to a
directory (a Kaggle kernel's output, turned into a dataset) and read back in seconds instead of from the 265 GB FP8
checkpoint.

Layout: `manifest.json`, `top.npz` (embed / norm / lm_head), `layer_NN.npz` (one per layer, the expert tables left
out), `vision.npz`, and the tokenizer files. Trees are flattened to "a/b/0/c" keys (lists as numbered keys); bf16
arrays are stored as uint16 bit patterns with the dtype recorded; non-array leaves go into the manifest."""
import json
import os
import shutil

import numpy as np
import jax

EXPERT_KEYS = ("gate_q", "up_q", "down_q")                     # resident expert tables (glm53.resident), never stored
HF_FILES = ("config.json", "generation_config.json", "tokenizer.json", "tokenizer_config.json", "chat_template.jinja",
            "special_tokens_map.json", "preprocessor_config.json", "video_preprocessor_config.json")


def flatten(tree, prefix=""):
    """-> {"a/b/0/c": leaf}; dicts and lists are containers, everything else is a leaf."""
    out = {}
    if isinstance(tree, dict):
        for k, v in tree.items():
            out.update(flatten(v, f"{prefix}{k}/"))
    elif isinstance(tree, (list, tuple)):
        for i, v in enumerate(tree):
            out.update(flatten(v, f"{prefix}{i}/"))
    else:
        out[prefix[:-1]] = tree
    return out


def unflatten(flat):
    """Inverse of `flatten`: a container whose keys are all integers becomes a list (in index order)."""
    root = {}
    for key, leaf in flat.items():
        parts = key.split("/")
        d = root
        for p in parts[:-1]:
            d = d.setdefault(p, {})
        d[parts[-1]] = leaf

    def fix(d):
        if not isinstance(d, dict):
            return d
        d = {k: fix(v) for k, v in d.items()}
        if d and all(k.isdigit() for k in d):
            return [d[str(i)] for i in range(len(d))]
        return d
    return fix(root)


def _host(x):
    a = np.asarray(jax.device_get(x)) if hasattr(x, "shape") else x
    if getattr(a, "dtype", None) is not None and a.dtype.name == "bfloat16":
        return a.view(np.uint16), "bfloat16"
    return a, str(a.dtype)


def save_tree(path, tree):
    """One npz for a tree: arrays stored natively (bf16 as uint16), other leaves in the "__meta__" entry."""
    arrays, dtypes, others = {}, {}, {}
    for k, v in flatten(tree).items():
        if hasattr(v, "shape") and hasattr(v, "dtype"):
            a, dt = _host(v)
            arrays[k] = a
            dtypes[k] = dt
        else:
            others[k] = v
    meta = json.dumps({"dtypes": dtypes, "others": others})
    np.savez(path, __meta__=np.frombuffer(meta.encode(), np.uint8), **arrays)


def load_tree(path):
    import ml_dtypes
    with np.load(path) as z:
        meta = json.loads(bytes(z["__meta__"]).decode())
        flat = {}
        for k, dt in meta["dtypes"].items():
            a = z[k]
            flat[k] = a.view(ml_dtypes.bfloat16) if dt == "bfloat16" else a
    flat.update(meta["others"])
    return unflatten(flat)


def dump(out_dir, params, vision=None, hf_dir=None, meta=None, log=print):
    """Write the store: `params` = the engine's parameter tree (device or host arrays; expert tables skipped),
    `vision` = the vision tower's host parameter tree (as `glm53.vision.load_vision` returns it, any dtype),
    `hf_dir` = where the tokenizer / config files are."""
    os.makedirs(out_dir, exist_ok=True)
    save_tree(os.path.join(out_dir, "top.npz"), {k: params[k] for k in ("embed", "norm", "lm_head")})
    n = len(params["layers"])
    for i, L in enumerate(params["layers"]):
        L = {**L, "mlp": {k: v for k, v in L["mlp"].items() if k not in EXPERT_KEYS}}
        save_tree(os.path.join(out_dir, f"layer_{i:02d}.npz"), L)
        if i % 10 == 0 or i == n - 1:
            log(f"   base store: layer {i + 1}/{n}")
    if vision is not None:
        save_tree(os.path.join(out_dir, "vision.npz"), vision)
    files = []
    if hf_dir:
        for f in HF_FILES:
            src = os.path.join(hf_dir, f)
            if os.path.exists(src):
                shutil.copy(src, os.path.join(out_dir, f)); files.append(f)
    manifest = {"n_layers": n, "vision": vision is not None, "files": files, **(meta or {})}
    json.dump(manifest, open(os.path.join(out_dir, "manifest.json"), "w"), indent=1)
    size = sum(os.path.getsize(os.path.join(out_dir, f)) for f in os.listdir(out_dir)) / 1e9
    log(f"   base store written: {out_dir} ({size:.2f} GB, {n} layers, vision {vision is not None})")
    return manifest


def load(in_dir):
    """-> {"top": {embed, norm, lm_head}, "layers": [layer trees without expert tables], "vision": tree or None,
    "manifest": dict}. Host arrays; bf16 restored."""
    manifest = json.load(open(os.path.join(in_dir, "manifest.json")))
    top = load_tree(os.path.join(in_dir, "top.npz"))
    layers = [load_tree(os.path.join(in_dir, f"layer_{i:02d}.npz")) for i in range(manifest["n_layers"])]
    vision = load_tree(os.path.join(in_dir, "vision.npz")) if manifest.get("vision") else None
    return {"top": top, "layers": layers, "vision": vision, "manifest": manifest}
'''
FILES["glm53/checkpoint.py"] = r'''"""Read the real GLM-5.3-Flash safetensors checkpoint into the `glm53.model` param layout.

Checkpoint facts (verified 2026-09-05): fp8 e4m3 tensors carry `<name>_scale_inv` float32 with 128x128
blocks; KDA projections / conv / gates / indexer / kv_b / router / mHC are bf16 or fp32. Expert tensors
are per-expert (`mlp.experts.N.{gate,up,down}_proj.weight`). All Linear weights are torch [out, in].
"""
from __future__ import annotations

import json
import os
import re
from functools import lru_cache

import numpy as np

PREFIX = "model.language_model."
BLOCK = 128


@lru_cache(maxsize=None)
def weight_map(model_dir: str) -> dict[str, str]:
    """model_dir may be one directory or several joined by ':' (e.g. the 4 Kaggle dataset mounts); the index is
    taken from the first dir that has it and each shard is resolved to whichever dir contains it."""
    for d in model_dir.split(":"):
        p = os.path.join(d, "model.safetensors.index.json")
        if os.path.exists(p):
            with open(p) as f:
                return json.load(f)["weight_map"]
    raise FileNotFoundError("model.safetensors.index.json not found in " + model_dir)


def resolve_shard(model_dir: str, fname: str) -> str:
    for d in model_dir.split(":"):
        p = os.path.join(d, fname)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"{fname} not found in {model_dir}")


class ShardReader:
    """Caches open safetensors handles (torch framework, so fp8 loads natively). `model_dir` may be ':'-joined."""

    def __init__(self, model_dir: str):
        self.model_dir = model_dir
        self.wm = weight_map(model_dir)
        self._open: dict[str, object] = {}

    def _handle(self, fname):
        from safetensors import safe_open
        if fname not in self._open:
            self._open[fname] = safe_open(resolve_shard(self.model_dir, fname), framework="pt")
        return self._open[fname]

    def release(self, fname: str):
        """Close a shard and drop its pages from the page cache (page cache counts toward the Kaggle cgroup)."""
        self._open.pop(fname, None)
        try:
            fd = os.open(resolve_shard(self.model_dir, fname), os.O_RDONLY)
            os.posix_fadvise(fd, 0, 0, os.POSIX_FADV_DONTNEED)
            os.close(fd)
        except Exception:  # noqa: BLE001
            pass

    def has(self, name: str) -> bool:
        return name in self.wm

    def raw(self, name: str):
        """torch tensor as stored (bf16 / fp8 / f32)."""
        return self._handle(self.wm[name]).get_tensor(name)

    def get(self, name: str, dtype=np.float32) -> np.ndarray:
        """Dequantized numpy array. fp8 tensors are multiplied by their blockwise scale_inv."""
        import torch
        t = self.raw(name)
        if t.dtype == torch.float8_e4m3fn:
            s = self.raw(name + "_scale_inv").float().numpy()
            return dequant_fp8(t.float().numpy(), s).astype(dtype)
        return t.float().numpy().astype(dtype)

    def get_fp8_raw(self, name: str):
        """(fp8 bytes as int8-viewed uint8 array [O,I], scale_inv f32 [O/128, I/128]) for host expert tables."""
        import torch
        t = self.raw(name)
        assert t.dtype == torch.float8_e4m3fn, name
        s = self.raw(name + "_scale_inv").float().numpy()
        return t.view(torch.uint8).numpy(), s


def dequant_fp8(w: np.ndarray, scale_inv: np.ndarray, block: int = BLOCK) -> np.ndarray:
    O, I = w.shape
    s = np.repeat(np.repeat(scale_inv, block, axis=0), block, axis=1)[:O, :I]
    return w.astype(np.float32) * s


def _lin(r: ShardReader, name: str, dtype):
    return np.ascontiguousarray(r.get(name, dtype).T)


def load_top(r: ShardReader, dtype=np.float32):
    return {
        "embed": r.get(PREFIX + "embed_tokens.weight", dtype),
        "norm": r.get(PREFIX + "norm.weight", dtype),
        "lm_head": _lin(r, "lm_head.weight", dtype),
    }


def load_hc(r: ShardReader, pfx: str, site: str, dtype):
    return {"fn": _lin(r, f"{pfx}hc_{site}_fn", dtype), "base": r.get(f"{pfx}hc_{site}_base"),
            "scale": r.get(f"{pfx}hc_{site}_scale")}


def load_kda(r: ShardReader, a: str, dtype):
    return {
        "q": _lin(r, a + "q_proj.weight", dtype), "k": _lin(r, a + "k_proj.weight", dtype),
        "v": _lin(r, a + "v_proj.weight", dtype),
        "conv_q": r.get(a + "q_conv1d.weight")[:, 0, :], "conv_k": r.get(a + "k_conv1d.weight")[:, 0, :],
        "conv_v": r.get(a + "v_conv1d.weight")[:, 0, :],
        "f_a": _lin(r, a + "f_a_proj.weight", dtype), "f_b": _lin(r, a + "f_b_proj.weight", dtype),
        "dt_bias": r.get(a + "dt_bias"), "A_log": r.get(a + "A_log"),
        "b": _lin(r, a + "b_proj.weight", dtype),
        "g_a": _lin(r, a + "g_a_proj.weight", dtype), "g_b": _lin(r, a + "g_b_proj.weight", dtype),
        "o_norm": r.get(a + "o_norm.weight"), "o": _lin(r, a + "o_proj.weight", dtype),
    }


def load_mla(r: ShardReader, a: str, dtype):
    p = {
        "q_a": _lin(r, a + "q_a_proj.weight", dtype), "q_a_norm": r.get(a + "q_a_layernorm.weight"),
        "q_b": _lin(r, a + "q_b_proj.weight", dtype),
        "kv_a": _lin(r, a + "kv_a_proj_with_mqa.weight", dtype), "kv_a_norm": r.get(a + "kv_a_layernorm.weight"),
        "kv_b": _lin(r, a + "kv_b_proj.weight", dtype), "o": _lin(r, a + "o_proj.weight", dtype),
    }
    ix = a + "indexer."
    if r.has(ix + "wk.weight"):
        p["indexer"] = {
            "wq_b": _lin(r, ix + "wq_b.weight", dtype), "wk": _lin(r, ix + "wk.weight", dtype),
            "k_norm_w": r.get(ix + "k_norm.weight"), "k_norm_b": r.get(ix + "k_norm.bias"),
            "weights_proj": _lin(r, ix + "weights_proj.weight", dtype),
            "ape": r.get(ix + "index_kpool_compress_ape"), "gate": _lin(r, ix + "index_kpool_compress_gate", dtype),
        }
    return p


def load_mlp(r: ShardReader, m: str, dtype):
    return {"gate": _lin(r, m + "gate_proj.weight", dtype), "up": _lin(r, m + "up_proj.weight", dtype),
            "down": _lin(r, m + "down_proj.weight", dtype)}


def load_moe_nonexpert(r: ShardReader, m: str, dtype):
    """Router + shared expert only. Routed experts are loaded separately (host tables)."""
    return {"router_w": _lin(r, m + "gate.weight", dtype), "router_bias": r.get(m + "gate.e_score_correction_bias"),
            "shared": load_mlp(r, m + "shared_experts.", dtype)}


def load_expert(r: ShardReader, layer: int, e: int, dtype=np.float32):
    """Dequantized (gate_up [D, 2mi], down [mi, D]) for one routed expert."""
    m = f"{PREFIX}layers.{layer}.mlp.experts.{e}."
    gate = _lin(r, m + "gate_proj.weight", dtype)
    up = _lin(r, m + "up_proj.weight", dtype)
    return np.concatenate([gate, up], axis=1), _lin(r, m + "down_proj.weight", dtype)


def load_layer(r: ShardReader, i: int, layer_type: str, mlp_type: str, dtype=np.float32, with_experts=False):
    pfx = f"{PREFIX}layers.{i}."
    p = {
        "ln1": r.get(pfx + "input_layernorm.weight"), "ln2": r.get(pfx + "post_attention_layernorm.weight"),
        "hc_attn": load_hc(r, pfx, "attn", dtype), "hc_ffn": load_hc(r, pfx, "ffn", dtype),
    }
    a = pfx + "self_attn."
    p["attn"] = load_kda(r, a, dtype) if layer_type == "linear_attention" else load_mla(r, a, dtype)
    m = pfx + "mlp."
    if mlp_type == "sparse":
        p["mlp"] = load_moe_nonexpert(r, m, dtype)
        if with_experts:
            n_exp = len({k for k in r.wm if k.startswith(m + "experts.") and k.endswith("gate_proj.weight")})
            gu, dn = zip(*(load_expert(r, i, e, dtype) for e in range(n_exp)))
            p["mlp"]["gate_up"] = np.stack(gu)
            p["mlp"]["down"] = np.stack(dn)
    else:
        p["mlp"] = load_mlp(r, m, dtype)
    return p


def load_mtp_layer(r: ShardReader, i: int, dtype=np.float32):
    """The MTP (NextN) layer: a plain pre-norm MLA (+ indexer) + MoE block without hyper-connections, fed with
    eh_proj(cat(enorm(embed(next token)), hnorm(previous hidden))); `head_norm` precedes the shared lm_head.
    Routed experts come from the GGUF (Q2_K / Q3_K planes), like every other sparse layer."""
    pfx = f"{PREFIX}layers.{i}."
    return {
        "ln1": r.get(pfx + "input_layernorm.weight"), "ln2": r.get(pfx + "post_attention_layernorm.weight"),
        "attn": load_mla(r, pfx + "self_attn.", dtype), "mlp": load_moe_nonexpert(r, pfx + "mlp.", dtype),
        "enorm": r.get(pfx + "enorm.weight"), "hnorm": r.get(pfx + "hnorm.weight"),
        "eh_proj": _lin(r, pfx + "eh_proj.weight", dtype), "head_norm": r.get(pfx + "shared_head.norm.weight"),
    }


def layer_shards(model_dir: str, i: int) -> set[str]:
    return {v for k, v in weight_map(model_dir).items() if re.search(rf"layers\.{i}\.", k)}


# ----------------------------------------------------------------------------- device-side fp8 dequant
def dequant_fp8_device(w_u8, scale_inv, out_dtype=None, block: int = BLOCK):
    """JAX: fp8-e4m3 bytes viewed as uint8 [..., O, I] + scale_inv [..., O/b, I/b] -> float [..., O, I].
    Used on TPU after streaming raw expert bytes from the host (halves PCIe traffic vs bf16)."""
    import jax
    import jax.numpy as jnp
    f8 = jax.lax.bitcast_convert_type(w_u8, jnp.float8_e4m3fn)
    w = f8.astype(jnp.float32)
    O, I = w.shape[-2:]
    s = jnp.repeat(jnp.repeat(scale_inv.astype(jnp.float32), block, axis=-2), block, axis=-1)[..., :O, :I]
    out = w * s
    return out if out_dtype is None else out.astype(out_dtype)


# ----------------------------------------------------------------------------- int4 (RTN, blockwise) pack/unpack
def quant_int4_blockwise(w: np.ndarray, block: int = BLOCK):
    """float [O, I] -> (packed uint8 [O, I//2] (low nibble = even column), scale f32 [O/b, I/b]); symmetric RTN
    to [-8, 7] per block. Requires O, I multiples of `block` and I even."""
    O, I = w.shape
    blk = w.reshape(O // block, block, I // block, block)
    amax = np.abs(blk).max(axis=(1, 3), keepdims=True)
    scale = np.where(amax > 0, amax / 7.0, 1.0).astype(np.float32)
    q = np.clip(np.round(blk / scale), -8, 7).astype(np.int8).reshape(O, I)
    u = (q & 0xF).astype(np.uint8)
    packed = (u[:, 0::2] | (u[:, 1::2] << 4)).astype(np.uint8)
    return packed, scale.reshape(O // block, I // block)


def dequant_int4_device(packed_u8, scale, out_dtype=None, block: int = BLOCK):
    """JAX: packed [..., O, I//2] uint8 + scale [..., O/b, I/b] -> float [..., O, I]."""
    import jax.numpy as jnp
    lo = (packed_u8 & 0xF).astype(jnp.int8)
    hi = (packed_u8 >> 4).astype(jnp.int8)
    lo = jnp.where(lo > 7, lo - 16, lo)
    hi = jnp.where(hi > 7, hi - 16, hi)
    q = jnp.stack([lo, hi], axis=-1).reshape(packed_u8.shape[:-1] + (packed_u8.shape[-1] * 2,)).astype(jnp.float32)
    O, I = q.shape[-2:]
    s = jnp.repeat(jnp.repeat(scale.astype(jnp.float32), block, axis=-2), block, axis=-1)[..., :O, :I]
    out = q * s
    return out if out_dtype is None else out.astype(out_dtype)


def dequant_int4_numpy(packed_u8, scale, block: int = BLOCK):
    lo = (packed_u8 & 0xF).astype(np.int8); hi = (packed_u8 >> 4).astype(np.int8)
    lo[lo > 7] -= 16; hi[hi > 7] -= 16
    q = np.stack([lo, hi], -1).reshape(packed_u8.shape[:-1] + (packed_u8.shape[-1] * 2,)).astype(np.float32)
    O, I = q.shape[-2:]
    s = np.repeat(np.repeat(scale, block, axis=-2), block, axis=-1)[..., :O, :I]
    return q * s


# ----------------------------------------------------------------------------- mmap-free shard reader
class RawShardReader(ShardReader):
    """Reads tensors with explicit pread instead of safetensors' mmap. With mmap, every touched page of every open
    shard counts toward the process RSS (and the Kaggle cgroup), which is what pushed the 2026-09-06 full build over
    the memory guard and forced int4 for half the layers. Here only our own arrays occupy memory."""

    _DT = {"F32": np.float32, "F16": np.float16, "BF16": "bf16", "F8_E4M3": "fp8", "I8": np.int8, "U8": np.uint8,
           "I32": np.int32, "I64": np.int64}

    def _header(self, fname):
        if fname not in self._open:
            path = resolve_shard(self.model_dir, fname)
            with open(path, "rb") as f:
                n = int.from_bytes(f.read(8), "little")
                hdr = json.loads(f.read(n))
            hdr.pop("__metadata__", None)
            self._open[fname] = (path, 8 + n, hdr)
        return self._open[fname]

    def _read(self, name):
        path, base, hdr = self._header(self.wm[name])
        info = hdr[name]
        b0, b1 = info["data_offsets"]
        buf = np.empty(b1 - b0, np.uint8)
        fd = os.open(path, os.O_RDONLY)
        try:
            view = memoryview(buf)
            off = 0
            while off < len(buf):
                k = os.preadv(fd, [view[off:]], base + b0 + off)
                if k <= 0:
                    raise IOError(f"short read on {name}")
                off += k
        finally:
            os.close(fd)
        return buf, info["dtype"], tuple(info["shape"])

    def raw(self, name):  # torch tensor for API compatibility (rarely needed)
        import torch
        buf, dt, shape = self._read(name)
        t = torch.frombuffer(buf.tobytes(), dtype={"F32": torch.float32, "BF16": torch.bfloat16,
                                                   "F8_E4M3": torch.float8_e4m3fn}[dt])
        return t.reshape(shape)

    def get(self, name, dtype=np.float32):
        buf, dt, shape = self._read(name)
        if dt == "F32":
            return buf.view(np.float32).reshape(shape).astype(dtype, copy=False)
        if dt == "BF16":
            u16 = buf.view(np.uint16).astype(np.uint32) << 16
            return u16.view(np.float32).reshape(shape).astype(dtype, copy=False)
        if dt == "F8_E4M3":
            f = _fp8_to_f32(buf).reshape(shape)
            s = self.get(name + "_scale_inv")
            return dequant_fp8(f, s).astype(dtype, copy=False)
        raise ValueError(f"unsupported dtype {dt} for {name}")

    def get_fp8_raw(self, name):
        buf, dt, shape = self._read(name)
        assert dt == "F8_E4M3", (name, dt)
        return buf.reshape(shape), self.get(name + "_scale_inv")

    def release(self, fname):
        self._open.pop(fname, None)
        try:
            fd = os.open(resolve_shard(self.model_dir, fname), os.O_RDONLY)
            os.posix_fadvise(fd, 0, 0, os.POSIX_FADV_DONTNEED)
            os.close(fd)
        except Exception:  # noqa: BLE001
            pass


_FP8_LUT = None


def _fp8_to_f32(u8: np.ndarray) -> np.ndarray:
    """e4m3fn bytes -> float32 via a 256-entry lookup table (pure numpy)."""
    global _FP8_LUT
    if _FP8_LUT is None:
        lut = np.empty(256, np.float32)
        for i in range(256):
            s = -1.0 if i & 0x80 else 1.0
            e = (i >> 3) & 0xF
            m = i & 0x7
            if e == 0:
                v = s * (m / 8.0) * 2.0 ** -6
            elif e == 15 and m == 7:
                v = np.nan
            else:
                v = s * (1 + m / 8.0) * 2.0 ** (e - 7)
            lut[i] = v
        _FP8_LUT = lut
    return _FP8_LUT[u8]
'''
FILES["glm53/engine.py"] = r'''"""Tensor-parallel (TP = #devices) engine for `glm53.model` with host-resident routed experts.

Sharding (mesh axis "x"):
  * attention heads (KDA 64, MLA 64) and MLP/expert intermediate columns are split across devices;
    every row-parallel matmul output is `psum`'d via `Cfg.tp_reduce`, so activations/streams stay replicated.
  * embed rows and lm_head columns are vocab-sharded.
  * routed experts are NOT on device: `HostExperts.fetch` gathers each device's slice of the selected experts on the
    host (NumPy) and ships it through `jax.pure_callback` inside the jitted `shard_map` program.

Everything runs on CPU with `XLA_FLAGS=--xla_force_host_platform_device_count=8` for tests.
"""
from __future__ import annotations

import dataclasses
import time
from functools import partial

import jax
import jax.numpy as jnp
import numpy as np
from jax import lax
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P

from glm53 import model as M

try:
    from jax import shard_map
except ImportError:  # older jax
    from jax.experimental.shard_map import shard_map

AXIS = "x"
R = P()  # replicated


# ----------------------------------------------------------------------------- device-side sampling
SAMPLE_IMPL = "approx"          # "approx": TPU-native approx_max_k candidates + bisection top-p (no sorts); "sort": exact top_k + sort


def device_sample(z_local, temperature, top_p, key, n_cand=2048, impl=None):
    """Inside a shard_map program: one token per row of the logits, whose vocab columns are sharded over AXIS
    (`z_local` [N, V/n], any float dtype). Candidates = the top n_cand/n logits of every chip, all-gathered (2048 on
    8 chips, the same truncation as the host sampler's partial sort); temperature and top-p in f32 over the
    candidates, then a categorical draw with `key` (identical on every chip -> the same token everywhere).
    temperature <= 0 -> the argmax (exact over the whole vocab). impl "approx" (default): `lax.approx_max_k`
    picks the candidates (recall 0.99; the exact per-chip argmax is forced into the set, so greedy stays exact and
    the mode is never missing) and the top-p set {p >= t} is found by bisection on t (24 steps) instead of a sort —
    XLA sorts are slow on the TPU (the "sort" variant cost ~2 ms per token, as much as the host sampler).
    Returns int32 [N] token ids."""
    impl = SAMPLE_IMPL if impl is None else impl
    N, Vl = z_local.shape
    n = lax.axis_size(AXIS)
    kc = min(max(1, n_cand // n), Vl)
    z = z_local.astype(jnp.float32)
    if impl == "sort" or kc >= Vl:
        vals, idx = lax.top_k(z, kc)                                             # [N, kc] per chip
    else:
        vals, idx = lax.approx_max_k(z, kc, recall_target=0.99)
        am = jnp.argmax(z, axis=1)                                               # force the exact argmax in
        present = jnp.any(idx == am[:, None], axis=1)
        last = jnp.arange(kc)[None, :] == kc - 1
        repl = (~present)[:, None] & last
        idx = jnp.where(repl, am[:, None], idx)
        vals = jnp.where(repl, jnp.take_along_axis(z, am[:, None], axis=1), vals)
    gidx = idx.astype(jnp.int32) + lax.axis_index(AXIS).astype(jnp.int32) * Vl
    vals = lax.all_gather(vals, AXIS, axis=1, tiled=True)                        # [N, n*kc], replicated
    gidx = lax.all_gather(gidx, AXIS, axis=1, tiled=True)
    greedy = jnp.take_along_axis(gidx, jnp.argmax(vals, axis=1, keepdims=True), axis=1)[:, 0]
    temperature = jnp.reshape(jnp.asarray(temperature, jnp.float32), (-1, 1))     # [] or [N] -> [1 or N, 1] per row
    top_p = jnp.reshape(jnp.asarray(top_p, jnp.float32), (-1, 1))
    t = jnp.where(temperature > 0, temperature, 1.0)
    lp = vals / t
    lp = lp - jnp.max(lp, axis=1, keepdims=True)
    p = jnp.exp(lp)
    p = p / jnp.sum(p, axis=1, keepdims=True)
    if impl == "sort":
        ps = -jnp.sort(-p, axis=1)                                               # descending
        cum = jnp.cumsum(ps, axis=1)
        n_keep = jnp.minimum(jnp.sum(cum <= top_p, axis=1) + 1, ps.shape[1])    # host rule: count(cum <= top_p) + 1
        thr = jnp.take_along_axis(ps, (n_keep - 1)[:, None], axis=1)
    else:                                                                        # largest t with mass(p >= t) >= top_p
        lo = jnp.zeros((N, 1), jnp.float32)
        hi = jnp.max(p, axis=1, keepdims=True)
        for _ in range(24):
            mid = 0.5 * (lo + hi)
            ok = jnp.sum(jnp.where(p >= mid, p, 0.0), axis=1, keepdims=True) >= top_p
            lo = jnp.where(ok, mid, lo)
            hi = jnp.where(ok, hi, mid)
        thr = lo
    lp = jnp.where(p >= thr, lp, -jnp.inf)
    draw = jax.random.categorical(key, lp, axis=1)
    sampled = jnp.take_along_axis(gidx, draw[:, None], axis=1)[:, 0]
    return jnp.where(temperature[:, 0] > 0, sampled, greedy).astype(jnp.int32)


class DeviceSampler:
    """Host-side handle of the device sampler: temperature, top_p and the PRNG key live on the chips as REPLICATED
    arrays (placed once per mesh by `bind`); the sampling programs return the next key, so a decode step moves no
    sampling state between host and device (per-step scalar transfers cost ~2 ms on the Kaggle TPU VM)."""

    def __init__(self, temperature=1.0, top_p=1.0, seed=None):
        self.seed = int(np.random.SeedSequence(seed).generate_state(1)[0]) if seed is None else int(seed)
        self._mesh = None
        self.set(temperature, top_p)

    def set(self, temperature, top_p):
        """Scalars (every row) or one value per row of the batch ([B] sequences, `decode_rows`); re-placed on the
        chips at the next `bind` only when they changed (a batch membership change, not every step)."""
        self.temperature = np.asarray(temperature, np.float32)
        self.top_p = np.asarray(top_p, np.float32)
        self._placed = None

    def bind(self, mesh):
        """-> (temperature, top_p, key) replicated on `mesh` (placed once per mesh / per `set`)."""
        sh = NamedSharding(mesh, P())
        if self._mesh is not mesh:
            self.key = jax.device_put(jax.random.PRNGKey(self.seed), sh)
            self._mesh, self._placed = mesh, None
        if self._placed is None:
            self.t = jax.device_put(self.temperature, sh)
            self.p = jax.device_put(self.top_p, sh)
            self._placed = True
        return self.t, self.p, self.key

    def advance(self, key):
        self.key = key


# ----------------------------------------------------------------------------- specs
def layer_specs(cfg: M.Cfg, i: int, p: dict) -> dict:
    col, row = P(None, AXIS), P(AXIS, None)
    s = {"ln1": R, "ln2": R,
         "hc_attn": {"fn": R, "base": R, "scale": R}, "hc_ffn": {"fn": R, "base": R, "scale": R}}
    if cfg.layer_types[i] == "linear_attention":
        s["attn"] = {"q": col, "k": col, "v": col, "conv_q": row, "conv_k": row, "conv_v": row,
                     "f_a": R, "f_b": col, "dt_bias": P(AXIS), "A_log": P(AXIS), "b": col,
                     "g_a": R, "g_b": col, "o_norm": R, "o": row}
    else:
        s["attn"] = {"q_a": R, "q_a_norm": R, "q_b": col, "kv_a": R, "kv_a_norm": R, "kv_b": col, "o": row}
        if "indexer" in p["attn"]:
            s["attn"]["indexer"] = {k: R for k in p["attn"]["indexer"]}
    if cfg.mlp_types[i] == "sparse":
        s["mlp"] = {"router_w": R, "router_bias": R, "shared": {"gate": col, "up": col, "down": row}}
        if "gate_up" in p["mlp"]:            # on-device experts (tests only): [E, D, 2, mi] / [E, mi, D]
            s["mlp"]["gate_up"] = P(None, None, None, AXIS)
            s["mlp"]["down"] = P(None, AXIS, None)
        for k in ("gate_q", "up_q", "down_q"):   # resident codebook-quantized experts, chip-major (glm53.resident)
            if k in p["mlp"]:                    # dict of planar arrays [n_dev, E*rows_p, R] (or one array, legacy)
                s["mlp"][k] = {pk: P(AXIS) for pk in p["mlp"][k]} if isinstance(p["mlp"][k], dict) else P(AXIS)
    else:
        s["mlp"] = {"gate": col, "up": col, "down": row}
    return s


def param_specs(cfg: M.Cfg, params: dict) -> dict:
    return {"embed": P(AXIS, None), "norm": R, "lm_head": P(None, AXIS),
            "layers": [layer_specs(cfg, i, params["layers"][i]) for i in range(cfg.n_layers)]}


def cache_specs(cfg: M.Cfg, caches: list, seq_shard: int = 1) -> list:
    """KDA caches are head-sharded; MLA/indexer caches are replicated, or sequence-sharded (axis 1) when the
    engine was built with seq_shard (glm53.model cache layout notes)."""
    out = []
    mla = P(None, AXIS) if seq_shard > 1 else R
    for i, c in enumerate(caches):
        if cfg.layer_types[i] == "linear_attention":
            out.append({"conv": P(None, None, AXIS), "state": P(None, AXIS)})
        else:
            out.append({k: (mla if k in M.ROW_KEYS else R) for k in c})       # tail buffers are replicated
    return out


def local_cfg(cfg: M.Cfg, n: int, seq_shard: bool = False, q_block: int = 128, cache_q8: bool = False) -> M.Cfg:
    assert cfg.kda_heads % n == 0 and cfg.n_heads % n == 0 and cfg.inter % n == 0 and cfg.moe_inter % n == 0
    return dataclasses.replace(cfg, kda_heads=cfg.kda_heads // n, n_heads=cfg.n_heads // n, inter=cfg.inter // n,
                               moe_inter=cfg.moe_inter // n, tp_reduce=lambda y: lax.psum(y, AXIS), tp_axis=AXIS,
                               seq_shard=n if seq_shard else 1, q_block=q_block, cache_q8=cache_q8 or cfg.cache_q8)


# ----------------------------------------------------------------------------- host experts
class HostExperts:
    """Routed expert tables kept in host RAM, pre-sliced per device.

    Layout is CHIP-MAJOR: every table is [n_dev, E, ...] so a device's expert slice is one contiguous block
    (measured on Kaggle: [E, chip] layout + strided take = 912 ms/layer; chip-major + threaded memcpy = 7 ms).
    mode "float": tables[layer] = (gate_up [n_dev, E, D, 2*ml], down [n_dev, E, ml, D]) float arrays.
    mode "fp8":   tables[layer] = (gu_bytes [n_dev, E, D, 2*ml] uint8, gu_scale [n_dev, E, D/128, 2*ml/128] f32,
                                   dn_bytes [n_dev, E, ml, D] uint8,   dn_scale [n_dev, E, ml/128, D/128] f32)
                  — raw e4m3 bytes streamed to the device and dequantized there (`checkpoint.dequant_fp8_device`).
    """

    def __init__(self, tables: dict, mode="float", out_dtype=jnp.float32):
        """mode: "float" | "fp8" | "int4", or a dict {layer: mode} for mixed tables (e.g. fp8 until host RAM ran
        out, int4 after). int4 tables: (gu_packed [n,E,D,ml] u8, gu_scale, dn_packed [n,E,ml,D/2] u8, dn_scale)."""
        self.tables = tables
        self.modes = dict(mode) if isinstance(mode, dict) else {L: mode for L in tables}
        assert all(m in ("float", "fp8", "int4") for m in self.modes.values())
        self.out_dtype = out_dtype

    @property
    def mode(self):
        ms = set(self.modes.values())
        return ms.pop() if len(ms) == 1 else "mixed"

    def mode_for(self, layer):
        return self.modes[layer]

    @classmethod
    def from_dense(cls, params: dict, cfg: M.Cfg, n_dev: int, out_dtype=jnp.float32) -> "HostExperts":
        """Split on-device style tables gate_up [E,D,2mi], down [E,mi,D] into per-device slices (float mode)."""
        tables = {}
        ml = cfg.moe_inter // n_dev
        for i in range(cfg.n_layers):
            if cfg.mlp_types[i] != "sparse":
                continue
            gu = np.asarray(params["layers"][i]["mlp"]["gate_up"])           # [E, D, 2mi]
            dn = np.asarray(params["layers"][i]["mlp"]["down"])              # [E, mi, D]
            E, D, _ = gu.shape
            tables[i] = (chip_major(split_gate_up(gu, n_dev)), chip_major(dn.reshape(E, n_dev, ml, D)))
        return cls(tables, "float", out_dtype)

    def gather(self, layer: int, idx: np.ndarray, chip: int):
        return tuple(t[chip][idx] for t in self.tables[layer])

    def make_fetch(self, cfg_local: M.Cfg, n_tokens: int):
        """Returns fetch(layer, idx) usable inside shard_map (per-device callback)."""
        from glm53.checkpoint import BLOCK, dequant_fp8_device, dequant_int4_device
        k, D, ml = cfg_local.topk, cfg_local.hidden, cfg_local.moe_inter

        def shapes_for(mode):
            if mode == "float":
                return (jax.ShapeDtypeStruct((n_tokens, k, D, 2 * ml), self.out_dtype),
                        jax.ShapeDtypeStruct((n_tokens, k, ml, D), self.out_dtype))
            if mode == "int4":
                return (jax.ShapeDtypeStruct((n_tokens, k, D, ml), jnp.uint8),
                        jax.ShapeDtypeStruct((n_tokens, k, D // BLOCK, 2 * ml // BLOCK), jnp.float32),
                        jax.ShapeDtypeStruct((n_tokens, k, ml, D // 2), jnp.uint8),
                        jax.ShapeDtypeStruct((n_tokens, k, ml // BLOCK, D // BLOCK), jnp.float32))
            return (jax.ShapeDtypeStruct((n_tokens, k, D, 2 * ml), jnp.uint8),
                    jax.ShapeDtypeStruct((n_tokens, k, D // BLOCK, 2 * ml // BLOCK), jnp.float32),
                    jax.ShapeDtypeStruct((n_tokens, k, ml, D), jnp.uint8),
                    jax.ShapeDtypeStruct((n_tokens, k, ml // BLOCK, D // BLOCK), jnp.float32))

        def fetch(layer, idx):
            chip = lax.axis_index(AXIS)
            mode = self.mode_for(layer)
            out_shapes = shapes_for(mode)

            def host_fn(idx_np, chip_np):
                outs = self.gather(layer, np.asarray(idx_np), int(chip_np))
                return tuple(np.ascontiguousarray(o, dtype=np.dtype(sh.dtype)) for o, sh in zip(outs, out_shapes))
            outs = jax.pure_callback(host_fn, out_shapes, idx, chip)
            if mode == "float":
                return outs
            gu_b, gu_s, dn_b, dn_s = outs
            if mode == "int4":
                return (dequant_int4_device(gu_b, gu_s, self.out_dtype), dequant_int4_device(dn_b, dn_s, self.out_dtype))
            return (dequant_fp8_device(gu_b, gu_s, self.out_dtype), dequant_fp8_device(dn_b, dn_s, self.out_dtype))
        return fetch


def split_gate_up(gu: np.ndarray, n_dev: int) -> np.ndarray:
    """[E, D, 2mi] (gate|up) -> [E, n_dev, D, 2*ml] with each device holding (gate_slice|up_slice)."""
    E, D, two_mi = gu.shape
    ml = two_mi // 2 // n_dev
    return np.ascontiguousarray(gu.reshape(E, D, 2, n_dev, ml).transpose(0, 3, 1, 2, 4).reshape(E, n_dev, D, 2 * ml))


def split_gate_up_scales(gs: np.ndarray, n_dev: int) -> np.ndarray:
    """Same split for blockwise scales [E, D/128, 2mi/128] -> [E, n_dev, D/128, 2*ml/128] (ml multiple of 128)."""
    return split_gate_up(gs, n_dev)


def chip_major(t: np.ndarray) -> np.ndarray:
    """[E, n_dev, ...] -> contiguous [n_dev, E, ...]."""
    return np.ascontiguousarray(np.moveaxis(t, 1, 0))


# ----------------------------------------------------------------------------- engine
class Engine:
    def __init__(self, cfg: M.Cfg, params: dict, host_experts: HostExperts | None = None, devices=None,
                 max_len: int = 4096, expert_fetch=None, int8_nonexpert: bool = False, seq_shard: bool = False,
                 q_block: int = 128, cache_q8: bool = False):
        devices = devices if devices is not None else jax.devices()
        cfg = dataclasses.replace(cfg, cache_q8=cache_q8 or cfg.cache_q8)     # int8 latent cache (half the HBM per set)
        self.max_len = max_len
        self.expert_fetch = expert_fetch      # e.g. glm53.resident.ResidentFetch (tables live inside params)
        self.n = len(devices)
        self.mesh = Mesh(np.array(devices), (AXIS,))
        self.cfg = cfg
        # seq_shard: MLA latent + indexer caches sharded over the chips (16.9 KB/token/chip replicated -> 2.1 KB;
        # needed beyond ~32k tokens); the capacity must be a multiple of kpool * n_dev
        assert not seq_shard or max_len % (cfg.idx_kpool * self.n) == 0, (max_len, cfg.idx_kpool, self.n)
        self.lcfg = local_cfg(cfg, self.n, seq_shard, q_block, cfg.cache_q8)
        self.host_experts = host_experts
        params = dict(params)
        if host_experts is not None:   # drop on-device expert tables
            params["layers"] = [{**L, "mlp": {k: v for k, v in L["mlp"].items() if k not in ("gate_up", "down")}}
                                if cfg.mlp_types[i] == "sparse" else L for i, L in enumerate(params["layers"])]
        else:
            params["layers"] = [{**L, "mlp": {**L["mlp"], "gate_up": np.asarray(L["mlp"]["gate_up"]).reshape(
                cfg.n_exp, cfg.hidden, 2, cfg.moe_inter)}} if cfg.mlp_types[i] == "sparse" and "gate_up" in L["mlp"]
                else L for i, L in enumerate(params["layers"])]
        self.specs = param_specs(cfg, params)
        from glm53 import quant8 as Q8
        if int8_nonexpert:                    # quantize the big non-expert matrices on the host before device_put
            layers, lspecs = [], []
            for L, sp in zip(params["layers"], self.specs["layers"]):
                qp, qs = Q8.quantize_layer(L, sp, lambda w, ax: jax.tree.map(np.asarray, Q8.quantize_array(w, ax)))
                layers.append(qp); lspecs.append(qs)
            params = {**params, "layers": layers}
            self.specs = {**self.specs, "layers": lspecs}
            if int8_nonexpert == "all":           # + the vocab-sharded embedding (per-row scales) and lm_head (per-column),
                q = Q8.quantize_array_host             # quantized on the host (a device pass OOMed at 262k)
                params = {**params, "embed": q(params["embed"], 0), "lm_head": q(params["lm_head"], 1)}
        self.specs = Q8.expand_specs(params, self.specs)   # params already holding q8 nodes (hot-swapped on device)
        self.params = jax.tree.map(lambda a, s: jax.device_put(a, NamedSharding(self.mesh, s)), params, self.specs,
                                   is_leaf=lambda x: isinstance(x, P))
        self._prefill = {}
        self._decode = {}

    def _cache_spec(self, i):
        if self.cfg.layer_types[i] == "linear_attention":
            return {"conv": P(None, None, AXIS), "state": P(None, AXIS)}
        mla = P(None, AXIS) if self.lcfg.seq_shard > 1 else R
        spec = {"c": mla, "pk": mla, "tk": R, "tg": R} if "indexer" in self.params["layers"][i]["attn"] else {"c": mla}
        if self.cfg.cache_q8:
            spec["cs"] = mla
        return spec

    # ---- per-device program pieces
    def _embed(self, embed_local, tokens):
        from glm53 import quant8 as Q8
        chip = lax.axis_index(AXIS)
        q8 = Q8.is_q8(embed_local)
        table = embed_local["q"] if q8 else embed_local
        rows = table.shape[0]
        local = tokens - chip * rows
        ok = (local >= 0) & (local < rows)
        idx = jnp.clip(local, 0, rows - 1)
        e = jnp.take(table, idx, axis=0)
        if q8:                                              # dequantize only the gathered rows
            e = e.astype(jnp.float32) * jnp.take(embed_local["s"], idx, axis=0)
        return lax.psum(jnp.where(ok[..., None], e, 0), AXIS)

    def _logits_local(self, lm_head_local, h):
        """This chip's vocab columns of the logits [..., V/n] (the shard `_logits` all-gathers)."""
        from glm53 import quant8 as Q8
        if Q8.is_q8(lm_head_local):                         # int8 columns: the convert fuses into the dot, scales after
            z = jnp.dot(h, lm_head_local["q"].astype(h.dtype), preferred_element_type=jnp.float32) * lm_head_local["s"]
            return z.astype(h.dtype)
        return h @ lm_head_local

    def _logits(self, lm_head_local, h):
        return lax.all_gather(self._logits_local(lm_head_local, h), AXIS, axis=-1, tiled=True)

    PREFILL_BUCKETS = (32, 64, 128, 256, 512, 1024, 2048, 4096, 8192)

    def _bucket(self, T):
        for b in self.PREFILL_BUCKETS:
            if b >= T and b <= self.max_len:
                return b
        return T

    def _program(self, params, tokens, caches, pos0, use_recurrent, n_tokens, length=None):
        # pos0 / length are traced int32 scalars; caches have fixed capacity self.max_len
        fetch = None if self.host_experts is None else self.host_experts.make_fetch(self.lcfg, n_tokens)
        if self.expert_fetch is not None:
            fetch = self.expert_fetch
        if self.host_experts is None:   # local on-device tables arrive as [E, D, 2, ml] -> [E, D, 2*ml]
            params = {**params, "layers": [
                {**L, "mlp": {**L["mlp"], "gate_up": L["mlp"]["gate_up"].reshape(L["mlp"]["gate_up"].shape[0],
                                                                                L["mlp"]["gate_up"].shape[1], -1)}}
                if "gate_up" in L["mlp"] else L for L in params["layers"]]}
        from glm53 import quant8 as Q8
        params = {**params, "layers": Q8.dequant_tree(params["layers"], self.lcfg.dtype)}
        emb = self._embed(params["embed"], tokens)
        h, new_caches = M.forward(params, tokens, self.lcfg, caches=caches, pos0=pos0, use_recurrent=use_recurrent,
                                  fetch=fetch, sparse="auto", embeds=emb, cap=self.max_len, length=length)
        last = h[:, -1] if length is None else lax.dynamic_index_in_dim(h, length - 1, axis=1, keepdims=False)
        return self._logits(params["lm_head"], last), new_caches

    def _build(self, B, T, caches, use_recurrent):
        # caches may be None (prefill) — we need their specs for in/out
        def prog(params, tokens, pos0, length, caches_in):
            return self._program(params, tokens, caches_in, pos0, use_recurrent, B * T, None if use_recurrent else length)

        cs = None if caches is None else cache_specs(self.cfg, caches, self.lcfg.seq_shard)
        # discover output cache structure by abstract evaluation is awkward under shard_map; build specs from a
        # dry structure: same layout as inputs, or generated for prefill
        cs_out = [self._cache_spec(i) for i in range(self.cfg.n_layers)] if cs is None else cs
        sm = shard_map(prog, mesh=self.mesh, in_specs=(self.specs, R, R, R, cs), out_specs=(R, cs_out), check_vma=False)
        return jax.jit(sm)

    def prefill(self, tokens: np.ndarray):
        B, T = tokens.shape
        assert T <= self.max_len
        Tp = self._bucket(T)
        padded = np.zeros((B, Tp), np.int32)
        padded[:, :T] = tokens
        key = (B, Tp)
        if key not in self._prefill:
            self._prefill[key] = self._build(B, Tp, None, False)
        logits, caches = self._prefill[key](self.params, jnp.asarray(padded), jnp.int32(0), jnp.int32(T), None)
        return logits, caches, T

    def decode(self, token: np.ndarray, caches, pos: int):
        B = token.shape[0]
        key = B
        if key not in self._decode:
            self._decode[key] = self._build(B, 1, caches, True)
        logits, caches = self._decode[key](self.params, jnp.asarray(token).reshape(B, 1), jnp.int32(pos), jnp.int32(1),
                                           caches)
        return logits, caches, pos + 1

    def generate(self, tokens: np.ndarray, max_new: int, eos=(), greedy=True):
        logits, caches, pos = self.prefill(tokens)
        out = []
        for _ in range(max_new):
            nxt = int(jnp.argmax(logits[0]))
            out.append(nxt)
            if nxt in eos:
                break
            logits, caches, pos = self.decode(np.array([nxt]), caches, pos)
        return out


# ----------------------------------------------------------------------------- segmented engine (no host callbacks)
class SegmentedEngine(Engine):
    """Same sharding as `Engine`, but each layer is its own jitted shard_map program and routed-expert weights are
    gathered on the host *between* programs and shipped with `device_put` (works on TPU runtimes where host
    callbacks inside jit are unavailable). Prefill and decode use the same per-layer programs.

    `transport(arrays: tuple[np.ndarray])` -> tuple of device arrays sharded P('x') on axis 0 (per-chip slices);
    default = numpy device_put. Override for pinned-host tables."""

    def __init__(self, cfg, params, host_experts: HostExperts, devices=None, transport=None, max_len: int = 4096):
        assert host_experts is not None
        super().__init__(cfg, params, host_experts, devices, max_len)
        self.transport = transport or self._default_transport
        self._progs = {}
        from concurrent.futures import ThreadPoolExecutor
        self._pool = ThreadPoolExecutor(max_workers=32)
        from glm53.checkpoint import BLOCK
        self.BLOCK = BLOCK

    CHUNK_BYTES = 320 * 2 ** 20   # GB-sized puts run at ~5 GB/s, ~300 MB strided chunks at ~22 GB/s (measured)

    def _default_transport(self, arrays):
        """arrays: tuple of host [n_dev, Eh, ...] -> tuple of LISTS of device arrays (chunks along axis 1),
        each sharded P('x') on axis 0. Chunks are re-joined on device inside the post program."""
        sh = NamedSharding(self.mesh, P(AXIS))
        out = []
        for a in arrays:
            per_e = a[:, :1].nbytes
            step = max(1, self.CHUNK_BYTES // max(per_e, 1))
            out.append([jax.device_put(a[:, i:i + step], sh) for i in range(0, a.shape[1], step)])
        return tuple(out)

    def _dequant(self, outs, layer):
        from glm53.checkpoint import dequant_fp8_device, dequant_int4_device
        he = self.host_experts
        mode = he.mode_for(layer)
        if mode == "float":
            return outs
        gu_b, gu_s, dn_b, dn_s = outs
        f = dequant_int4_device if mode == "int4" else dequant_fp8_device
        return f(gu_b, gu_s, he.out_dtype), f(dn_b, dn_s, he.out_dtype)

    # ---- per-layer programs (built lazily per (layer, B, T, has_cache))
    def _prog_embed(self, B, T):
        key = ("embed", B, T)
        if key not in self._progs:
            def prog(params, tokens):
                x = self._embed(params["embed"], tokens).astype(self.lcfg.dtype)
                return jnp.broadcast_to(x[:, :, None, :], (B, T, self.cfg.hc, x.shape[-1]))
            self._progs[key] = jax.jit(shard_map(prog, mesh=self.mesh, in_specs=(self.specs, R), out_specs=R,
                                                 check_vma=False))
        return self._progs[key]

    def _prog_pre(self, i, B, T, has_cache, use_recurrent):
        key = ("pre", i, B, T, has_cache, use_recurrent)
        if key not in self._progs:
            lt, mt = self.cfg.layer_types[i], self.cfg.mlp_types[i]
            cs = self._cache_spec(i)
            def prog(params, streams, pos0, length, cache):
                p = params["layers"][i]
                streams, h, post, comb, new_cache, idx, w = M.layer_pre(
                    p, streams, self.lcfg, lt, mt, cache, pos0, use_recurrent, "auto", self.max_len,
                    None if use_recurrent else length)
                if idx is None:
                    idx = jnp.zeros((B * T, 1), jnp.int32); w = jnp.zeros((B * T, 1), jnp.float32)
                return streams, h, post, comb, new_cache, idx, w
            in_specs = (self.specs, R, R, R, cs if has_cache else None)
            out_specs = (R, R, R, R, cs, R, R)
            self._progs[key] = jax.jit(shard_map(prog, mesh=self.mesh, in_specs=in_specs, out_specs=out_specs,
                                                 check_vma=False))
        return self._progs[key]

    def _prog_post(self, i, B, T, Eh=None):
        """Eh: None (dense layer) or (Ep, (n_chunks per table...)) — both fix the program's input structure."""
        key = ("post", i, B, T, Eh)
        if key not in self._progs:
            mt = self.cfg.mlp_types[i]
            n_in = (2 if self.host_experts.mode_for(i) == "float" else 4) if mt == "sparse" else 0
            def prog(params, streams, h, post, comb, w, idx_local, *outs):
                p = params["layers"][i]
                if mt == "sparse":
                    # each table arrives as a list of chunks [1, e_i, ...]; join along the expert axis
                    outs = tuple(jnp.concatenate([c[0] for c in chunks], axis=0) for chunks in outs)
                    gu_w, dn_w = self._dequant(outs, i)
                    return M.layer_post(p, streams, h, post, comb, self.lcfg, mt, gu_w, dn_w, w, idx_local)
                return M.layer_post(p, streams, h, post, comb, self.lcfg, mt)
            in_specs = (self.specs, R, R, R, R, R, R) + (tuple([P(AXIS)] * nc for nc in Eh[1]) if mt == "sparse" else ())
            self._progs[key] = jax.jit(shard_map(prog, mesh=self.mesh, in_specs=in_specs, out_specs=R,
                                                 check_vma=False))
        return self._progs[key]

    def _prog_head(self, B, T):
        key = ("head", B, T)
        if key not in self._progs:
            def prog(params, streams, length):
                h = M.rmsnorm(streams.mean(2).astype(self.lcfg.dtype), params["norm"], self.cfg.eps)
                last = lax.dynamic_index_in_dim(h, length - 1, axis=1, keepdims=False)
                return self._logits(params["lm_head"], last)
            self._progs[key] = jax.jit(shard_map(prog, mesh=self.mesh, in_specs=(self.specs, R, R), out_specs=R,
                                                 check_vma=False))
        return self._progs[key]

    # ---- host side
    BUCKETS = (8, 16, 32, 64, 128, 192, 288)

    def _gather_unique(self, layer, idx_np):
        """Unique experts hit by idx [N,k] -> (host arrays [n_dev, Eh_pad, ...], idx_local [N,k] int32).
        Eh is padded to a bucket so the post program compiles once per bucket; padding experts are unused."""
        tables = self.host_experts.tables[layer]          # chip-major [n_dev, E, ...]
        E = tables[0].shape[1]
        uniq, inv = np.unique(idx_np, return_inverse=True)
        Eh = len(uniq)
        if Eh >= 0.75 * E:                                 # prefill: ship the whole table, no gather at all
            return tuple(tables), idx_np.astype(np.int32), E
        Ep = next(b for b in self.BUCKETS if b >= Eh) if Eh <= self.BUCKETS[-1] else Eh
        outs = [np.empty((self.n, Ep) + t.shape[2:], t.dtype) for t in tables]
        for o in outs:
            o[:, Eh:] = 0                                  # padding experts must not hold NaN-decodable garbage

        def task(j, c, i):                                 # one contiguous memcpy per (table, chip, expert)
            outs[j][c, i] = tables[j][c, uniq[i]]

        list(self._pool.map(lambda a: task(*a), [(j, c, i) for j in range(len(tables)) for c in range(self.n)
                                                  for i in range(Eh)]))
        return tuple(outs), inv.reshape(idx_np.shape).astype(np.int32), Ep

    def _gather_and_put(self, layer, idx_np):
        """Per-chip pipeline: each chip's slices are memcpy'd then immediately device_put (8 chips in parallel),
        so transfer of chip c overlaps the gather of the others. Returns device arrays sharded P('x')."""
        tables = self.host_experts.tables[layer]          # chip-major [n_dev, E, ...]
        N, k = idx_np.shape
        flat = idx_np.reshape(-1)
        devs = list(self.mesh.devices.flat)
        sh = NamedSharding(self.mesh, P(AXIS))

        def chip_task(c):
            pieces = []
            for t in tables:
                buf = np.empty((1, N, k) + t.shape[2:], t.dtype)
                dst = buf.reshape((N * k,) + t.shape[2:])
                src = t[c]
                for i in range(N * k):
                    dst[i] = src[flat[i]]
                pieces.append(jax.device_put(buf, devs[c]))
            return pieces

        per_chip = list(self._pool.map(chip_task, range(self.n)))
        out = []
        for j, t in enumerate(tables):
            shape = (self.n, N, k) + t.shape[2:]
            out.append(jax.make_array_from_single_device_arrays(shape, sh, [per_chip[c][j] for c in range(self.n)]))
        return tuple(out)

    def _run(self, tokens, caches, pos0, use_recurrent, length=None):
        B, T = tokens.shape
        length = T if length is None else length
        tokens = jnp.asarray(tokens)
        streams = self._prog_embed(B, T)(self.params, tokens)
        new_caches = []
        self.stats = {"gather_s": 0.0, "transfer_s": 0.0}
        for i in range(self.cfg.n_layers):
            has_cache = caches is not None
            pre = self._prog_pre(i, B, T, has_cache, use_recurrent)
            streams, h, post, comb, nc, idx, w = pre(self.params, streams, jnp.int32(pos0), jnp.int32(length),
                                                     caches[i] if has_cache else None)
            new_caches.append(nc)
            if self.cfg.mlp_types[i] == "sparse":
                t0 = time.perf_counter()
                idx_np = np.asarray(idx)
                arrays, idx_local, Ep = self._gather_unique(i, idx_np)
                t1 = time.perf_counter()
                dev = self.transport(arrays)
                post_prog = self._prog_post(i, B, T, (Ep, tuple(len(c) for c in dev)))
                streams = post_prog(self.params, streams, h, post, comb, w, jnp.asarray(idx_local), *dev)
                streams.block_until_ready()
                t2 = time.perf_counter()
                self.stats["gather_s"] += t1 - t0
                self.stats["transfer_s"] += t2 - t1
            else:
                post_prog = self._prog_post(i, B, T)
                streams = post_prog(self.params, streams, h, post, comb, w, idx)
        logits = self._prog_head(B, T)(self.params, streams, jnp.int32(length))
        return logits, new_caches

    def prefill(self, tokens: np.ndarray):
        tokens = np.asarray(tokens)
        B, T = tokens.shape
        Tp = self._bucket(T)
        padded = np.zeros((B, Tp), np.int32)
        padded[:, :T] = tokens
        logits, caches = self._run(padded, None, 0, False, length=T)
        return logits, caches, T

    def decode(self, token: np.ndarray, caches, pos: int):
        B = token.shape[0]
        logits, caches = self._run(np.asarray(token).reshape(B, 1), caches, pos, True)
        return logits, caches, pos + 1
'''
FILES["glm53/gguf_reader.py"] = r'''"""Minimal GGUF reader for the Unsloth GLM-5.3-Flash mirrors: header/tensor-map parsing plus positional reads that
work either on whole .gguf files or on the 4 GB byte-range pieces produced by kaggle/glm_mirror_gguf.py
(manifest.json next to the pieces). Reads use pread (no mmap → no page-cache pressure on the TPU host cgroup).

Tensor naming (llama.cpp glm5next): blk.{L}.ffn_{gate,up,down}_exps.weight with GGUF dims [ne0=in, ne1=out, ne2=E]
i.e. row-major numpy shape (E, out, in); quant blocks run along `in`.
"""
import json
import os
import struct
from concurrent.futures import ThreadPoolExecutor

GGML_SIZES = {0: (1, 4), 1: (1, 2), 8: (32, 34), 10: (256, 84), 11: (256, 110), 12: (256, 144), 13: (256, 176),
              14: (256, 210), 16: (256, 66), 17: (256, 74), 18: (256, 98), 19: (256, 50), 20: (32, 18), 21: (256, 110),
              22: (256, 82), 23: (256, 136), 29: (256, 56), 30: (1, 2)}
GGML_NAMES = {0: "F32", 1: "F16", 8: "Q8_0", 10: "Q2_K", 11: "Q3_K", 12: "Q4_K", 13: "Q5_K", 14: "Q6_K", 16: "IQ2_XXS",
              17: "IQ2_XS", 18: "IQ3_XXS", 19: "IQ1_S", 20: "IQ4_NL", 21: "IQ3_S", 22: "IQ2_S", 23: "IQ4_XS",
              29: "IQ1_M", 30: "BF16"}


class PiecewiseFile:
    """A logical file backed by one real file or by ordered byte-range pieces."""

    def __init__(self, pieces):          # pieces: list of (path, start, length) covering [0, size)
        self.pieces = sorted(pieces, key=lambda p: p[1])
        self.size = self.pieces[-1][1] + self.pieces[-1][2]
        self._fds = {}

    @classmethod
    def single(cls, path):
        return cls([(path, 0, os.path.getsize(path))])

    def _fd(self, path):
        if path not in self._fds:
            self._fds[path] = os.open(path, os.O_RDONLY)
        return self._fds[path]

    def pread(self, offset, length):
        out = bytearray(length)
        mv = memoryview(out)
        done = 0
        for path, start, plen in self.pieces:
            if start + plen <= offset + done or start > offset + length:
                continue
            lo = max(offset + done, start)
            hi = min(offset + length, start + plen)
            if hi <= lo:
                continue
            got = os.pread(self._fd(path), hi - lo, lo - start)
            mv[lo - offset:hi - offset] = got
            done = hi - offset
            if done >= length:
                break
        assert done == length, f"short read {done}/{length} at {offset}"
        return out

    def close(self):
        for fd in self._fds.values():
            os.close(fd)
        self._fds.clear()


def open_mirror(dirs):
    """dirs: dataset mount dirs containing manifest.json + pieces (or raw .gguf files). Returns {shard_name: PiecewiseFile}."""
    files = {}
    for d in dirs:
        man = os.path.join(d, "manifest.json")
        if os.path.exists(man):
            m = json.load(open(man))
            for name, info in m["files"].items():
                files[name] = PiecewiseFile([(os.path.join(d, p["name"]), p["start"], p["length"]) for p in info["pieces"]])
        for fn in os.listdir(d):
            if fn.endswith(".gguf"):
                files[fn] = PiecewiseFile.single(os.path.join(d, fn))
    return files


class GGUFShard:
    def __init__(self, pf: PiecewiseFile):
        self.pf = pf
        self.kv, self.tensors, self.data0 = self._parse()

    def _parse(self):
        buf = self.pf.pread(0, min(self.pf.size, 64 << 20))
        pos = [0]

        def rd(fmt):
            v = struct.unpack_from(fmt, buf, pos[0])[0]
            pos[0] += struct.calcsize(fmt)
            return v

        def rstr():
            n = rd("<Q")
            s = bytes(buf[pos[0]:pos[0] + n]).decode(errors="replace")
            pos[0] += n
            return s

        def rval(t):
            if t in (0, 1): return rd("<B" if t == 0 else "<b")
            if t in (2, 3): return rd("<H" if t == 2 else "<h")
            if t in (4, 5): return rd("<I" if t == 4 else "<i")
            if t == 6: return rd("<f")
            if t == 7: return rd("<B")
            if t == 8: return rstr()
            if t == 9:
                et = rd("<I"); n = rd("<Q"); return [rval(et) for _ in range(n)]
            if t in (10, 11): return rd("<Q" if t == 10 else "<q")
            if t == 12: return rd("<d")
            raise ValueError(t)

        assert bytes(buf[:4]) == b"GGUF", "not a GGUF file"
        pos[0] = 4
        rd("<I"); nt = rd("<Q"); nkv = rd("<Q")
        kv = {}
        for _ in range(nkv):
            k = rstr(); t = rd("<I"); v = rval(t)
            if not (isinstance(v, list) and len(v) > 64):
                kv[k] = v
        align = kv.get("general.alignment", 32)
        tensors = {}
        for _ in range(nt):
            name = rstr(); nd = rd("<I"); dims = [rd("<Q") for _ in range(nd)]; ty = rd("<I"); off = rd("<Q")
            n = 1
            for d in dims:
                n *= d
            bs, tb = GGML_SIZES[ty]
            tensors[name] = {"dims": dims, "type": GGML_NAMES.get(ty, ty), "offset": off, "nbytes": n // bs * tb}
        data0 = (pos[0] + align - 1) // align * align
        return kv, tensors, data0

    def read_tensor(self, name, offset=0, length=None):
        t = self.tensors[name]
        length = t["nbytes"] - offset if length is None else length
        return self.pf.pread(self.data0 + t["offset"] + offset, length)


class GGUFModel:
    """All shards of a split GGUF merged into one tensor map."""

    def __init__(self, files):
        self.shards = {n: GGUFShard(pf) for n, pf in sorted(files.items())}
        self.where = {}
        self.kv = {}
        for n, s in self.shards.items():
            self.kv.update(s.kv)
            for t in s.tensors:
                self.where[t] = n

    def info(self, name):
        return self.shards[self.where[name]].tensors[name]

    def read(self, name, offset=0, length=None):
        return self.shards[self.where[name]].read_tensor(name, offset, length)

    def expert_bytes(self, name):
        """bytes of one expert (dims [in, out, E]) = out * in/bs * tb."""
        t = self.info(name)
        return t["nbytes"] // t["dims"][2]

    def read_expert(self, name, e):
        return self.read(name, e * self.expert_bytes(name), self.expert_bytes(name))

    def read_experts(self, name, es, threads=8):
        with ThreadPoolExecutor(threads) as pool:
            return list(pool.map(lambda e: self.read_expert(name, e), es))
'''
FILES["glm53/hf_convert.py"] = r'''"""Convert Glm5Next weights to the JAX param layout of `glm53.model`.

Two sources:
  * `from_hf_module(model)`: a transformers `Glm5NextTextModel` (used for tiny random-init tests).
  * `dequant_fp8(w, scale_inv, block=128)`: helper for real checkpoints (fp8 e4m3 blockwise).
"""
from __future__ import annotations

import numpy as np


def _np(t):
    import torch
    if t.dtype == torch.bfloat16:
        return t.detach().float().cpu().numpy()
    return t.detach().cpu().numpy()


def _lin(t):
    """torch Linear weight [out, in] -> JAX [in, out]."""
    return np.ascontiguousarray(_np(t).T)


def convert_hc(mod):
    return {"fn": _lin(mod.fn), "base": _np(mod.base), "scale": _np(mod.scale)}


def convert_kda(a):
    C = a.qkv_dim
    conv_w = _np(a.conv1d.weight)[:, 0, :]                     # [3C, K]
    return {
        "q": _lin(a.q_proj.weight), "k": _lin(a.k_proj.weight), "v": _lin(a.v_proj.weight),
        "conv_q": conv_w[:C], "conv_k": conv_w[C:2 * C], "conv_v": conv_w[2 * C:],
        "f_a": _lin(a.forget_gate.f_a_proj.weight), "f_b": _lin(a.forget_gate.f_b_proj.weight),
        "dt_bias": _np(a.forget_gate.dt_bias), "A_log": _np(a.forget_gate.A_log),
        "b": _lin(a.b_proj.weight),
        "g_a": _lin(a.g_a_proj.weight), "g_b": _lin(a.g_b_proj.weight),
        "o_norm": _np(a.o_norm.weight), "o": _lin(a.o_proj.weight),
    }


def convert_mla(a):
    p = {
        "q_a": _lin(a.q_a_proj.weight), "q_a_norm": _np(a.q_a_layernorm.weight), "q_b": _lin(a.q_b_proj.weight),
        "kv_a": _lin(a.kv_a_proj_with_mqa.weight), "kv_a_norm": _np(a.kv_a_layernorm.weight),
        "kv_b": _lin(a.kv_b_proj.weight), "o": _lin(a.o_proj.weight),
    }
    if a.indexer is not None:
        ix = a.indexer
        p["indexer"] = {
            "wq_b": _lin(ix.wq_b.weight), "wk": _lin(ix.wk.weight),
            "k_norm_w": _np(ix.k_norm.weight), "k_norm_b": _np(ix.k_norm.bias),
            "weights_proj": _lin(ix.weights_proj.weight),
            "ape": _np(ix.index_kpool_compress_ape), "gate": _lin(ix.index_kpool_compress_gate),
        }
    return p


def convert_mlp(m):
    return {"gate": _lin(m.gate_proj.weight), "up": _lin(m.up_proj.weight), "down": _lin(m.down_proj.weight)}


def convert_moe(m):
    gu = _np(m.experts.gate_up_proj)                              # [E, 2mi, D]
    dn = _np(m.experts.down_proj)                                 # [E, D, mi]
    return {
        "router_w": _lin(m.gate.weight), "router_bias": _np(m.gate.e_score_correction_bias),
        "gate_up": np.ascontiguousarray(np.transpose(gu, (0, 2, 1))),   # [E, D, 2mi]
        "down": np.ascontiguousarray(np.transpose(dn, (0, 2, 1))),      # [E, mi, D]
        "shared": convert_mlp(m.shared_experts),
    }


def from_hf_module(model, lm_head=None):
    """`model`: transformers Glm5NextTextModel. Returns nested dict of numpy arrays."""
    layers = []
    for L in model.layers:
        p = {
            "ln1": _np(L.input_layernorm.weight), "ln2": _np(L.post_attention_layernorm.weight),
            "hc_attn": convert_hc(L.attn_hc), "hc_ffn": convert_hc(L.ffn_hc),
        }
        p["attn"] = convert_kda(L.self_attn) if L.block_type == "linear_attention" else convert_mla(L.self_attn)
        p["mlp"] = convert_moe(L.mlp) if hasattr(L.mlp, "experts") else convert_mlp(L.mlp)
        layers.append(p)
    params = {"embed": _np(model.embed_tokens.weight), "norm": _np(model.norm.weight), "layers": layers}
    if lm_head is not None:
        params["lm_head"] = _lin(lm_head.weight)
    return params


def dequant_fp8(w_fp8: np.ndarray, scale_inv: np.ndarray, block: int = 128) -> np.ndarray:
    """fp8 e4m3 weight [O, I] (as float32/np) with per-block scales [ceil(O/b), ceil(I/b)] -> float32 [O, I]."""
    O, I = w_fp8.shape
    so, si = scale_inv.shape
    s = np.repeat(np.repeat(scale_inv.astype(np.float32), block, axis=0), block, axis=1)[:O, :I]
    assert s.shape == (O, I), (s.shape, O, I, so, si)
    return w_fp8.astype(np.float32) * s
'''
FILES["glm53/iq_grids_data.py"] = r'''"""Auto-generated: llama.cpp IQ codebook grids (from gguf-py), npz base64 — so the package needs no data files."""
import base64, io
import numpy as np

_B64 = (
    "UEsDBC0AAAAIAAAAIQB/XAco//////////8JABQASVEyX1MubnB5AQAQAIAgAAAAAAAAdQcAAAAAAACdmb2OXkUMhkNncQfutluQ"
    "UxBEgbgAOhANBRWKyCKQIoI2iAa4Cm6YPWM/79iTLynY1ZH32z0/Y/v98Zz999vvv/nuh4+e/fnsr/tXD29/erz/6u7+719f3D+/"
    "u//5zeMfjy9/+/HN46uH6/dfv3z99uHp929/efn7w9PnT1589vkXz+++/PT53T93/+/rY6uvqOg+fxEV3XycEBU95gVR0X3ewK1u"
    "UDeKik+njRtHRff5ILcYD4yKHnMBUfHpqrEgN7+5sKePeV3U5zpYcFS8ruoJRMVrmT0ht7iZWFS8Tu+JuvlIOCp6zAK4VyGiPlsV"
    "5ChMVHSvaHGzUBzXzXrh3PxmAa/yRJWlFzQqXun2Al9p9EJft8v7zMKHZeGpufXkeFhrSFRc1WkNWqttjYqK6ymtcVFxta01Miq6"
    "+WhsVFzlb42OiqvMbWkAICp6TECssuS6K5Wo62IAht5wAKCo6OYDUFFxLacBbD3Os/DeSgfwouI6rQExKrpPYLrFTYBGRfcCbNRn"
    "85sApnX0msNjAtt9Atx9At3NJ+BLGVapPkAAME9tOQCJ9aSiHt6IstAdyYuCnncCRcXV5UaoqLiqGVW9qGyjsmzEW3+NolWUTjWo"
    "n4SMigu2jaALjo2gCy5RMImCwyBueKfSWlgjtJsPYofFIHgcRL+u7lS01mOJSRMAyARYAQtFJvk4BIJW+GAlbNnCEd7RuYUkrKMK"
    "FG3psNGtLTjuU3i8qnQKEBDxcVdXTflGoKJiyswWrKiYcrIFjCspQVRMGdjCFhWT3lvoeOIpeEA66bgFkJUhhKyQ0oNJepqw3xmc"
    "Apmw3ZkhmGRo7xFMKIYmwmEqwoFIISaIAKRAYKmg9aS1uC28UTHLuIWYyp+CjARwaZZhd+gUbDoGdNFEOG69pr4PhJ1OnwKPBLF0"
    "BB9EnMIPQkSdwwBIHc0ESachgCxR7zCGS9868qBm+NaybhzkygFSERfISktWoRqSt8HEQLakoEmyt5biCTDgNCYYISkpg7pcKO8X"
    "HzSqq7jdsLAEoIQXwThrNZY5NmND7KOmLhgqEepglPu/a3zpPtsAowwQhksih0rFxqJNQ8TiRAnr6NwGmd19v1HKg+o7Fxo1t0ZZ"
    "TDfSOkMp5ZkoUUlMQS005OI9cIqhBbPXsZ+4xbcZNGJyGnUWvxv2nKyRCFqAUp6GjnJmQbexx6LtNvhEfjd6PDUN3xdct/FLmmxr"
    "ZVfmbPceDHaR8wPKbVjjKkckO/rsAFYZHPYtVkT5bVOUElk1SZ6C5qEJcKw7h7WcrbrfkYAU205hmwAivEURUXMTgua2DNKJLDam"
    "Lw02FFPF6ic3a4B6OB4ehuYe2xRc14UV6263pQ7I8SiQj0vgCqj9OVDBENQWdUU1pZI1aKFmUicxa6jC1nY4ZoNNsEfOLjYU2kGv"
    "UDtQsK3VZvU1GdjMVlpvc1XiIE/jar6xEKhMC1EMTBlTxbzOgRBlQUSRXsRGTyjyAl7rzWhSYv1i37MCXopmn4MlPeBA+rmFMqqB"
    "c7m9a3zYmR4DKApZdq+BlCWpMgkwYViVEgUW0Ja79smMSiKhUAVFZkZihmCSOwdcsHfY3Wlv2Nnu1DEIo/yQDhJQcjWnFglX1WlR"
    "27qMn/KNbHdN6zKqkRHLptUgiBnukDV5Ld6DJ6AlNnryjgwpJYZRhkmf8oFcyBEZHhgtgCTIRqxBOCLkk54qJaAFZDTx3AjwELBO"
    "zzn0p5qWfcK3Se6AmzYMUEm3rlkNTRYTq3aXwfWNhM/2aST12RZBB5VHvX2WkzJKW32WRznrKPb6TKdZyliOTIau+rytSOnzz23x"
    "uXHhBymO1Wy9HDnz6BsZJiMuZATlBmphCpBuKOUqrOZmpvDZHiRlS2HqJFoboDXV5vOjb4hYIIrIuy7bFB8LZ09DAufGiYTQIhJT"
    "r97ZUGHlmTBDCgXAlCmEoF2iSWFEJoErF+srpZsbsDa1VkHaxozCsiWgwKJUCqcKLkeQhlj0RkiVpA7JFkZUmoSkqhvJODVSTpNC"
    "q2tX6dpGb82ykTNJb3jqsP79wr9btBEECEgGS2GvJiQbzmbSSjShAwas0hOOE0BlDQISTslwxRCExQA0WgvgZF5lGsxkABGRA5CQ"
    "nkPgqiannVfyjaH1elZARlIBtBy8PASA4+hszQG8dctsBACyECH1cc+YEAPPhyDSwloiuemQ+3cX3YTC7SCWLKJShWhSZ6lqI1Mj"
    "oppUyXiyLRrLGlo2Yeka1VC2tUp2ghA6abqHhSWCbbuYGpoaOF8pxxAAKyaxfbMxymyB0JOZrLKR2okiHKwMAZFHFWTAnHqjYyt6"
    "FxqsGcHR8FBmWuOvhEjkMMbJXDRKkWm7Xq0jXFQSAdMoUNBG0DTeaGxJTufLgC14Ggdk82XPJcUIIWITmfySxS6MsqdSichSYh+Z"
    "VnvhUKvQG4GSRWRQAst/bZEz/Zey/gmB8EqOSmYEpVQGDadZSt6O8VZrv4kAmUG0W2839o8IvOAouC04RCIt9lFtlZVUGSqdvex6"
    "RnYjylH2KyydNlfbllan1dd/UEsDBC0AAAAIAAAAIQCfBHCG//////////8JABQASVEzX1MubnB5AQAQAIAIAAAAAAAA6QMAAAAA"
    "AACdlc2KHUcMhacRhZCF0Cvc3Y1hNmOcjR/Au4RsvMjKDPGYBIwnzJhs4jxFXjjfUbXJ3rc5VHd1tf7Oke6/P7/76Zdfj5u/bv6+"
    "fnh4/u3p+uZy/frH3fX2cv34+PTl6f7z+8enDw/af3v/6fmB/eff7/984PmHH+9e3V5ev7y9/HP5zt+Lg5+BBRL0uWFggQB1HjCQ"
    "wJ3nYAskaGB87DJQ7APDUIA+DepaIE7jBvJ04qCBOXsgcGo4TbB6O9fXCfoMxEEBW+wvBbWOACt4Bg2MYF0BN6vAZeb4JYHlxwJG"
    "Mg4KRLCf7IMu3jVneyfrFkcDI+kAy0k62Ad2FkERhqq4iBQUcIpikcSUHGGv2GtWLrPCLqsXn7B2DQPKMsTE4h4UcOc52c9dUNt0"
    "WJzFtV0+S9BnsSmL0jXSMqWDe8O93BomhwjbR7Q1VgLY4t7Z970hopbIas71Jk1XggYEaQGWiIxlDjK5L/Z6G1MFyWKi1MVha0Bi"
    "xMdzwGjx3LxFdVQSyzEXBomFZ4RREkeymzswZey2A4RszHKf3K8ipsIHaxTf1SQAs1MhRU5wls4aPJMYYh2x2ahDLIqBLT5lGWNV"
    "yfMeICmJSjIi1JHEyi0D0b+gcwQLdStP4YqWY7KXd56xliom6asEK87CYiKGfZ3ABUDjq30XG12vBiS4oncD6ECqm8znixAh7gt9"
    "c4iI01fVJgklTbPoFIEjeUgL7mmcbCmcqGggEsEnTRSJT9ZWQxW+apRJ4VYma0ntIpuKr91wTmBodoKzYVoM8cZ2sLZbz2k3wphW"
    "8pZQ1Db1v2DUpcjWpQSbjEYiNC2JQmn1bmgpUGzoIl2adwvsGPXgfirhimw8S3gcxCddmty3wokRI6qZopksJR1U4QE0odAcvpKq"
    "sBJ9RHrJQLPSaRZbyGiNeHpYQ2czSDQ1RABqoNqHKgQJZBMzOtT2mlyMzmnFCUaMq4VSEashknsgSbqmMNJBMiH1uO8BBU3In/te"
    "M6XUREpGKhXhKgkRzxBTKHwSLQa4tY6ZWK1GI/lVFECd5xUMMPwxyayHLTgmHk1bop/OnHE8gxBrvOEZY1XboBpWBUxNZxuli6VU"
    "K6rl1Mi+RsaJVKfImuJ9jARQk2MXGfGGAyorNpkGgFJgK5JIUxNIymTRSapRI96QgOkc9CiSio/L9vivuRiRdHOhZI2METgKKFUT"
    "K6WLmnKOYiA9NFVIpuSA8vE2eEdkgDCxmKWpf0QxgPhXOYd87L/GGfbUS383M/T1NzEeNKT4ALFAixG4xvrSv0xrepfGh8aDOiC/"
    "NZwU6CM2asK/DJRKSmuoxA/l4w+E3p2uUQj/AVBLAwQtAAAACAAAACEAuFJVlf//////////CgAUAElRMl9YUy5ucHkBABAAgBAA"
    "AAAAAADaAwAAAAAAAJ1XMW4UQRA02YgfVHbZgcqJkZAQDyADkRAQIQsfAglhdEYkwCv4MLfdXT09s7sJZ63bt7sz011dVTP+++bd"
    "67fvH139vPp1vDs9fDwfXx6Ov7/cHK8Px0/35x/n228f7s93p+X+q9uvD6fL/YfPt99Pl+9Pnt88uz68eHp9+HP4z8/jFh9GBMYb"
    "jIiG4QVGBMcBjAiME6DFBDERI15eGyZmRGBcCI2bCzLi5a0hATQMiVyGDQkhLiXGiMtbNVFGXNKpiTMiGjcLYcRlWC0MDdsFkvE8"
    "Ct0pWNfycgUADZtAoAUQEyDL8hWQyy+yYNJqMpqkAMaIVkUB0FbfAJIRDcYCLBoGgDkBraX2gBYWugQ8GjYboJTnRsBXzoaopLkh"
    "C41qY6gGwSiRjVJvlJMuRhQocyOtK+xQrhrLmemkz+tR0O81PkpoVsagCA5EuCzDuG9RrWwFiz53J4yahYHNyGJmIqlUDN1W99YE"
    "I4lKqT3CCToMb/efmYi6syJkEhEWGVEjlTJ3icqBsGqluCkOKIOZyMpIxdhkldiWTie4Mt8lehLcarH5WCqdic9ZALCB/qgIQakQ"
    "iRDMIxf6FGGIukq5Ve6l266FI6R3heOszQ7sCogegVFI+pkFpU7OwgK5KTBHyEZahAOSTKBjtFwcBFhNWCZBnyWe0zOTQE2hjn6M"
    "Z4xjqcC/xih2JrbeOqU+CFyVizvigLCchS+EREG1IpZMxs/GgHBPNXXPKHRT1FaLpZzZQHTtGQmCzd6RrrjZWBgRo72m0QgNVacq"
    "lJWywGhbmdu+IY1KXxkUBrkvWqsydkrE3NXA5BQrA4ueAdE7uEyYJtBySdQMne4m9rjfCs1SGbIgSVGpr4yQcToZ27syRowwJ9g+"
    "czdI9kVaSd9tqThjIhklYVzG+epldFFWY2VWyfhOlD+aNwi+gxUj9qnzCCJjhptEeGjUw6hnY6LsfFK8Zc791EKz6L67F2M3zzIr"
    "iaOBMT5dgjL4JLNr1yqAMw7oZ1kzfUuQQc7mezcQzhfvdAcUIt4qjoXNG4YOJyo4GT1tHHJWmWMMTICkAEoJKdKGEUBMGw6HjYe5"
    "VnXdDnAqLKwlFs7Od/eIY4ZvPn3jCkW60HzvNYSpnjZtZM0ppyz8bT3VThKbBNMEtb00PfEYTGEgl4RIR0hLdJuOgmSbuVl4OXna"
    "SGb42n0DFdPTYSRj14oT1O/bFIzo7AyLozHOKOZiR0qwxUPRjqLfSKtsrWijK9qQM9ps5QQtsrUkgWDvG3f+q4UiEEvSczZlqBPl"
    "LJJDGfmwXI6+H0X9yEWUjT9eyxOZPv8AUEsDBC0AAAAIAAAAIQBJNKzI//////////8LABQASVEzX1hYUy5ucHkBABAAgAQAAAAA"
    "AAAaAgAAAAAAAJ2TMavUMRDEn5BKttxyiysWTmEbZbWwSGmn2FhYycN3oiA+uRMb9VP4hf1NcmBvwpD/7SXZ2dnJn9dvX7159+Dm"
    "x83P493p8uF8fHE4/vr85FiH48f78/fz7df39+e7k+Ivb79cTsQvn26/nfj96Omz53Xox3X4ffjP8XAwHCQwGyPAtB10YMHKnzHH"
    "KGDMANrswNxG+T7owNJG1r9LNBOY7QsdWPgIMIEVK6jp6+YJlNRBOd8zuDpZcmSyVsIl2Z/sL/YXeUHVqOiR3ZyfI0GDyDms52Ic"
    "m74oi45orGJIYwmuV67ixFhDRbrvD7HSaU0FR/A1d/F+FcAirBM037AoMAp4WYJGHJtFvLkCkXILJXVsS7hEk1TtOyn6r8SiCr0l"
    "phhoKnVw1QRovkTWcVu3b8E10JpGODJt4qo+bTchQUUiI0D16XQ0i7g6O4lNR0uv3MWOVfkuWq1Es6gmdm2eXxuomSRpoNIILrWh"
    "QEIwt2AaNIq2JjE6sTpZMK6ogpl1aAaiRmCIlhuWclnBCrAAdLHIxCy+SsIwoB1L8JsEldBhixVO6qRCRWqZCielHOByQWgdNOxq"
    "MFzaght0OG2L/RaK0jVxEIQTG3PtTDm6LGTKRtW5xEwfZB3dIFsGsNakT/x2FFwVttjpIM3aRY+5OqzXAJnJM5Shd2NgCK9leLlR"
    "ycRMLxYCU4KNDlJImVyvWK8NLyJJz79QSwMELQAAAAgAAAAhACXOngv//////////woAFABrc2lnbnMubnB5AQAQAAABAAAAAAAA"
    "zQAAAAAAAACb7BfqGxDJyFDGUK2eklqcXKRupaBeU2qorqOgnpZfVFKUmBefX5SSChJ3S8wpTgWKF2ckFqQC+RqGRhY6mjoKtQpk"
    "Ay6GxibmFla29g5Orm6e3j7+CYJCk0WmThOXmDlLeo6s3PwFikqLVZYuU9dYuUp7ja7eeoONm4y3mJpt32Fptdtm7z77A45Oh12O"
    "HnP3OHnK+4yv3/mAi5eCr4SGXb8RGXU75u69+ISHj5KfpKY9f5GZ9Trn7bv8D4VFn0u+fiuv+Pmr+k9t3X8AUEsDBC0AAAAIAAAA"
    "IQC18arc//////////8RABQAa3ZhbHVlc19pcTRubC5ucHkBABAAkAAAAAAAAABWAAAAAAAAAJvsF+obEMnIUMZQrZ6SWpxcpG6l"
    "oF6Taaiuo6Cell9UUpSYF59flJIKEndLzClOBYoXZyQWpAL5GoZmOpo6CrUK5AOuxhlr95+/++obI6+kmqlrZCEAUEsBAi0DLQAA"
    "AAgAAAAhAH9cByh1BwAAgCAAAAkAAAAAAAAAAAAAAIABAAAAAElRMl9TLm5weVBLAQItAy0AAAAIAAAAIQCfBHCG6QMAAIAIAAAJ"
    "AAAAAAAAAAAAAACAAbAHAABJUTNfUy5ucHlQSwECLQMtAAAACAAAACEAuFJVldoDAACAEAAACgAAAAAAAAAAAAAAgAHUCwAASVEy"
    "X1hTLm5weVBLAQItAy0AAAAIAAAAIQBJNKzIGgIAAIAEAAALAAAAAAAAAAAAAACAAeoPAABJUTNfWFhTLm5weVBLAQItAy0AAAAI"
    "AAAAIQAlzp4LzQAAAAABAAAKAAAAAAAAAAAAAACAAUESAABrc2lnbnMubnB5UEsBAi0DLQAAAAgAAAAhALXxqtxWAAAAkAAAABEA"
    "AAAAAAAAAAAAAIABShMAAGt2YWx1ZXNfaXE0bmwubnB5UEsFBgAAAAAGAAYAVgEAAOMTAAAAAA=="
)


def load():
    return np.load(io.BytesIO(base64.b64decode("".join(_B64))))
'''
FILES["glm53/iqquant.py"] = r'''"""Dequantization of llama.cpp "IQ" codebook formats (as used by Unsloth's UD-IQ3_XXS / UD-Q2_K_XL GLM-5.3-Flash GGUFs)
written in jax.numpy so the same code runs on CPU (tests vs gguf-py) and on TPU (inside the resident engine).

Block = 256 weights (QK_K). Inputs are the raw GGUF block bytes, uint8 [..., block_bytes]; output float [..., 256].
Layouts (bytes): IQ2_S 82 = d2|qs32|signs32|qh8|scales8 · IQ3_S 110 = d2|qs64|qh8|signs32|scales4 ·
IQ4_XS 136 = d2|sh2|sl4|qs128 · IQ2_XS 74 = d2|qs64(u16)|scales8 · IQ3_XXS 98 = d2|qs64|scales32(u32).
The grid lookup is `jnp.take(grid, idx)`; `LOOKUP` can be swapped for a faster TPU implementation.
"""
import os
import numpy as np
import jax
import jax.numpy as jnp

QK_K = 256
BLOCK_BYTES = {"IQ2_S": 82, "IQ3_S": 110, "IQ4_XS": 136, "IQ2_XS": 74, "IQ3_XXS": 98, "Q2_K": 84, "Q3_K": 110}
from glm53 import iq_grids_data as _gd
_g = _gd.load()
GRIDS = {k: np.asarray(_g[k]) for k in ("IQ2_S", "IQ3_S", "IQ2_XS", "IQ3_XXS")}   # int8 [entries, 8 or 4]
KSIGNS = np.asarray(_g["ksigns"])                                                  # uint8 [128]
KVALUES_IQ4NL = np.asarray(_g["kvalues_iq4nl"])                                    # int8 [16]


def _u16(b):   # uint8 [..., 2k] -> uint16 [..., k] (little endian)
    return jax.lax.bitcast_convert_type(b.reshape(b.shape[:-1] + (-1, 2)), jnp.uint16)


def _u32(b):
    return jax.lax.bitcast_convert_type(b.reshape(b.shape[:-1] + (-1, 4)), jnp.uint32)


def _f16(b):   # uint8 [..., 2] -> float32 [...]
    return jax.lax.bitcast_convert_type(b, jnp.float16).astype(jnp.float32)


def _bits8(b):   # uint8 [..., n] -> [..., n, 8] bit k of each byte (LSB first)
    return (b[..., None] >> jnp.arange(8, dtype=jnp.uint8)) & 1


def _nibbles(b):  # uint8 [..., n] -> [..., 2n] low nibble first
    return jnp.stack([b & 0x0F, b >> 4], -1).reshape(b.shape[:-1] + (-1,))


# ---------------------------------------------------------------------------------------------------- grid lookup
# XLA's gather is ~0.1 G lookups/s on v5e (measured) — useless. The "tree" implementation is gather-free: every grid
# entry is encoded as level indices (2 or 3 bits per value), bit-sliced into 32-entry uint32 words, and each output bit
# is fetched with a binary tree of `where`s over the word index (n_words-1 selects) + one variable shift.
LOOKUP_IMPL = "tree"          # "take" (jnp.take) or "tree"
LEVELS = {"IQ2_S": np.array([8, 25, 43], np.int8), "IQ2_XS": np.array([8, 25, 43], np.int8),
          "IQ3_S": np.array([1, 3, 5, 7, 9, 11, 13, 15], np.int8),
          "IQ3_XXS": np.array([4, 12, 20, 28, 36, 44, 52, 62], np.int8)}


class _TreeLUT:
    def __init__(self, name):
        grid, levels = GRIDS[name], LEVELS[name]
        self.levels = levels
        self.w = grid.shape[1]
        self.lb = int(np.ceil(np.log2(len(levels))))                       # bits per level index (2 or 3)
        n = grid.shape[0]
        lv = np.searchsorted(levels, grid)                                 # [n, w] level indices
        assert np.array_equal(levels[lv], grid)
        code = np.zeros(n, np.int64)
        for k in range(self.w):
            code |= lv[:, k].astype(np.int64) << (self.lb * k)
        self.nbits = self.lb * self.w
        self.n_words = n // 32
        # words[b][j] = 32 bits (entries 32j..32j+31) of output bit b
        self.words = np.zeros((self.nbits, self.n_words), np.uint32)
        for b in range(self.nbits):
            bits = ((code >> b) & 1).astype(np.uint32)
            self.words[b] = (bits.reshape(self.n_words, 32) << np.arange(32, dtype=np.uint32)).sum(1, dtype=np.uint32)

    def __call__(self, idx):
        wi = idx >> 5
        sh = (idx & 31).astype(jnp.uint32)
        bits = [((wi >> k) & 1).astype(bool) for k in range(int(np.log2(self.n_words)))]
        code = jnp.zeros(idx.shape, jnp.uint32)
        for b in range(self.nbits):
            nodes = [jnp.uint32(int(x)) for x in self.words[b]]
            for k, bit in enumerate(bits):                                 # select tree over word index bits
                nodes = [jnp.where(bit, nodes[2 * j + 1], nodes[2 * j]) for j in range(len(nodes) // 2)]
            word = nodes[0] if len(nodes) == 1 else jnp.broadcast_to(jnp.uint32(int(self.words[b][0])), idx.shape)
            code = code | (((word >> sh) & 1) << b)
        mask = (1 << self.lb) - 1
        lv = jnp.stack([((code >> (self.lb * k)) & mask).astype(jnp.int32) for k in range(self.w)], -1)   # [..., w]
        levels = self.levels
        out = jnp.full(lv.shape, int(levels[0]), jnp.int8)
        for i in range(1, len(levels)):
            out = jnp.where(lv == i, jnp.int8(int(levels[i])), out)
        return out


_TREES = {}


def lookup(name, idx):
    """name: grid name; idx int32 [...] -> int8 [..., w] grid values."""
    if LOOKUP_IMPL == "take":
        return jnp.take(jnp.asarray(GRIDS[name]), idx, axis=0)
    if name not in _TREES:
        _TREES[name] = _TreeLUT(name)
    return _TREES[name](idx)


def _signs_from_bits(signs_u8):   # uint8 [..., 32] -> float32 [..., 256] of ±1
    s = _bits8(signs_u8).reshape(signs_u8.shape[:-1] + (QK_K,))
    return 1.0 - 2.0 * s.astype(jnp.float32)


def dequant_iq2_s(blk):
    d = _f16(blk[..., 0:2])                                   # [...]
    qs = blk[..., 2:34].astype(jnp.int32)                     # [..., 32] low 8 bits of grid index (32 groups of 8)
    signs = _signs_from_bits(blk[..., 34:66])                 # [..., 256]
    qh = blk[..., 66:74]                                      # [..., 8] 2 bits per group, 4 groups per byte
    sc = _nibbles(blk[..., 74:82]).astype(jnp.float32)        # [..., 16] one 4-bit scale per 16 weights
    hi = jnp.stack([(qh >> (2 * k)) & 3 for k in range(4)], -1).reshape(qh.shape[:-1] + (32,)).astype(jnp.int32)
    idx = qs | (hi << 8)                                      # [..., 32]
    g = lookup("IQ2_S", idx).astype(jnp.float32).reshape(idx.shape[:-1] + (16, 16))  # 2 groups per scale
    db = d[..., None, None] * (0.5 + sc)[..., None] * 0.25
    return (db * g).reshape(idx.shape[:-1] + (QK_K,)) * signs


def dequant_iq3_s(blk):
    d = _f16(blk[..., 0:2])
    qs = blk[..., 2:66].astype(jnp.int32)                     # [..., 64] low 8 bits (64 groups of 4)
    qh = _bits8(blk[..., 66:74]).reshape(blk.shape[:-1] + (64,)).astype(jnp.int32)   # high bit per group
    signs = _signs_from_bits(blk[..., 74:106])
    sc = _nibbles(blk[..., 106:110]).astype(jnp.float32)      # [..., 8] one 4-bit scale per 32 weights
    idx = qs | (qh << 8)
    g = lookup("IQ3_S", idx).astype(jnp.float32).reshape(idx.shape[:-1] + (8, 32))   # 8 groups of 4 per scale
    db = d[..., None, None] * (1.0 + 2.0 * sc)[..., None]
    return (db * g).reshape(idx.shape[:-1] + (QK_K,)) * signs


def dequant_iq4_xs(blk):
    d = _f16(blk[..., 0:2])
    sh = _u16(blk[..., 2:4])[..., 0]                          # [...] 2 bits per 32-block (8 blocks)
    sl = _nibbles(blk[..., 4:8]).astype(jnp.int32)            # [..., 8]
    qs = blk[..., 8:136]                                      # [..., 128] two 4-bit values per byte, low nibble first
    shb = jnp.stack([(sh >> (2 * k)) & 3 for k in range(8)], -1).astype(jnp.int32)   # [..., 8]
    scales = (sl | (shb << 4)) - 32                           # [..., 8]
    q = qs.reshape(qs.shape[:-1] + (8, 16))                   # 8 sub-blocks of 16 bytes = 32 weights each
    q = jnp.concatenate([q & 0x0F, q >> 4], -1).astype(jnp.int32)   # [..., 8, 32]: first 16 low nibbles then 16 high
    v = _kv16(q)
    out = d[..., None, None] * scales.astype(jnp.float32)[..., None] * v
    return out.reshape(qs.shape[:-1] + (QK_K,))


def dequant_iq2_xs(blk):
    d = _f16(blk[..., 0:2])
    qs = _u16(blk[..., 2:66]).astype(jnp.int32)               # [..., 32]: 9-bit grid index | 7-bit sign index << 9
    sc = _nibbles(blk[..., 66:74]).astype(jnp.float32)        # [..., 16]
    sidx = qs >> 9
    sbits = jnp.take(jnp.asarray(KSIGNS), sidx, axis=0)       # uint8 [..., 32]
    signs = 1.0 - 2.0 * _bits8(sbits).astype(jnp.float32)     # [..., 32, 8]
    g = lookup("IQ2_XS", qs & 511).astype(jnp.float32) # [..., 32, 8]
    db = (d[..., None] * (0.5 + sc) * 0.25)                    # [..., 16]
    out = (g * signs).reshape(qs.shape[:-1] + (16, 16)) * db[..., None]
    return out.reshape(qs.shape[:-1] + (QK_K,))


def dequant_iq3_xxs(blk):
    d = _f16(blk[..., 0:2])
    qs = blk[..., 2:66].astype(jnp.int32)                     # [..., 64] 8-bit grid index (64 groups of 4)
    sc = _u32(blk[..., 66:98])                                 # [..., 8]: 4 x 7-bit sign idx | 4-bit scale << 28
    db = d[..., None] * (0.5 + (sc >> 28).astype(jnp.float32)) * 0.5     # [..., 8] one scale per 32 weights
    sidx = jnp.stack([(sc >> s) & 0x7F for s in (0, 7, 14, 21)], -1).astype(jnp.int32)   # [..., 8, 4]
    sbits = jnp.take(jnp.asarray(KSIGNS), sidx, axis=0)                                 # [..., 8, 4] uint8
    signs = 1.0 - 2.0 * _bits8(sbits).astype(jnp.float32)                               # [..., 8, 4, 8]
    g = lookup("IQ3_XXS", qs).astype(jnp.float32).reshape(qs.shape[:-1] + (8, 4, 8))   # 8 x (4 groups x 4... see below)
    # each 32-weight sub-block = 8 groups of 4 = the [8 groups] x [4 values]; regroup to match sign layout (4 sign-bytes x 8)
    g = g.reshape(qs.shape[:-1] + (8, 32)); signs = signs.reshape(qs.shape[:-1] + (8, 32))
    return (db[..., None] * g * signs).reshape(qs.shape[:-1] + (QK_K,))


# ------------------------------------------------------------------------------------ K-quants (the MTP layer's experts)
def _f16_at(blk, off):
    """f16 at byte offset `off` of every block -> f32 [..., nblk]."""
    u = blk[..., off].astype(jnp.uint32) | (blk[..., off + 1].astype(jnp.uint32) << 8)
    return jax.lax.bitcast_convert_type(u.astype(jnp.uint16), jnp.float16).astype(jnp.float32)


def _kq_layout(q):
    """q u8 [..., nblk, 64] -> list of 8 (n, s) groups of 32 2-bit values [..., nblk, 32] in ggml order
    (weight index 128 n + 32 s + l; byte q[32 n + l] shifted by 2 s)."""
    q = q.astype(jnp.int32)
    return [((q[..., 32 * n:32 * (n + 1)] >> (2 * s)) & 3) for n in range(2) for s in range(4)]


def q3k_scales(sc12):
    """Q3_K 12 packed bytes -> 16 six-bit scales (uint8 [..., 16]) in `is` order (ggml `dequantize_row_q3_K`)."""
    sc = np.asarray(sc12).astype(np.uint32)
    aux = [sc[..., 4 * i] | (sc[..., 4 * i + 1] << 8) | (sc[..., 4 * i + 2] << 16) | (sc[..., 4 * i + 3] << 24) for i in range(3)]
    k1, k2 = np.uint32(0x03030303), np.uint32(0x0F0F0F0F)
    tmp = aux[2]
    a2 = ((aux[0] >> 4) & k2) | (((tmp >> 4) & k1) << 4)
    a3 = ((aux[1] >> 4) & k2) | (((tmp >> 6) & k1) << 4)
    a0 = (aux[0] & k2) | (((tmp >> 0) & k1) << 4)
    a1 = (aux[1] & k2) | (((tmp >> 2) & k1) << 4)
    words = np.stack([a0, a1, a2, a3], axis=-1)                                   # [..., 4] u32
    return np.stack([(words >> (8 * b)) & 255 for b in range(4)], axis=-1).reshape(words.shape[:-1] + (16,)).astype(np.uint8)


def dequant_q2_k(blk):
    """blk uint8 [..., nblk, 84] -> float32 [..., nblk, 256] (natural order)."""
    sc = blk[..., 0:16].astype(jnp.int32)
    d, dmin = _f16_at(blk, 80), _f16_at(blk, 82)
    outs = []
    for g, v in enumerate(_kq_layout(blk[..., 16:80])):                          # g = 4 n + s
        for h in range(2):                                                        # l < 16 / l >= 16
            b = sc[..., 2 * g + h]
            dl, ml = d * (b & 15).astype(jnp.float32), dmin * (b >> 4).astype(jnp.float32)
            outs.append(dl[..., None] * v[..., 16 * h:16 * (h + 1)].astype(jnp.float32) - ml[..., None])
    return jnp.concatenate(outs, axis=-1)


def dequant_q3_k(blk):
    """blk uint8 [..., nblk, 110] -> float32 [..., nblk, 256] (natural order)."""
    hm = blk[..., 0:32].astype(jnp.int32)
    sc = jnp.asarray(q3k_scales(np.asarray(blk[..., 96:108]))).astype(jnp.int32) - 32
    d = _f16_at(blk, 108)
    outs = []
    for g, v in enumerate(_kq_layout(blk[..., 32:96])):
        hbit = (hm >> g) & 1                                                      # bit 4 n + s of hmask byte l
        val = v - 4 * (1 - hbit)
        for h in range(2):
            dl = d * sc[..., 2 * g + h].astype(jnp.float32)
            outs.append(dl[..., None] * val[..., 16 * h:16 * (h + 1)].astype(jnp.float32))
    return jnp.concatenate(outs, axis=-1)


DEQUANT = {"IQ2_S": dequant_iq2_s, "IQ3_S": dequant_iq3_s, "IQ4_XS": dequant_iq4_xs,
           "IQ2_XS": dequant_iq2_xs, "IQ3_XXS": dequant_iq3_xxs, "Q2_K": dequant_q2_k, "Q3_K": dequant_q3_k}


def dequant_rows(raw, qtype, ne0):
    """raw uint8 [nbytes] of a row-major GGUF tensor slice with ne0 weights per row -> float32 [rows, ne0]."""
    bb = BLOCK_BYTES[qtype]
    blk = jnp.asarray(raw).reshape(-1, ne0 // QK_K, bb)
    return DEQUANT[qtype](blk).reshape(blk.shape[0], ne0)


# ------------------------------------------------------------------------------------ position-major ("pm") dequant
# TPU pads the last two dims of every materialised array to (8,128) tiles, so [..., groups, 8] int8 intermediates cost
# 16x their size. Instead each codebook *position* is produced as its own lane-dense [..., n_groups] array and the row is
# emitted in position-major order: flat index = pos * G + group, G = groups per row (all blocks). The contraction dim of a
# matmul does not care about order as long as the activation is permuted the same way: x_pm = x[..., perm_pm(...)].
GROUP_W = {"IQ2_S": 8, "IQ2_XS": 8, "IQ3_S": 4, "IQ3_XXS": 4, "IQ4_XS": 2}   # positions per group (IQ4_XS: 2 nibbles)


def perm_pm(qtype, n_blocks):
    """Natural index of each position-major slot: x_pm = x[..., perm]; out_natural = out_pm[..., inv]."""
    w = GROUP_W[qtype]
    G = n_blocks * QK_K // w
    slots = np.arange(n_blocks * QK_K)
    if qtype == "IQ4_XS":   # weight d = sub*32 + pos*16 + j ; group g = sub*16 + j ; pos in {0 (low nibble), 1 (high)}
        pos, g = slots // G, slots % G
        return (32 * (g // 16) + 16 * pos + (g % 16)).astype(np.int32)
    pos, g = slots // G, slots % G                         # weight d = g*w + pos
    return (g * w + pos).astype(np.int32)


def _tree_pos(name, idx):
    """Codebook lookup returning one lane-dense int8 array per position: list of w arrays shaped like idx."""
    lut = _TREES.setdefault(name, _TreeLUT(name))
    wi = idx >> 5
    sh = (idx & 31).astype(jnp.uint32)
    bits = [((wi >> k) & 1).astype(bool) for k in range(int(np.log2(lut.n_words)))]
    code = jnp.zeros(idx.shape, jnp.uint32)
    for b in range(lut.nbits):
        nodes = [jnp.uint32(int(x)) for x in lut.words[b]]
        for bit in bits:
            nodes = [jnp.where(bit, nodes[2 * j + 1], nodes[2 * j]) for j in range(len(nodes) // 2)]
        code = code | (((nodes[0] >> sh) & 1) << b)
    mask = (1 << lut.lb) - 1
    outs = []
    for k in range(lut.w):
        lv = ((code >> (lut.lb * k)) & mask).astype(jnp.int32)
        v = jnp.full(lv.shape, int(lut.levels[0]), jnp.int8)
        for i in range(1, len(lut.levels)):
            v = jnp.where(lv == i, jnp.int8(int(lut.levels[i])), v)
        outs.append(v)
    return outs


def _rowflat(a):   # [..., nblk, m] -> [..., nblk*m]
    return a.reshape(a.shape[:-2] + (a.shape[-2] * a.shape[-1],))


def dequant_pm_iq2_s(blk):
    """blk uint8 [..., nblk, 82] -> float32 [..., 8 * nblk*32] position-major."""
    d = _f16(blk[..., 0:2])                                          # [..., nblk]
    qs = _rowflat(blk[..., 2:34].astype(jnp.int32))                  # [..., G]  G = nblk*32
    qh = blk[..., 66:74]
    hi = _rowflat(jnp.stack([(qh >> (2 * k)) & 3 for k in range(4)], -1).reshape(qh.shape[:-1] + (32,)).astype(jnp.int32))
    idx = qs | (hi << 8)                                             # [..., G]
    sc = _rowflat(_nibbles(blk[..., 74:82]).astype(jnp.float32))     # [..., nblk*16]  one per 2 groups
    scale = _rowflat(jnp.repeat(d, 16, axis=-1).reshape(d.shape + (16,))) * (0.5 + sc) * 0.25   # [..., nblk*16]
    scale = jnp.repeat(scale, 2, axis=-1)                            # [..., G]
    signs = _rowflat(blk[..., 34:66])                                # [..., G] one byte per group
    vals = _tree_pos("IQ2_S", idx)
    outs = [scale * v.astype(jnp.float32) * (1.0 - 2.0 * ((signs >> k) & 1).astype(jnp.float32)) for k, v in enumerate(vals)]
    return jnp.concatenate(outs, -1)


def dequant_pm_iq3_s(blk):
    """blk uint8 [..., nblk, 110] -> float32 [..., 4 * nblk*64] position-major."""
    d = _f16(blk[..., 0:2])
    qs = _rowflat(blk[..., 2:66].astype(jnp.int32))                  # [..., G]  G = nblk*64
    qh = _rowflat(_bits8(blk[..., 66:74]).reshape(blk.shape[:-1] + (64,)).astype(jnp.int32))
    idx = qs | (qh << 8)
    sc = _rowflat(_nibbles(blk[..., 106:110]).astype(jnp.float32))   # [..., nblk*8] one per 8 groups
    scale = _rowflat(jnp.repeat(d, 8, axis=-1).reshape(d.shape + (8,))) * (1.0 + 2.0 * sc)
    scale = jnp.repeat(scale, 8, axis=-1)                            # [..., G]
    sb = _rowflat(blk[..., 74:106])                                  # [..., nblk*32] sign bytes: byte j = groups 2j, 2j+1
    s_even = jnp.repeat(sb, 2, axis=-1)                              # [..., G] byte for each group
    shift = jnp.tile(jnp.array([0, 4], jnp.uint8), sb.shape[-1])     # group parity picks the nibble
    vals = _tree_pos("IQ3_S", idx)
    outs = [scale * v.astype(jnp.float32) * (1.0 - 2.0 * ((s_even >> (shift + k)) & 1).astype(jnp.float32))
            for k, v in enumerate(vals)]
    return jnp.concatenate(outs, -1)


def dequant_pm_iq4_xs(blk):
    """blk uint8 [..., nblk, 136] -> float32 [..., 2 * nblk*128] position-major (pos = nibble)."""
    d = _f16(blk[..., 0:2])
    sh = _u16(blk[..., 2:4])[..., 0]
    sl = _nibbles(blk[..., 4:8]).astype(jnp.int32)                   # [..., nblk, 8]
    shb = jnp.stack([(sh >> (2 * k)) & 3 for k in range(8)], -1).astype(jnp.int32)
    scales = ((sl | (shb << 4)) - 32).astype(jnp.float32) * d[..., None]          # [..., nblk, 8] per 32 weights
    scale = _rowflat(jnp.repeat(scales, 16, axis=-1))                # [..., nblk*128] per group (16 groups per sub-block)
    qs = _rowflat(blk[..., 8:136])                                   # [..., G] G = nblk*128
    lo = _kv16(qs & 0x0F)
    hi = _kv16(qs >> 4)
    return jnp.concatenate([scale * lo, scale * hi], -1)


def _kv16(nib):
    """16-entry IQ4_NL value table without a gather (XLA's generic gather ran at 0.11 G lookups/s = 74 ms per
    IQ4_XS layer): 4-bit nibble -> float32 via a 4-level select tree (15 selects)."""
    kv = [float(v) for v in KVALUES_IQ4NL]
    bits = [((nib >> k) & 1).astype(bool) for k in range(4)]
    nodes = [jnp.float32(v) for v in kv]
    for bit in bits:
        nodes = [jnp.where(bit, nodes[2 * j + 1], nodes[2 * j]) for j in range(len(nodes) // 2)]
    return nodes[0]


DEQUANT_PM = {"IQ2_S": dequant_pm_iq2_s, "IQ3_S": dequant_pm_iq3_s, "IQ4_XS": dequant_pm_iq4_xs}


def pm_permute(qtype, x):
    """x[..., n] natural order -> position-major order along the last axis, via reshapes/transposes only
    (XLA's generic gather costs ~2.5 ms per call on v5e, a transpose is free). Equals jnp.take(x, perm_pm(...), -1)."""
    n = x.shape[-1]
    lead = x.shape[:-1]
    if qtype == "IQ4_XS":                      # d = sub*32 + pos*16 + j  ->  [pos, sub, j]
        y = x.reshape(lead + (n // 32, 2, 16))
        return jnp.swapaxes(y, -3, -2).reshape(lead + (n,))
    w = GROUP_W[qtype]                          # d = g*w + pos  ->  [pos, g]
    y = x.reshape(lead + (n // w, w))
    return jnp.swapaxes(y, -2, -1).reshape(lead + (n,))
'''
FILES["glm53/model.py"] = r'''"""Glm5Next text model in pure JAX (functional, params = nested dict of arrays).

Reference: transformers `modeling_glm5_next.py` (v5.16). Every block is written so that
prefill (full sequence) and single-token decode share the same math; caches are explicit
pytrees threaded through the functions.

Parameter layout (all Linear weights stored as [in, out], i.e. y = x @ W):
  embed [V, D], norm [D], lm_head [D, V]
  layers[i]:
    ln1 [D], ln2 [D]
    hc_attn / hc_ffn: fn [hc*D, (2+hc)*hc], base [(2+hc)*hc], scale [3]
    attn (KDA):  q,k,v [D, H*hd]; conv_q, conv_k, conv_v [H*hd, K]; f_a [D, hd]; f_b [hd, H*hd];
                 dt_bias [H*hd]; A_log [H]; b [D, H]; g_a [D, hd]; g_b [hd, H*hd]; o_norm [hd]; o [H*hd, D]
    attn (MLA):  q_a [D, qr]; q_a_norm [qr]; q_b [qr, H*qk]; kv_a [D, kvr]; kv_a_norm [kvr];
                 kv_b [kvr, H*(qk+v)]; o [H*v, D]; indexer {...} (unused in v0)
    mlp (dense): gate [D, I], up [D, I], down [I, D]
    mlp (moe):   router_w [D, E], router_bias [E], gate_up [E, D, 2*mi], down [E, mi, D], shared {gate, up, down}
"""
from __future__ import annotations

import dataclasses
from typing import Any

import jax
import jax.numpy as jnp
from jax import lax

Params = dict[str, Any]


@dataclasses.dataclass(frozen=True)
class Cfg:
    hidden: int
    vocab: int
    n_layers: int
    layer_types: tuple[str, ...]      # "linear_attention" | "deepseek_sparse_attention"
    mlp_types: tuple[str, ...]        # "dense" | "sparse"
    eps: float
    # KDA
    kda_heads: int
    kda_hd: int
    conv_k: int
    gate_lb: float | None
    # MLA (NoPE)
    n_heads: int
    q_lora: int
    kv_lora: int
    qk_nope: int
    v_hd: int
    # indexer
    idx_heads: int
    idx_hd: int
    idx_topk: int
    idx_kpool: int
    # mHC
    hc: int
    hc_iters: int
    hc_eps: float
    # MLP / MoE
    inter: int
    moe_inter: int
    n_exp: int
    topk: int
    n_shared: int
    scaling: float
    norm_topk: bool
    swiglu_limit: float
    # compute dtype for matmuls (fp32 for CPU tests, bf16 on TPU)
    dtype: Any = jnp.float32
    # tensor-parallel reduction applied after every row-parallel matmul (None = single device)
    tp_reduce: Any = None
    tp_axis: Any = None               # shard_map axis name (collectives inside the model)
    seq_shard: int = 1                # MLA/indexer caches sharded over this many chips (interleaved by k-pool)
    q_block: int = 128                # queries per attention block (bounds the per-query temporaries)
    union_pools: int = 512            # per-block key-dedup budget (local pools); over budget -> per-query gathers; 0 = always per-query
    topk_local: int = 128             # first-stage local top-k per chip (exact: falls back to the full K when a chip's list was cut short)
    cache_cap: int = 0                # set inside mla_attention: global cache capacity
    cache_q8: bool = False            # MLA latent cache stored as int8 rows + a per-token f32 scale ("cs"): half the HBM

    def reduce(self, y):
        return y if self.tp_reduce is None else self.tp_reduce(y)

    @classmethod
    def from_hf(cls, c, dtype=jnp.float32) -> "Cfg":
        return cls(
            hidden=c.hidden_size, vocab=c.vocab_size, n_layers=c.num_hidden_layers,
            layer_types=tuple(c.layer_types), mlp_types=tuple(c.mlp_layer_types), eps=c.rms_norm_eps,
            kda_heads=c.linear_num_heads, kda_hd=c.linear_head_dim, conv_k=c.linear_conv_kernel_dim,
            gate_lb=c.linear_lower_bound,
            n_heads=c.num_attention_heads, q_lora=c.q_lora_rank, kv_lora=c.kv_lora_rank,
            qk_nope=c.qk_nope_head_dim, v_hd=c.v_head_dim,
            idx_heads=c.index_n_heads, idx_hd=c.index_head_dim, idx_topk=c.index_topk, idx_kpool=c.index_kpool,
            hc=c.hc_mult, hc_iters=c.hc_sinkhorn_iters, hc_eps=c.hc_eps,
            inter=c.intermediate_size, moe_inter=c.moe_intermediate_size, n_exp=c.n_routed_experts,
            topk=c.num_experts_per_tok, n_shared=c.n_shared_experts, scaling=c.routed_scaling_factor,
            norm_topk=c.norm_topk_prob, swiglu_limit=c.swiglu_limit, dtype=dtype,
        )


# ----------------------------------------------------------------------------- basics
def rmsnorm(x, w, eps):
    xf = x.astype(jnp.float32)
    xf = xf * lax.rsqrt(jnp.mean(xf * xf, -1, keepdims=True) + eps)
    return (w.astype(jnp.float32) * xf).astype(x.dtype)


def unweighted_rmsnorm_f32(x, eps):
    return x * lax.rsqrt(jnp.mean(x * x, -1, keepdims=True) + eps)


def l2norm(x, eps=1e-6):
    return x / jnp.sqrt(jnp.sum(x * x, -1, keepdims=True) + eps)


def silu(x):
    return x * jax.nn.sigmoid(x)


def swiglu_clamped(gate, up, limit):
    gate = jnp.minimum(gate, limit)
    up = jnp.clip(up, -limit, limit)
    return silu(gate) * up


def mlp_partial(p, x, cfg: Cfg):
    """Column/row-parallel MLP without the final TP reduction."""
    return swiglu_clamped(x @ p["gate"], x @ p["up"], cfg.swiglu_limit) @ p["down"]


def mlp(p, x, cfg: Cfg):
    return cfg.reduce(mlp_partial(p, x, cfg))


# ----------------------------------------------------------------------------- mHC
def hc_pre(p, streams, cfg: Cfg):
    """streams [B,T,hc,D] -> (post [B,T,hc], comb [B,T,hc,hc], collapsed [B,T,D])."""
    hc = cfg.hc
    B, T = streams.shape[:2]
    flat = unweighted_rmsnorm_f32(streams.reshape(B, T, -1).astype(jnp.float32), cfg.eps)
    mix = flat @ p["fn"].astype(jnp.float32)                       # [B,T,(2+hc)*hc]
    pre_w, post_w, comb_w = mix[..., :hc], mix[..., hc:2 * hc], mix[..., 2 * hc:]
    base, scale = p["base"].astype(jnp.float32), p["scale"].astype(jnp.float32)
    pre = jax.nn.sigmoid(pre_w * scale[0] + base[:hc]) + cfg.hc_eps
    post = 2.0 * jax.nn.sigmoid(post_w * scale[1] + base[hc:2 * hc])
    comb_logits = comb_w.reshape(B, T, hc, hc) * scale[2] + base[2 * hc:].reshape(hc, hc)
    comb = jax.nn.softmax(comb_logits, -1) + cfg.hc_eps
    # Sinkhorn on an hc x hc matrix. Row/column sums are written as explicit slice adds instead of reductions so the
    # whole (2*iters-1)-step chain stays one elementwise XLA fusion: with reductions every step is its own tiny kernel
    # (~40 launches per layer per token ≈ 10 ms/token on v5e at decode).
    rsum = lambda c: sum(c[..., :, j:j + 1] for j in range(hc))          # [B,T,hc,1]
    csum = lambda c: sum(c[..., j:j + 1, :] for j in range(hc))          # [B,T,1,hc]
    comb = comb / (csum(comb) + cfg.hc_eps)
    for _ in range(cfg.hc_iters - 1):
        comb = comb / (rsum(comb) + cfg.hc_eps)
        comb = comb / (csum(comb) + cfg.hc_eps)
    collapsed = jnp.einsum("bth,bthd->btd", pre, streams.astype(jnp.float32)).astype(streams.dtype)
    return post, comb, collapsed


def hc_post(post, comb, y, residual):
    """new_streams[h] = post[h] * y + sum_h' comb[h', h] * residual[h']  (matches torch code)."""
    dt = residual.dtype
    return post.astype(dt)[..., None] * y[..., None, :] + jnp.einsum("bthk,bthd->btkd", comb.astype(dt), residual)


# ----------------------------------------------------------------------------- KDA
def causal_conv1d(x, w, state=None, length=None):
    """Depthwise causal conv over time. x [B,T,C], w [C,K], state [B,K-1,C] or None (zeros).
    `length` (traced int, <= T): only the first `length` tokens are real; the new state is taken from them.
    Returns (y [B,T,C], new_state [B,K-1,C])."""
    B, T, C = x.shape
    K = w.shape[-1]
    if state is None:
        state = jnp.zeros((B, K - 1, C), x.dtype)
    xp = jnp.concatenate([state.astype(x.dtype), x], axis=1)        # [B, T+K-1, C]
    y = jnp.zeros((B, T, C), jnp.float32)
    for j in range(K):
        y = y + xp[:, j:j + T].astype(jnp.float32) * w[:, j].astype(jnp.float32)
    if length is None:
        new_state = xp[:, -(K - 1):]
    else:
        new_state = lax.dynamic_slice_in_dim(xp, length, K - 1, axis=1)   # inputs at t in [length-K+1, length-1]
    return y.astype(x.dtype), new_state


def kda_gates(p, x, cfg: Cfg):
    """Forget gate g [B,T,H,hd] (log-decay, <= 0) and beta [B,T,H] in fp32."""
    B, T, _ = x.shape
    f = (x @ p["f_a"]) @ p["f_b"]
    g = f.astype(jnp.float32) + p["dt_bias"].astype(jnp.float32)
    g = g.reshape(B, T, cfg.kda_heads, cfg.kda_hd)
    decay_rate = jnp.exp(p["A_log"].astype(jnp.float32))[None, None, :, None]
    if cfg.gate_lb is not None:
        g = cfg.gate_lb * jax.nn.sigmoid(decay_rate * g)
    else:
        g = -decay_rate * jax.nn.softplus(g)
    beta = jax.nn.sigmoid((x @ p["b"]).astype(jnp.float32))
    return g, beta


def kda_chunked(q, k, v, g, beta, state0=None, chunk=64):
    """Chunked KDA (HF `chunk_kimi_delta_attention`). q,k,v [B,T,H,d]; g [B,T,H,d]; beta [B,T,H].
    Returns out [B,T,H,d] (fp32) and final state [B,H,dk,dv] (fp32).
    Memory is O(one chunk) in the per-channel decay `exp(g_i - g_j)` [c,c,d]: it is computed per chunk inside a
    `lax.map` (intra-chunk term) and again inside the state scan, never for all chunks at once (that tensor was
    262 KB/token/chip and capped a single prefill call at ~2048 tokens)."""
    B, T, H, dk = k.shape
    dv = v.shape[-1]
    f32 = jnp.float32
    q, k, v, g = [jnp.swapaxes(t.astype(f32), 1, 2) for t in (q, k, v, g)]   # [B,H,T,*]
    beta = jnp.swapaxes(beta.astype(f32), 1, 2)                              # [B,H,T]
    q = l2norm(q) * (dk ** -0.5)
    k = l2norm(k)
    pad = (-T) % chunk
    Tp = T + pad
    if pad:
        q, k, v, g = [jnp.pad(t, ((0, 0), (0, 0), (0, pad), (0, 0))) for t in (q, k, v, g)]
        beta = jnp.pad(beta, ((0, 0), (0, 0), (0, pad)))
    n = Tp // chunk
    q, k, v, g = [t.reshape(B, H, n, chunk, -1) for t in (q, k, v, g)]
    beta = beta.reshape(B, H, n, chunk)
    k_beta = k * beta[..., None]
    v_beta = v * beta[..., None]
    g = jnp.cumsum(g, axis=-2)                                             # per-channel cumsum in chunk
    ii = jnp.arange(chunk)
    strict_lower = ii[:, None] > ii[None, :]
    lower_incl = ii[:, None] >= ii[None, :]
    eye = jnp.eye(chunk, dtype=f32)

    def chunk_decay(g_i):
        # decay[i,j,d] = exp(g_i[d] - g_j[d]) for i >= j (<= 1); the upper triangle is never used, clamp it so no
        # inf/NaN is ever produced (g spans down to -320 per chunk)
        return jnp.exp(jnp.minimum(g_i[..., :, None, :] - g_i[..., None, :, :], 0.0))   # [B,H,c,c,d]

    def intra(xs):
        k_i, g_i, kb_i = xs                                                  # [B,H,c,d]
        a = -jnp.einsum("bhid,bhjd,bhijd->bhij", kb_i, k_i, chunk_decay(g_i))
        return jnp.where(strict_lower, a, 0.0)

    per_chunk = lambda *ts: tuple(jnp.moveaxis(t, 2, 0) for t in ts)      # [n,B,H,c,*]
    attn = jnp.moveaxis(lax.map(intra, per_chunk(k, g, k_beta)), 0, 2)     # [B,H,n,c,c]
    # HF: forward substitution computing (I - attn)^{-1} with unit diagonal, i.e. inverse of (I + M)
    L = eye - attn                                                          # lower unit-triangular
    attn = jnp.vectorize(lambda m: jax.scipy.linalg.solve_triangular(m, eye, lower=True, unit_diagonal=True),
                         signature="(c,c)->(c,c)")(L)
    u = attn @ v_beta                                                       # [B,H,n,c,dv]
    w = attn @ (k_beta * jnp.exp(g))                                        # [B,H,n,c,dk]
    S0 = jnp.zeros((B, H, dk, dv), f32) if state0 is None else state0.astype(f32)

    def step(S, xs):
        q_i, k_i, g_i, u_i, w_i = xs                                          # leading dims [B,H,...]
        attn_inter = (q_i * jnp.exp(g_i)) @ S
        attn_intra = jnp.einsum("bhid,bhjd,bhijd->bhij", q_i, k_i, chunk_decay(g_i))
        attn_intra = jnp.where(lower_incl, attn_intra, 0.0)
        v_new = u_i - w_i @ S
        out = attn_inter + attn_intra @ v_new
        g_last = g_i[..., -1:, :]
        S = S * jnp.exp(g_last).swapaxes(-1, -2) + jnp.swapaxes(k_i * jnp.exp(g_last - g_i), -1, -2) @ v_new
        return S, out

    S, out = lax.scan(step, S0, per_chunk(q, k, g, u, w))
    out = jnp.moveaxis(out, 0, 2).reshape(B, H, Tp, dv)[:, :, :T]
    return jnp.swapaxes(out, 1, 2), S


def kda_recurrent(q, k, v, g, beta, state, hist=False):
    """One (or a few) tokens, recurrent form. Shapes as kda_chunked; state [B,H,dk,dv] fp32.
    hist=True also returns the state after every token ([T,B,H,dk,dv]; speculative-decoding rollback)."""
    f32 = jnp.float32
    B, T, H, dk = k.shape
    q = l2norm(q.astype(f32)) * (dk ** -0.5)
    k = l2norm(k.astype(f32))
    v, g, beta = v.astype(f32), g.astype(f32), beta.astype(f32)

    def step(S, xs):
        q_i, k_i, v_i, g_i, b_i = xs                                          # [B,H,d] / [B,H]
        S = S * jnp.exp(g_i)[..., None]
        kv_mem = jnp.einsum("bhkv,bhk->bhv", S, k_i)
        delta = (v_i - kv_mem) * b_i[..., None]
        S = S + k_i[..., None] * delta[..., None, :]
        o = jnp.einsum("bhkv,bhk->bhv", S, q_i)
        return S, ((o, S) if hist else o)

    xs = tuple(jnp.moveaxis(t, 1, 0) for t in (q, k, v, g, beta))
    S, out = lax.scan(step, state.astype(f32), xs)
    if hist:
        out, states = out
        return jnp.moveaxis(out, 0, 1), S, states
    return jnp.moveaxis(out, 0, 1), S


def kda_project(p, x, cfg: Cfg):
    """Position-free, batched part of a KDA layer: x [B,T,D] (input-normed) -> dict(qkv [B,T,3C] before the conv,
    conv_w [3C,K], g [B,T,H,hd], beta [B,T,H], gate [B,T,H,hd])."""
    B, T, _ = x.shape
    H, hd = cfg.kda_heads, cfg.kda_hd
    qkv = jnp.concatenate([x @ p["q"], x @ p["k"], x @ p["v"]], -1)            # [B,T,3C]
    conv_w = jnp.concatenate([p["conv_q"], p["conv_k"], p["conv_v"]], 0)        # [3C,K]
    g, beta = kda_gates(p, x, cfg)
    gate = ((x @ p["g_a"]) @ p["g_b"]).reshape(B, T, H, hd)
    return {"qkv": qkv, "conv_w": conv_w, "g": g, "beta": beta, "gate": gate}


def kda_core(proj, cfg: Cfg, cache=None, use_recurrent=False, length=None, hist=False):
    """Conv + KDA recurrence of `proj` (kda_project; any B in lockstep, ONE cache) -> (core [B,T,H,hd] f32,
    new_cache). cache: None or {"conv": [B,K-1,3C], "state": [B,H,dk,dv]}. `length`: number of real tokens (rest
    is padding: beta=0, g=0 so the state ignores them). hist=True (recurrent, T > 1: speculative verify) adds
    "state_hist" [T,B,H,dk,dv] and "conv_hist" [T,B,K-1,3C] = the cache after each token, for `rollback_caches`."""
    qkv_raw, conv_w, g, beta = proj["qkv"], proj["conv_w"], proj["g"], proj["beta"]
    B, T, _ = qkv_raw.shape
    H, hd = cfg.kda_heads, cfg.kda_hd
    conv_state = None if cache is None else cache["conv"]
    qkv, new_conv = causal_conv1d(qkv_raw, conv_w, conv_state, length)
    qkv = silu(qkv)
    C = H * hd
    q, k, v = qkv[..., :C], qkv[..., C:2 * C], qkv[..., 2 * C:]
    q, k, v = [t.reshape(B, T, H, hd) for t in (q, k, v)]
    if length is not None:
        valid = jnp.arange(T)[None, :] < length                                    # [1,T]
        beta = jnp.where(valid[..., None], beta, 0.0)
        g = jnp.where(valid[..., None, None], g, 0.0)
    state = None if cache is None else cache["state"]
    extra = {}
    if use_recurrent:
        assert state is not None
        if hist:
            core, new_state, states = kda_recurrent(q, k, v, g, beta, state, hist=True)
            K = conv_w.shape[-1]
            full = jnp.concatenate([conv_state.astype(qkv_raw.dtype), qkv_raw], axis=1)       # [B,K-1+T,3C]
            extra = {"state_hist": states, "conv_hist": jnp.stack([full[:, t + 1:t + K] for t in range(T)])}
        else:
            core, new_state = kda_recurrent(q, k, v, g, beta, state)
    else:
        core, new_state = kda_chunked(q, k, v, g, beta, state)
    return core, {"conv": new_conv, "state": new_state, **extra}


def kda_out(p, core, gate, cfg: Cfg, dtype):
    """Gated RMSNorm over hd (fp32, sigmoid gate) + output projection (TP-reduced): -> y [B,T,D]."""
    B, T, H, hd = gate.shape
    o = core.astype(jnp.float32)
    o = o * lax.rsqrt(jnp.mean(o * o, -1, keepdims=True) + cfg.eps) * p["o_norm"].astype(jnp.float32)
    o = o * jax.nn.sigmoid(gate.astype(jnp.float32))
    return cfg.reduce(o.astype(dtype).reshape(B, T, H * hd) @ p["o"])


def kda_attention(p, x, cfg: Cfg, cache=None, use_recurrent=False, length=None, hist=False):
    """x [B,T,D] (already input-normed). cache: None or {"conv": [B,K-1,3C], "state": [B,H,dk,dv]} (one cache, all
    rows in lockstep). Returns (y [B,T,D], new_cache); see kda_core for `length` / `hist`."""
    proj = kda_project(p, x, cfg)
    core, new_cache = kda_core(proj, cfg, cache, use_recurrent, length, hist)
    return kda_out(p, core, proj["gate"], cfg, x.dtype), new_cache


def kda_attention_rows(p, x, cfg: Cfg, caches):
    """Recurrent (decode) step of B INDEPENDENT streams: x [B,T,D], caches[b] = row b's own cache (batch dim 1).
    The projections and the output run batched, the conv/recurrence per row -> (y [B,T,D], [new cache per row])."""
    proj = kda_project(p, x, cfg)
    cores, new = [], []
    for b, c in enumerate(caches):
        pr = {k: (v if k == "conv_w" else v[b:b + 1]) for k, v in proj.items()}
        core, nc = kda_core(pr, cfg, c, True, None, False)
        cores.append(core); new.append(nc)
    return kda_out(p, jnp.concatenate(cores, 0), proj["gate"], cfg, x.dtype), new


def layernorm(x, w, b, eps):
    xf = x.astype(jnp.float32)
    mu = xf.mean(-1, keepdims=True)
    var = jnp.mean((xf - mu) ** 2, -1, keepdims=True)
    return ((xf - mu) * lax.rsqrt(var + eps) * w.astype(jnp.float32) + b.astype(jnp.float32)).astype(x.dtype)


# ----------------------------------------------------------------------------- DSA indexer (k-pool)
# Cache layouts. Replicated (cfg.seq_shard == 1): slot t of the MLA/indexer caches holds position t.
# Sequence-sharded over n = cfg.seq_shard chips, interleaved by k-pool: position t belongs to pool p = t // kp, lives
# on chip p % n at local slot (p // n) * kp + t % kp. Every chip therefore holds an equal share of complete pools, a
# prefill piece scatters T/n rows per chip, and the indexer scores its own pools with all heads.
def _pos_to_slot(t, kp, n):
    """Global position(s) -> (owner chip, local slot)."""
    pool = t // kp
    return pool % n, (pool // n) * kp + t % kp


def _slot_to_pos(s, kp, n, chip):
    return ((s // kp) * n + chip) * kp + s % kp


def cache_write(cache, new, pos0, cfg: Cfg, chip=None, valid=None):
    """Write new [B,T,R] at global positions pos0.. into cache [B,S_local,R] (see layouts above). `valid` [T] bool
    (optional) keeps the cache untouched for rows that are False."""
    new = new.astype(cache.dtype)
    n, kp = cfg.seq_shard, cfg.idx_kpool
    if n == 1 and valid is None:
        return lax.dynamic_update_slice_in_dim(cache, new, pos0, axis=1)
    B, T = new.shape[:2]
    S_local = cache.shape[1]
    if n == 1:
        mine, slot = jnp.ones((T,), bool), pos0 + jnp.arange(T)
    else:
        owner, slot = _pos_to_slot(pos0 + jnp.arange(T), kp, n)
        mine = owner == chip
    if valid is not None:
        mine = mine & valid
    if T <= 2:                                     # decode: read-select-write the row(s), no scatter
        for t in range(T):
            cur = lax.dynamic_slice_in_dim(cache, slot[t], 1, axis=1)
            cache = lax.dynamic_update_slice_in_dim(cache, jnp.where(mine[t], new[:, t:t + 1], cur), slot[t], axis=1)
        return cache
    # one scatter: rows of other chips (or invalid rows) go to distinct out-of-bounds slots, which XLA drops
    slot = jnp.where(mine, slot, S_local + jnp.arange(T))
    return cache.at[:, slot].set(new, mode="drop", unique_indices=True)


def mla_cache_shapes(cfg: Cfg, B, S, has_indexer, n=1):
    """Global cache shapes of one MLA layer for capacity S (per chip: the sequence/pool axis divided by n):
    "c" latent [B,S,kvr] (+ "cs" [B,S] per-token scales when cfg.cache_q8: "c" is then int8); with an indexer "pk"
    pooled indexer keys [B,S/kp,ihd] (one row per complete k-pool) and the raw keys/gates of the incomplete tail
    pool "tk"/"tg" [B,kp,ihd] (replicated, tiny). Dtypes: `cache_dtype`."""
    kp = cfg.idx_kpool
    out = {"c": (B, S // n, cfg.kv_lora)}
    if cfg.cache_q8:
        out["cs"] = (B, S // n)
    if has_indexer:
        out.update(pk=(B, S // kp // n, cfg.idx_hd), tk=(B, kp, cfg.idx_hd), tg=(B, kp, cfg.idx_hd))
    return out


ROW_KEYS = ("c", "cs", "pk")         # positional cache entries (rows per token / per pool); the rest are tail buffers


def cache_dtype(cfg: Cfg, key, dtype):
    """dtype of one MLA cache entry: int8 latents + f32 scales under cache_q8, else the model dtype."""
    if cfg.cache_q8 and key == "c":
        return jnp.int8
    if key == "cs":
        return jnp.float32
    return dtype


def latent_dequant(q, s, dtype):
    """int8 latent rows [..., kvr] x per-row scales [...] -> dtype."""
    return (q.astype(jnp.float32) * s.astype(jnp.float32)[..., None]).astype(dtype)


def write_latent(cache, new_c, pos0, cfg: Cfg, chip=None):
    """New latents [B,T,kvr] into the cache at pos0 -> the updated {"c"} (or {"c", "cs"}: per-token absmax int8)."""
    if not cfg.cache_q8:
        return {"c": cache_write(cache["c"], new_c, pos0, cfg, chip)}
    x = new_c.astype(jnp.float32)
    s = jnp.maximum(jnp.max(jnp.abs(x), axis=-1), 1e-12) / 127.0                  # [B,T]
    q = jnp.clip(jnp.round(x / s[..., None]), -127, 127).astype(jnp.int8)
    return {"c": cache_write(cache["c"], q, pos0, cfg, chip), "cs": cache_write(cache["cs"], s, pos0, cfg, chip)}


def pool_new_keys(ix, tail_k, tail_g, new_k, new_g, pos0, cfg: Cfg, length, hist=False):
    """k-pool compression of the NEW tokens (the first `length` of new_k/new_g, at positions pos0..) together with
    the raw keys of the incomplete pool they may extend (tail_k/tail_g [B,kp,ihd]: row i = position (pos0//kp)*kp + i
    for i < pos0 % kp). -> (pooled [B,NP,ihd] f32, complete [NP] bool, pool0 = global id of pooled[0], new tails)."""
    B, T, ihd = new_k.shape
    kp = cfg.idx_kpool
    f32 = jnp.float32
    a = pos0 % kp                                                             # tokens of the tail pool already seen
    NP = (T + 2 * kp - 2) // kp + 1                                           # pools covering a + T rows (+1 spare)
    L = NP * kp

    def fill(tail, new):
        buf = jnp.zeros((B, L, ihd), f32).at[:, :kp].set(tail.astype(f32))
        return lax.dynamic_update_slice_in_dim(buf, new.astype(f32), a, axis=1)

    kb, gb = fill(tail_k, new_k), fill(tail_g, new_g)
    present = jnp.arange(L) < a + length                                      # [L] rows that hold real tokens
    logits = jnp.where(present[None, :, None], gb + jnp.tile(ix["ape"].astype(f32), (L // kp, 1))[None], -jnp.inf)
    probs = jnp.nan_to_num(jax.nn.softmax(logits.reshape(B, NP, kp, ihd), axis=2))
    pooled = (probs * kb.reshape(B, NP, kp, ihd)).sum(2)                      # [B,NP,ihd]
    complete = present.reshape(NP, kp).all(-1)
    start = ((a + length) // kp) * kp                                         # first row of the (incomplete) last pool
    tail_k2 = lax.dynamic_slice_in_dim(kb, start, kp, axis=1)
    tail_g2 = lax.dynamic_slice_in_dim(gb, start, kp, axis=1)
    if hist:                                                                  # tails after each of the T tokens
        st = [((a + t + 1) // kp) * kp for t in range(T)]
        th_k = jnp.stack([lax.dynamic_slice_in_dim(kb, s_, kp, axis=1) for s_ in st])
        th_g = jnp.stack([lax.dynamic_slice_in_dim(gb, s_, kp, axis=1) for s_ in st])
        return pooled, complete, pos0 // kp, tail_k2, tail_g2, (th_k, th_g)
    return pooled, complete, pos0 // kp, tail_k2, tail_g2


def indexer_pool(ix, keys, gates, cfg: Cfg):
    """k-pool compression of ALL cached keys (query independent). keys/gates [B,S_local,ihd] ->
    (pool_keys [B,P_local,ihd] f32, pool_valid [P_local] bool). Pools straddling the end of a replicated cache
    whose size is not a multiple of kp are invalid (model-level calls with cap=None)."""
    B, S, ihd = keys.shape
    kp = cfg.idx_kpool
    P = -(-S // kp)
    pad = P * kp - S
    kpad = jnp.pad(keys, ((0, 0), (0, pad), (0, 0))).reshape(B, P, kp, ihd).astype(jnp.float32)
    gpad = jnp.pad(gates, ((0, 0), (0, pad), (0, 0))).reshape(B, P, kp, ihd).astype(jnp.float32)
    tok_valid = jnp.arange(P * kp).reshape(P, kp) < S                             # [P,kp]
    logits = jnp.where(tok_valid[None, :, :, None], gpad + ix["ape"].astype(jnp.float32)[None, None], -jnp.inf)
    probs = jnp.nan_to_num(jax.nn.softmax(logits, axis=2))
    return (probs * kpad).sum(2), tok_valid.all(-1)


def indexer_select(scores, pool_valid, qpos, cfg: Cfg, chip=None):
    """scores [B,Tq,P_local] f32 (this chip's pools); qpos [Tq] global query positions -> (pools [B,Tq,K] global pool
    ids, valid [B,Tq,K]): the top-K pools per query (two-stage top-k across chips when the cache is sharded: local
    top-K, all-gather the candidates, global top-K)."""
    B, Tq, P_local = scores.shape
    n, kp = cfg.seq_shard, cfg.idx_kpool
    lo = jnp.finfo(jnp.float32).min
    j = jnp.arange(P_local)
    gpool = j * n + (chip if n > 1 else 0)                                            # global pool ids [P_local]
    pool_end = gpool * kp + kp - 1
    visible = (pool_end[None, :] <= qpos[:, None]) & pool_valid[None, :]              # [Tq,P_local]
    scores = jnp.where(visible[None], scores, lo)
    K = min(cfg.idx_topk // kp, P_local)
    if n == 1:
        vals, sel = lax.top_k(scores, K)
        return sel, vals > lo
    off = chip

    def two_stage(k_local):
        """local top-k_local -> all-gather -> global top-K; also returns whether any chip's list was cut short of
        the global threshold (then the result may be inexact)."""
        v, j = lax.top_k(scores, k_local)                                             # [B,Tq,k_local]
        j = j * n + off
        va = lax.all_gather(v, cfg.tp_axis, axis=-1, tiled=True)                      # [B,Tq,n*k_local]
        ca = lax.all_gather(j, cfg.tp_axis, axis=-1, tiled=True)
        kk = min(cfg.idx_topk // kp, n * k_local)
        neg, ca = lax.sort((-va, ca), dimension=-1, num_keys=1)                       # descending by score, ids carried
        vs, ss = -neg[..., :kk], ca[..., :kk]
        thr = vs[..., -1:]                                                            # weakest globally selected score
        cut = (k_local < K) & jnp.any(v[..., -1:] > thr)                              # a chip had more above the bar
        cut = lax.pmax(cut.astype(jnp.int32), cfg.tp_axis) > 0
        return vs, ss, cut

    k_fast = min(cfg.topk_local, K) if Tq > 1 else K       # a single query (decode) gains nothing from the fallback dance
    vs, ss, cut = two_stage(k_fast)
    if k_fast < K:
        vs, ss = lax.cond(cut, lambda _: two_stage(K)[:2], lambda _: (vs, ss), None)
    return ss, vs > lo


def select_tokens(pools, valid, qpos, cfg: Cfg):
    """Selected pools -> global token indices [B,Tq,W] int32 (-1 = invalid), W = idx_topk + kp - 1 padded to a
    multiple of 16 (a tile-aligned slot count keeps the gathered keys free of relayout copies): the pools' tokens plus
    the incomplete tail pool of the visible prefix (positions 0..qpos)."""
    B, Tq, K = pools.shape
    kp, S = cfg.idx_kpool, cfg.cache_cap
    tok = (pools[..., None] * kp + jnp.arange(kp)).reshape(B, Tq, K * kp)
    tok = jnp.where(jnp.repeat(valid, kp, axis=-1), tok, -1)
    tok = jnp.pad(tok, ((0, 0), (0, 0), (0, cfg.idx_topk - K * kp)), constant_values=-1)
    vis_count = qpos + 1
    tail_count = vis_count % kp
    tail_start = vis_count - tail_count
    off = jnp.arange(kp - 1)
    tail = tail_start[:, None] + off[None, :]                                         # [Tq,kp-1]
    tail_ok = (off[None, :] < tail_count[:, None]) & (tail < S)
    tail = jnp.where(tail_ok, tail, -1)
    tok = jnp.concatenate([tok, jnp.broadcast_to(tail[None], (B, Tq, kp - 1))], -1)
    W = tok.shape[-1]
    align = 32 if cfg.cache_q8 else 16                                              # int8 rows tile by 32 sublanes
    return jnp.pad(tok, ((0, 0), (0, 0), (0, (-W) % align)), constant_values=-1).astype(jnp.int32)


def union_pools(pools, valid, qpos, cfg: Cfg, chip=None):
    """Per-block key deduplication. pools/valid [B,Tq,K] global pool ids -> (member [B,Tq,U] bool, upool [U] local
    pool ids of the block's union (sentinel P_local past the end), count): the union over the block's queries of the
    selected pools that live on this chip (plus each query's incomplete tail pool), compacted into U slots."""
    B, Tq, K = pools.shape
    n, kp = cfg.seq_shard, cfg.idx_kpool
    P_local = -(-(cfg.cache_cap // n) // kp)                                           # pools per chip (last may be partial)
    U = min(cfg.union_pools, P_local)
    tail_pool = jnp.broadcast_to((qpos // kp)[None, :, None], (B, Tq, 1))
    tail_on = jnp.broadcast_to(((qpos + 1) % kp != 0)[None, :, None], (B, Tq, 1))
    cand = jnp.concatenate([pools, tail_pool], -1)                                    # [B,Tq,K+1] global pools
    ok = jnp.concatenate([valid, tail_on], -1)
    if n > 1:
        ok = ok & (cand % n == chip)
        cand = cand // n
    local = jnp.where(ok, cand, P_local)                                              # sentinel = P_local
    srt = jnp.sort(local.reshape(-1))
    first = jnp.concatenate([jnp.ones((1,), jnp.bool_), srt[1:] != srt[:-1]]) & (srt < P_local)
    count = first.sum()
    # the U smallest unique pool ids (top-k of the negated keys; a scatter compaction costs more on TPU)
    keys = jnp.where(first, -srt, -P_local)
    if keys.shape[0] < U:
        keys = jnp.pad(keys, (0, U - keys.shape[0]), constant_values=-P_local)
    upool = -lax.top_k(keys, U)[0]                                                    # ascending union, sentinel P_local
    member = (local[..., :, None] == upool[None, None, None, :]).any(-2) & (upool < P_local)[None, None]   # [B,Tq,U]
    return member, upool, count, U


def _attend(q, keys, mask, cfg: Cfg, per_query_keys):
    """Softmax attention of q [B,Tq,Hq,r] over keys ([B,Tq,W,r] if per_query_keys else [B,S,r]) restricted to
    `mask` ([B,Tq,W] or [Tq,S]). With a sharded cache Hq = all heads and every chip holds a slice of the keys:
    partial softmax with the global max, then reduce-scatter over heads -> [B,Tq,H_local,r]."""
    lo = jnp.finfo(jnp.float32).min
    scale = cfg.qk_nope ** -0.5
    if per_query_keys:
        s = jnp.einsum("bthr,btwr->bhtw", q, keys).astype(jnp.float32) * scale
        s = jnp.where(mask[:, None], s, lo)
    else:
        s = jnp.einsum("bthr,bsr->bhts", q, keys).astype(jnp.float32) * scale
        mask = mask if mask.ndim == 3 else mask[None]                                 # [B or 1, Tq, S]
        s = jnp.where(mask[:, None], s, lo)
    pv = "bhtw,btwr->bthr" if per_query_keys else "bhts,bsr->bthr"
    if cfg.seq_shard == 1:
        probs = jax.nn.softmax(s, -1).astype(keys.dtype)
        return jnp.einsum(pv, probs, keys)
    m = lax.pmax(s.max(-1), cfg.tp_axis)                                              # [B,Hq,Tq] global max
    e = jnp.exp(s - m[..., None])                                                     # 0 where masked / no local keys
    l = jnp.sum(e, -1)                                                                # [B,Hq,Tq]
    o = jnp.einsum(pv, e.astype(keys.dtype), keys, preferred_element_type=jnp.float32)   # [B,Tq,Hq,r]
    o = lax.psum_scatter(o, cfg.tp_axis, scatter_dimension=2, tiled=True)             # [B,Tq,H_local,r]
    l = lax.psum_scatter(l, cfg.tp_axis, scatter_dimension=1, tiled=True)             # [B,H_local,Tq]
    return (o / jnp.swapaxes(l, 1, 2)[..., None]).astype(keys.dtype)


# ----------------------------------------------------------------------------- MLA (NoPE)
def mla_project(p, x, cfg: Cfg):
    """Position-free, batched part of an MLA layer: x [B,T,D] -> dict(q_lat [B,T,H,kvr] absorbed queries, new_c
    [B,T,kvr] new latents; with an indexer also nk / ng [B,T,ihd] (its keys / gates), qi [B,T,IH,ihd] f32 and
    wts [B,T,IH] f32 (its queries / head weights))."""
    B, T, _ = x.shape
    H, qk, dv, kvr = cfg.n_heads, cfg.qk_nope, cfg.v_hd, cfg.kv_lora
    q_resid = rmsnorm(x @ p["q_a"], p["q_a_norm"], cfg.eps)
    q = (q_resid @ p["q_b"]).reshape(B, T, H, qk)
    new_c = rmsnorm(x @ p["kv_a"], p["kv_a_norm"], cfg.eps)                         # [B,T,kvr]
    kv_b = p["kv_b"].reshape(kvr, H, qk + dv)
    Wk = kv_b[:, :, :qk]
    out = {"q_lat": jnp.einsum("bthq,rhq->bthr", q, Wk), "new_c": new_c}            # absorbed query [B,T,H,kvr]
    if "indexer" in p:
        ix = p["indexer"]
        IH, ihd = cfg.idx_heads, cfg.idx_hd
        out["nk"] = layernorm(x @ ix["wk"], ix["k_norm_w"], ix["k_norm_b"], 1e-6)
        out["ng"] = x @ ix["gate"]
        out["qi"] = (q_resid @ ix["wq_b"]).reshape(B, T, IH, ihd).astype(jnp.float32)
        out["wts"] = (x @ ix["weights_proj"]).astype(jnp.float32) * (IH ** -0.5)     # [B,T,IH]
    return out


def mla_core(p, proj, cfg: Cfg, cache=None, pos0=0, sparse=None, cap=None, length=None, hist=False):
    """Position-dependent part of an MLA layer for ONE cache (all rows in lockstep at pos0): cache update, DSA
    selection and attention -> (o_lat [B,T,H_local,kvr], new_cache). Arguments as in `mla_attention`."""
    q_lat, new_c = proj["q_lat"], proj["new_c"]
    B, T, H, kvr = q_lat.shape
    n, kp = cfg.seq_shard, cfg.idx_kpool
    chip = lax.axis_index(cfg.tp_axis) if n > 1 else None
    has_ix = "indexer" in p
    q8 = cfg.cache_q8
    if cache is None:
        S = T if cap is None else cap
        assert S % (kp * n) == 0 or n == 1, (S, kp, n)
        S_alloc = -(-S // kp) * kp if n == 1 else S                                # a whole number of pools
        cache = {k: jnp.zeros(sh, cache_dtype(cfg, k, new_c.dtype)) for k, sh in mla_cache_shapes(cfg, B, S_alloc, has_ix, n).items()}
        if n == 1 and S_alloc != S:
            cache["c"] = cache["c"][:, :S]
            if q8:
                cache["cs"] = cache["cs"][:, :S]
    S = cache["c"].shape[1] * n                                                    # global capacity
    cfg = dataclasses.replace(cfg, cache_cap=S)
    new_cache = write_latent(cache, new_c, pos0, cfg, chip)
    if has_ix:
        ix = p["indexer"]
        res = pool_new_keys(ix, cache["tk"], cache["tg"], proj["nk"], proj["ng"], pos0, cfg,
                            T if length is None else length, hist)
        pooled, complete, pool0, tk2, tg2 = res[:5]
        pcfg = dataclasses.replace(cfg, idx_kpool=1)                             # pool-level slots: pool % n, pool // n
        new_cache["pk"] = cache_write(cache["pk"], pooled, pool0, pcfg, chip, valid=complete)
        new_cache["tk"], new_cache["tg"] = tk2.astype(cache["tk"].dtype), tg2.astype(cache["tg"].dtype)
        if hist:
            new_cache["tk_hist"] = res[5][0].astype(cache["tk"].dtype)
            new_cache["tg_hist"] = res[5][1].astype(cache["tg"].dtype)
    c_all = new_cache["c"]                                                         # [B,S_local,kvr] (int8 under q8)
    cs_all = new_cache.get("cs")                                                   # [B,S_local] scales under q8
    dtype = new_c.dtype
    qpos = pos0 + jnp.arange(T)
    use_sparse = sparse == "always" or (sparse == "auto" and S > cfg.idx_topk)
    if use_sparse:
        pool_keys = new_cache["pk"].astype(jnp.float32)                            # [B,P_local,ihd] cached pooled keys
        pool_valid = jnp.ones((pool_keys.shape[1],), bool)                         # only complete pools are written;
        ihd = cfg.idx_hd                                                           # later ones are masked by position
        qi, wts = proj["qi"], proj["wts"]
    if n > 1 and not use_sparse:
        kpos = _slot_to_pos(jnp.arange(c_all.shape[1]), kp, n, chip)               # global position per local slot
    elif not use_sparse:
        kpos = jnp.arange(S)
    if use_sparse:                                                                 # pool-major view for the key-dedup path (one relayout per call)
        S_local = c_all.shape[1]
        P_local = -(-S_local // kp)
        c_pools = jnp.pad(c_all, ((0, 0), (0, P_local * kp - S_local), (0, 0))) if S_local % kp else c_all
        c_view = c_pools.reshape(B, P_local, kp * kvr)
        if q8:
            cs_pools = jnp.pad(cs_all, ((0, 0), (0, P_local * kp - S_local))) if S_local % kp else cs_all
            cs_view = cs_pools.reshape(B, P_local, kp)

    def per_query_attend(q_b, pools, pvalid, qpos_b):
        """Per-query gather of the selected keys that live on this chip (the pre-dedup path; decode + fallback)."""
        tok = select_tokens(pools, pvalid, qpos_b, cfg)                            # [B,Tq,W] global positions
        valid = (tok >= 0) & (tok <= qpos_b[None, :, None])
        if n > 1:
            owner, slot = _pos_to_slot(jnp.maximum(tok, 0), kp, n)
            valid = valid & (owner == chip)
        else:
            slot = tok
        slot = jnp.where(valid, slot, 0)
        c_sel = jnp.take_along_axis(c_all[:, None], slot[..., None], axis=2, mode="promise_in_bounds")
        if q8:
            s_sel = jnp.take_along_axis(cs_all[:, None], slot, axis=2, mode="promise_in_bounds")   # [B,Tq,W]
            c_sel = latent_dequant(c_sel, s_sel, dtype)
        return _attend(q_b, c_sel, valid, cfg, True)

    def block(q_lat_b, qi_b, wts_b, qpos_b):
        """One block of queries -> o_lat [B,Tq,H_local,kvr]."""
        if n > 1:
            q_lat_b = lax.all_gather(q_lat_b, cfg.tp_axis, axis=2, tiled=True)      # all heads [B,Tq,H*n,kvr]
        if use_sparse:
            sc = jax.nn.relu(jnp.einsum("bthd,bpd->bthp", qi_b, pool_keys) * (ihd ** -0.5))
            sc = jnp.einsum("bth,bthp->btp", wts_b, sc)                             # [B,Tq,P_local]
            pools, pvalid = indexer_select(sc, pool_valid, qpos_b, cfg, chip)      # [B,Tq,K] global pools
            if qpos_b.shape[0] <= 4 or cfg.union_pools == 0:                       # decode / spec verify / dedup off
                return per_query_attend(q_lat_b, pools, pvalid, qpos_b)             # (no union bookkeeping, no conds)
            member, upool, count, U = union_pools(pools, pvalid, qpos_b, cfg, chip)
            over = count > U
            if n > 1:
                over = lax.pmax(over.astype(jnp.int32), cfg.tp_axis) > 0          # every chip takes the same branch
            def dedup(args):
                q_b, member, upool = args
                idx = jnp.minimum(upool, P_local - 1)
                keys = c_view.at[:, idx].get(mode="promise_in_bounds")                # [B,U,kp*kvr]
                keys = keys.reshape(B, U * kp, kvr)                                 # the union's tokens, pool-major
                if q8:
                    keys = latent_dequant(keys, cs_view.at[:, idx].get(mode="promise_in_bounds").reshape(B, U * kp), dtype)
                slots = (upool[:, None] * kp + jnp.arange(kp)[None, :]).reshape(-1)
                kpos_u = _slot_to_pos(slots, kp, n, chip) if n > 1 else slots
                mask = jnp.repeat(member, kp, axis=-1) & (kpos_u[None, None, :] <= qpos_b[None, :, None])
                return _attend(q_b, keys, mask, cfg, False)

            per_query = lambda args: per_query_attend(args[0], pools, pvalid, qpos_b)
            return lax.cond(over, per_query, dedup, (q_lat_b, member, upool))
        c_dense = latent_dequant(c_all, cs_all, dtype) if q8 else c_all
        return _attend(q_lat_b, c_dense, kpos[None, :] <= qpos_b[:, None], cfg, False)

    Tq = min(cfg.q_block, T)
    nb = -(-T // Tq)
    if nb == 1:
        o_lat = block(q_lat, qi if use_sparse else None, wts if use_sparse else None, qpos)
    else:
        Tpad = nb * Tq - T
        padq = lambda a: jnp.pad(a, ((0, 0), (0, Tpad)) + ((0, 0),) * (a.ndim - 2)) if Tpad else a
        blk = lambda a: jnp.moveaxis(padq(a).reshape((B, nb, Tq) + a.shape[2:]), 1, 0)   # [nb,B,Tq,...]
        qpos_blk = jnp.pad(qpos, (0, Tpad)).reshape(nb, Tq)
        xs = (blk(q_lat), blk(qi) if use_sparse else None, blk(wts) if use_sparse else None, qpos_blk)
        o_lat = lax.map(lambda t: block(*t), xs)                                    # [nb,B,Tq,H,kvr]
        o_lat = jnp.moveaxis(o_lat, 0, 1).reshape(B, nb * Tq, H, kvr)[:, :T]
    return o_lat, new_cache


def mla_out(p, o_lat, cfg: Cfg):
    """o_lat [B,T,H_local,kvr] -> value up-projection + output projection (TP-reduced) [B,T,D]."""
    B, T, H, kvr = o_lat.shape
    qk, dv = cfg.qk_nope, cfg.v_hd
    Wv = p["kv_b"].reshape(kvr, cfg.n_heads, qk + dv)[:, :, qk:]
    o = jnp.einsum("bthr,rhv->bthv", o_lat, Wv).reshape(B, T, H * dv)
    return cfg.reduce(o @ p["o"])


def mla_attention(p, x, cfg: Cfg, cache=None, pos0=0, sparse=None, cap=None, length=None, hist=False):
    """Causal NoPE MLA with absorbed kv_b and a FIXED-CAPACITY latent cache (no recompiles as the sequence grows).
    x [B,T,D]. cache: None (prefill; allocates `cap` slots, or exactly T when cap is None) or
    {"c": [B,S_local,kvr], "pk": [B,P_local,ihd] pooled indexer keys, "tk"/"tg": [B,kp,ihd] tail}; new tokens are written at pos0 (may be
    traced). Slots beyond the current query position hold stale/zero data and are always masked by causality.
    sparse: None -> dense causal; "auto" -> DSA indexer when the (global) capacity > idx_topk; "always".
    Queries are processed in blocks of cfg.q_block (lax.map) so the per-query temporaries (indexer scores over all
    pools, the gathered top-k keys) never scale with the prompt length. cfg.seq_shard > 1: cache sharded over the
    chips (see the layout notes above), queries all-gathered over heads, partial softmax combined across chips.
    = mla_project (batched, position-free) -> mla_core (one cache, all rows at pos0) -> mla_out."""
    proj = mla_project(p, x, cfg)
    o_lat, new_cache = mla_core(p, proj, cfg, cache, pos0, sparse, cap, length, hist)
    return mla_out(p, o_lat, cfg), new_cache


def mla_attention_rows(p, x, cfg: Cfg, caches, pos, sparse="auto", cap=None):
    """Decode step of B INDEPENDENT streams: x [B,T,D]; caches[b] = row b's own cache (batch dim 1) at position
    pos[b] (traced scalars). Projections and the output run batched, the cache update / DSA selection / attention
    per row -> (y [B,T,D], [new cache per row])."""
    proj = mla_project(p, x, cfg)
    outs, new = [], []
    for b, (c, pb) in enumerate(zip(caches, pos)):
        pr = {k: v[b:b + 1] for k, v in proj.items()}
        o_b, nc = mla_core(p, pr, cfg, c, pb, sparse, cap, None, False)
        outs.append(o_b); new.append(nc)
    return mla_out(p, jnp.concatenate(outs, 0), cfg), new


# ----------------------------------------------------------------------------- MoE
def moe_route(p, x, cfg: Cfg):
    """x [N,D] -> (topk_idx [N,k] int32, topk_w [N,k] fp32). n_group == 1 (no group masking)."""
    logits = x.astype(jnp.float32) @ p["router_w"].astype(jnp.float32)
    scores = jax.nn.sigmoid(logits)
    choice = scores + p["router_bias"].astype(jnp.float32)
    _, idx = lax.top_k(choice, cfg.topk)
    # one-hot pick instead of take_along_axis: XLA's generic gather costs ~2.5 ms per call on TPU v5e
    w = jnp.sum(jax.nn.one_hot(idx, scores.shape[-1], dtype=scores.dtype) * scores[:, None, :], axis=-1)
    if cfg.norm_topk:
        w = w / (w.sum(-1, keepdims=True) + 1e-20)
    return idx.astype(jnp.int32), w * cfg.scaling


def moe_apply(x, gate_up_w, down_w, w, limit):
    """x [N,D]; gate_up_w [N,k,D,2mi]; down_w [N,k,mi,D]; w [N,k] -> [N,D]."""
    gu = jnp.einsum("nd,nkdm->nkm", x, gate_up_w)
    mi = gu.shape[-1] // 2
    h = swiglu_clamped(gu[..., :mi], gu[..., mi:], limit)
    y = jnp.einsum("nkm,nkmd->nkd", h, down_w)
    return jnp.einsum("nkd,nk->nd", y.astype(jnp.float32), w).astype(x.dtype)


def moe_apply_grouped(x, gu_w, dn_w, idx_local, w, limit, chunk=256):
    """Grouped experts: x [N,D]; gu_w [Eh,D,2mi]; dn_w [Eh,mi,D]; idx_local [N,k] (ids into Eh, -1 = none);
    w [N,k]. Every token is run through every gathered expert (dense, masked) — Eh/k× extra FLOPs but no per-token
    weight copies, so prefill fits HBM. Token chunks of `chunk` bound the [N,Eh,mi] intermediate."""
    N, D = x.shape
    Eh = gu_w.shape[0]
    mi = dn_w.shape[1]
    # routing weights as a dense [N, Eh] matrix
    onehot = jax.nn.one_hot(jnp.maximum(idx_local, 0), Eh, dtype=jnp.float32) * (idx_local >= 0)[..., None]
    rw = jnp.einsum("nk,nke->ne", w.astype(jnp.float32), onehot)            # [N,Eh]
    outs = []
    for s0 in range(0, N, chunk):
        xs, rws = x[s0:s0 + chunk], rw[s0:s0 + chunk]
        gu = jnp.einsum("nd,edm->nem", xs, gu_w)                              # [n,Eh,2mi]
        h = swiglu_clamped(gu[..., :mi], gu[..., mi:], limit)
        h = jnp.where(rws[..., None] > 0, h.astype(jnp.float32) * rws[..., None], 0.0).astype(x.dtype)   # fold routing weights in
        outs.append(jnp.einsum("nem,emd->nd", h, dn_w))
    return jnp.concatenate(outs, 0) if len(outs) > 1 else outs[0]


def moe_dense(p, x, cfg: Cfg, fetch=None, layer=0):
    """MoE block for x [B,T,D]. `fetch(layer, idx)` returns (gate_up_w [N,k,D,2mi], down_w [N,k,mi,D]) for
    idx [N,k]; default = gather from the on-device tables p["gate_up"] [E,D,2mi], p["down"] [E,mi,D].
    Under TP the expert/shared outputs are partial sums; one reduction covers both."""
    B, T, D = x.shape
    xf = x.reshape(-1, D)
    idx, w = moe_route(p, xf, cfg)
    if fetch is not None and hasattr(fetch, "apply"):      # resident quantized experts (glm53.resident)
        y = fetch.apply(p, xf, idx, w, layer, cfg.swiglu_limit).astype(xf.dtype)
    else:
        if fetch is None:
            gu_w, dn_w = p["gate_up"][idx], p["down"][idx]
        else:
            gu_w, dn_w = fetch(layer, idx)
        y = moe_apply(xf, gu_w, dn_w, w, cfg.swiglu_limit)
    y = cfg.reduce(y + mlp_partial(p["shared"], xf, cfg))
    return y.reshape(B, T, D)


# ----------------------------------------------------------------------------- layer / model
def decoder_layer(p, streams, cfg: Cfg, ltype, mtype, cache=None, pos0=0, use_recurrent=False, fetch=None,
                  sparse="auto", layer=0, cap=None, length=None, hist=False):
    post, comb, h = hc_pre(p["hc_attn"], streams, cfg)
    h = rmsnorm(h, p["ln1"], cfg.eps)
    if ltype == "linear_attention":
        h, new_cache = kda_attention(p["attn"], h, cfg, cache, use_recurrent, length, hist)
    else:
        h, new_cache = mla_attention(p["attn"], h, cfg, cache, pos0, sparse, cap, length, hist)
    streams = hc_post(post, comb, h, streams)
    post, comb, h = hc_pre(p["hc_ffn"], streams, cfg)
    h = rmsnorm(h, p["ln2"], cfg.eps)
    if mtype == "sparse":
        h = moe_dense(p["mlp"], h, cfg, fetch, layer)
    else:
        h = mlp(p["mlp"], h, cfg)
    streams = hc_post(post, comb, h, streams)
    return streams, new_cache


def decoder_layer_rows(p, streams, cfg: Cfg, ltype, mtype, caches, pos, fetch=None, sparse="auto", layer=0, cap=None):
    """Recurrent (decode) step of B INDEPENDENT streams: streams [B,T,hc,D]; caches[b] = row b's own cache (batch
    dim 1), pos[b] its position (traced scalar). Attention runs per row on its own cache, everything else (mHC,
    projections, MoE) batched over the rows. Returns (streams, [new cache per row])."""
    post, comb, h = hc_pre(p["hc_attn"], streams, cfg)
    h = rmsnorm(h, p["ln1"], cfg.eps)
    if ltype == "linear_attention":
        h, new = kda_attention_rows(p["attn"], h, cfg, caches)
    else:
        h, new = mla_attention_rows(p["attn"], h, cfg, caches, pos, sparse, cap)
    streams = hc_post(post, comb, h, streams)
    post, comb, h = hc_pre(p["hc_ffn"], streams, cfg)
    h = rmsnorm(h, p["ln2"], cfg.eps)
    h = moe_dense(p["mlp"], h, cfg, fetch, layer) if mtype == "sparse" else mlp(p["mlp"], h, cfg)
    return hc_post(post, comb, h, streams), new


def layer_pre(p, streams, cfg: Cfg, ltype, mtype, cache=None, pos0=0, use_recurrent=False, sparse="auto", cap=None,
              length=None):
    """First half of a decoder layer up to (and including) MoE routing. Returns
    (streams_after_attn, h [B,T,D] normed MLP input, post, comb, new_cache, idx [N,k] or None, w [N,k] or None)."""
    post, comb, h = hc_pre(p["hc_attn"], streams, cfg)
    h = rmsnorm(h, p["ln1"], cfg.eps)
    if ltype == "linear_attention":
        h, new_cache = kda_attention(p["attn"], h, cfg, cache, use_recurrent, length)
    else:
        h, new_cache = mla_attention(p["attn"], h, cfg, cache, pos0, sparse, cap, length)
    streams = hc_post(post, comb, h, streams)
    post, comb, h = hc_pre(p["hc_ffn"], streams, cfg)
    h = rmsnorm(h, p["ln2"], cfg.eps)
    idx = w = None
    if mtype == "sparse":
        idx, w = moe_route(p["mlp"], h.reshape(-1, h.shape[-1]), cfg)
    return streams, h, post, comb, new_cache, idx, w


def layer_post(p, streams, h, post, comb, cfg: Cfg, mtype, gu_w=None, dn_w=None, w=None, idx_local=None):
    """Second half: MLP / experts (weights already gathered for this layer's routing) + residual update.
    If idx_local is given the experts are grouped ([Eh,...] tables + local ids), else per-token ([N,k,...])."""
    B, T, D = h.shape
    if mtype == "sparse":
        xf = h.reshape(-1, D)
        if idx_local is not None:
            y = moe_apply_grouped(xf, gu_w, dn_w, idx_local, w, cfg.swiglu_limit)
        else:
            y = moe_apply(xf, gu_w, dn_w, w, cfg.swiglu_limit)
        y = cfg.reduce(y + mlp_partial(p["mlp"]["shared"], xf, cfg)).reshape(B, T, D)
    else:
        y = mlp(p["mlp"], h, cfg)
    return hc_post(post, comb, y, streams)


HIST_KEYS = {"state_hist": "state", "conv_hist": "conv", "tk_hist": "tk", "tg_hist": "tg"}


def split_hist(cache):
    """cache dict with *_hist entries -> (cache without them, hist dict or None)."""
    hist = {k: cache[k] for k in cache if k in HIST_KEYS}
    return {k: v for k, v in cache.items() if k not in HIST_KEYS}, (hist or None)


def rollback_states(hists, n_keep, dtypes):
    """Speculative verify of T tokens produced `hists` (per layer: the recurrent/tail state after each token, or
    None); return per layer the small state entries as after the first `n_keep` (traced, 1..T) tokens (the
    positional cache rows beyond are overwritten later). `dtypes`: per layer {key: dtype} of the cache entries."""
    out = []
    for h, dt in zip(hists, dtypes):
        if not h:
            out.append(None); continue
        out.append({k: lax.dynamic_index_in_dim(h[hk], n_keep - 1, axis=0, keepdims=False).astype(dt[k])
                    for hk, k in HIST_KEYS.items() if hk in h})
    return out


def mtp_layer(p, embeds, h_prev, cfg: Cfg, cache=None, pos0=0, fetch=None, sparse="auto", cap=None, length=None,
              layer=None):
    """GLM-5 MTP (NextN) block (vLLM `glm5next/nvidia/mtp.py`): x = eh_proj(cat(rmsnorm(embeds, enorm),
    rmsnorm(h_prev, hnorm))) — embeds [B,T,D] of the token AFTER each position (zero at position 0), h_prev [B,T,D]
    the main model's post-norm hidden at that position — then one plain pre-norm MLA + MoE block (no
    hyper-connections) and the shared-head norm. Returns (h [B,T,D] for the shared lm_head, new attention cache)."""
    x = jnp.concatenate([rmsnorm(embeds.astype(cfg.dtype), p["enorm"], cfg.eps),
                         rmsnorm(h_prev.astype(cfg.dtype), p["hnorm"], cfg.eps)], -1) @ p["eh_proj"]
    h = rmsnorm(x, p["ln1"], cfg.eps)
    a, new_cache = mla_attention(p["attn"], h, cfg, cache, pos0, sparse, cap, length)
    x = x + a
    h = rmsnorm(x, p["ln2"], cfg.eps)
    x = x + moe_dense(p["mlp"], h, cfg, fetch, layer)
    return rmsnorm(x, p["head_norm"], cfg.eps), new_cache


def forward(params, tokens, cfg: Cfg, caches=None, pos0=0, use_recurrent=False, fetch=None, sparse="auto",
            embeds=None, cap=None, length=None):
    """tokens [B,T] -> (hidden [B,T,D] after final norm, new_caches list). Prefill: caches=None.
    `embeds` [B,T,D] overrides the embedding lookup (used by the TP engine with a vocab-sharded table)."""
    x = params["embed"][tokens].astype(cfg.dtype) if embeds is None else embeds.astype(cfg.dtype)
    streams = jnp.broadcast_to(x[:, :, None, :], x.shape[:2] + (cfg.hc, x.shape[-1]))
    new_caches = []
    for i in range(cfg.n_layers):
        c = None if caches is None else caches[i]
        streams, nc = decoder_layer(params["layers"][i], streams, cfg, cfg.layer_types[i], cfg.mlp_types[i],
                                    c, pos0, use_recurrent, fetch, sparse, layer=i, cap=cap, length=length)
        new_caches.append(nc)
    h = rmsnorm(streams.mean(2).astype(cfg.dtype), params["norm"], cfg.eps)
    return h, new_caches


def logits(params, h):
    return h @ params["lm_head"]
'''
FILES["glm53/pallas_moe.py"] = r'''"""Pallas TPU kernel: fused codebook-dequant + matvec of routed experts straight from the planar HBM tables.

`moe_matvec(planes, qtype, nblk, idx, X)` computes, for every expert slot i (one (token, expert) pair):
    out[i, r] = sum_n  W_{idx[i]}[r, n] * x_i[n]              (W dequantized on the fly, never written anywhere)
where X[i, w, c] = x_i[input(w, c)] is the per-word activation matrix of glm53.planes (see `planes.pm_x`).

Grid = expert slots; the expert id comes from a scalar-prefetch operand so each grid step DMAs exactly the planes
of the chosen expert (double-buffered by Pallas). Inside, one (8, 128) vreg of the qs plane = 8 packed words x 128
matrix rows; the codebook index of every group is looked up with in-vreg lane gathers (`jnp.take_along_axis` along
lanes on the (8, 128)-tiled code table, one gather per 128-entry table row + a select tree over the high index
bits), the 2-/3-bit levels and sign bits are unpacked with shifts, and the values are multiplied-accumulated on the
VPU against the activation broadcast along lanes: acc[s, r] += v[s, r] * X[8b + s, c]. The sublane sum of the
accumulator over all words is the output row block. Everything is 32-bit (no packed dtypes), loads are (n, 128)
windows at 8-aligned sublane offsets, so the kernel also runs under `interpret=True` on CPU for the tests.

Codebook tables: `code_table(qtype)` packs each grid entry as GROUP_W[qtype] level indices (2 or 3 bits each) into
one int32; entries are laid out [n_entries // 128, 128].
"""
import functools
import numpy as np
import jax
import jax.numpy as jnp
from jax import lax
from jax.experimental import pallas as pl
from jax.experimental.pallas import tpu as pltpu

from glm53 import iqquant as Q
from glm53 import planes as PL
from glm53 import model as M

LANES = 128
SUB = 8


@functools.lru_cache(None)
def code_table(qtype):
    """int32 [n // 128, 128]: entry e -> sum_k level_k << (lb * k)."""
    grid, levels = Q.GRIDS[qtype], Q.LEVELS[qtype]
    lb = int(np.ceil(np.log2(len(levels))))
    lv = np.searchsorted(levels, grid)                                   # [n, w]
    assert np.array_equal(levels[lv], grid)
    code = np.zeros(grid.shape[0], np.int64)
    for k in range(grid.shape[1]):
        code |= lv[:, k].astype(np.int64) << (lb * k)
    return code.astype(np.int32).reshape(-1, LANES)


@functools.lru_cache(None)
def kv16_table():
    t = np.zeros((1, LANES), np.float32)
    t[0, :16] = Q.KVALUES_IQ4NL
    return t


def _f16_bits_to_f32(h):
    """u32 holding f16 bits (low 16) -> f32 (normal + subnormal + zero; no inf/nan expected)."""
    h = h & 0xFFFF
    s = (h >> 15) & 1
    e = (h >> 10) & 31
    m = h & 1023
    bits = (s << 31) | ((e + 112) << 23) | (m << 13)
    normal = lax.bitcast_convert_type(bits.astype(jnp.uint32), jnp.float32)
    sub = m.astype(jnp.int32).astype(jnp.float32) * (2.0 ** -24)
    sub = jnp.where(s == 1, -sub, sub)
    return jnp.where(e == 0, sub, normal)


def _rows8(src, reps):
    """src (n, 128) with n * reps == 8 -> (8, 128) whose sublane s is src[s // reps]."""
    n = src.shape[0]
    if n == 1:
        return jnp.broadcast_to(src, (SUB, LANES))
    row = lax.broadcasted_iota(jnp.int32, (SUB, LANES), 0)
    out = jnp.broadcast_to(src[n - 1:n], (SUB, LANES))
    for r in range(n - 2, -1, -1):
        out = jnp.where(row < (r + 1) * reps, jnp.broadcast_to(src[r:r + 1], (SUB, LANES)), out)
    return out


def _sub_iota():
    return lax.broadcasted_iota(jnp.uint32, (SUB, LANES), 0)


def _lookup(tb, idx):
    """tb: list of (8,128) int32 table rows (128 entries each); idx (8,128) int32 -> codes (8,128) int32."""
    lo = idx & (LANES - 1)
    hi = idx >> 7
    vals = [jnp.take_along_axis(t, lo, axis=1) for t in tb]
    bit = 0
    while len(vals) > 1:
        sel = ((hi >> bit) & 1) == 1
        vals = [jnp.where(sel, vals[2 * m + 1], vals[2 * m]) for m in range(len(vals) // 2)]
        bit += 1
    return vals[0]


def _signed(v_int, sgn_bits, k):
    v = v_int.astype(jnp.int32).astype(jnp.float32)
    return jnp.where(((sgn_bits >> k) & 1) == 1, -v, v)


# ------------------------------------------------------------------------------------------ per-format word decoders
# `rows(name, r0, n)` returns rows r0..r0+n-1 of that plane for the current expert as an (n, 128) value (static lanes).
# Each decoder returns a list of (scale (8,128) f32, [(c, value (8,128) f32), ...]) for the 8 words 8b..8b+7.
def _decode_iq2_s(b, rows, tb):
    q = rows("qs", 8 * b, 8)
    sgw = rows("sg", 8 * b, 8)
    sub = _sub_iota()
    shift8 = (sub & 3) * 8
    qhw = (_rows8(rows("qh", 2 * b, 2), 4) >> shift8) & 255                 # qh byte of word 8b+s
    scw = (_rows8(rows("sc", 2 * b, 2), 4) >> shift8) & 255                 # sc byte of word 8b+s
    dw = rows("d", b // 2, 1) >> (16 * (b % 2))                             # block b for all 8 words
    dv = jnp.broadcast_to(_f16_bits_to_f32(dw), (SUB, LANES))
    scale = [dv * (0.5 + ((scw >> (4 * h)) & 15).astype(jnp.int32).astype(jnp.float32)) * 0.25 for h in range(2)]
    groups = [(scale[0], []), (scale[1], [])]
    for j in range(4):
        idx = (((q >> (8 * j)) & 255) | (((qhw >> (2 * j)) & 3) << 8)).astype(jnp.int32)
        code = _lookup(tb, idx)
        sgn = (sgw >> (8 * j)) & 255
        for k in range(8):
            lv = (code >> (2 * k)) & 3
            v = 8 + 17 * lv + (lv >> 1)
            groups[j // 2][1].append((8 * j + k, _signed(v, sgn, k)))
    return groups


def _decode_iq3_s(b, rows, tb):
    q = rows("qs", 8 * b, 8)
    sub = _sub_iota()
    sgh = (_rows8(rows("sg", 4 * b, 4), 2) >> ((sub & 1) * 16)) & 0xFFFF   # 16 sign bits of word 8b+s
    qhn = (_rows8(rows("qh", b, 1), 8) >> (sub * 4)) & 15                    # 4 high bits of word 8b+s
    scn = (_rows8(rows("sc", b // 2, 1), 8) >> ((4 * (b % 2) + (sub >> 1)) * 4)) & 15
    dw = rows("d", b // 4, 1) >> (16 * ((b // 2) % 2))
    dv = jnp.broadcast_to(_f16_bits_to_f32(dw), (SUB, LANES))
    scale = dv * (1.0 + 2.0 * scn.astype(jnp.int32).astype(jnp.float32))
    items = []
    for j in range(4):
        idx = (((q >> (8 * j)) & 255) | (((qhn >> j) & 1) << 8)).astype(jnp.int32)
        code = _lookup(tb, idx)
        sgn = (sgh >> (4 * j)) & 15
        for k in range(4):
            lv = (code >> (3 * k)) & 7
            items.append((4 * j + k, _signed(2 * lv + 1, sgn, k)))
    return [(scale, items)]


def _decode_iq4_xs(b, rows, tb):
    q = rows("qs", 8 * b, 8)
    sub = _sub_iota()
    i = (2 * b + (sub >> 2)) % 8                                             # sub-block of word 8b+s within its block
    slw = (_rows8(rows("sl", b // 4, 1), 8) >> (4 * i)) & 15
    shw = (_rows8(rows("sh", b // 4, 1), 8) >> (2 * i)) & 3
    dw = rows("d", b // 8, 1) >> (16 * ((b // 4) % 2))
    dv = jnp.broadcast_to(_f16_bits_to_f32(dw), (SUB, LANES))
    scale = dv * ((slw | (shw << 4)).astype(jnp.int32) - 32).astype(jnp.float32)
    kv = tb[0]                                                               # (8,128) f32, entries at lanes 0..15
    items = []
    for pos in range(2):
        for j in range(4):
            nib = ((q >> (8 * j + 4 * pos)) & 15).astype(jnp.int32)
            items.append((4 * pos + j, jnp.take_along_axis(kv, nib, axis=1)))
    return [(scale, items)]


def _kq_scales(rows, b, sub):
    """K-quant scale bytes for the 8 words of vreg b (half n = b % 2 of block b // 2): per shift s an (8,128) u32 with
    the scale byte `is` = 8 n + 2 s + (sublane >= 4) of the block's 16, plus the block's d word (1,128)."""
    blk, n = b // 2, b % 2
    scw = rows("sc", 4 * blk, 4)                                                 # (4,128): the 16 scale bytes
    hi = (sub >= 4).astype(jnp.uint32)
    out = []
    for s_ in range(4):
        r = 2 * n + s_ // 2
        row = jnp.broadcast_to(scw[r:r + 1], (SUB, LANES))
        out.append((row >> (8 * (2 * (s_ % 2) + hi))) & 255)
    return out, rows("d", blk, 1)


def _decode_q2_k(b, rows, tb):
    q = rows("qs", 8 * b, 8)
    sub = _sub_iota()
    sc, dw = _kq_scales(rows, b, sub)
    d = jnp.broadcast_to(_f16_bits_to_f32(dw), (SUB, LANES))
    dmin = jnp.broadcast_to(_f16_bits_to_f32(dw >> 16), (SUB, LANES))
    items = []
    for s_ in range(4):
        dl = d * (sc[s_] & 15).astype(jnp.int32).astype(jnp.float32)
        ml = dmin * (sc[s_] >> 4).astype(jnp.int32).astype(jnp.float32)
        for j in range(4):
            v = ((q >> (8 * j + 2 * s_)) & 3).astype(jnp.int32).astype(jnp.float32)
            items.append((4 * s_ + j, dl * v - ml))
    return [(jnp.ones((SUB, LANES), jnp.float32), items)]


def _decode_q3_k(b, rows, tb):
    q = rows("qs", 8 * b, 8)
    sub = _sub_iota()
    blk, n = b // 2, b % 2
    hm = rows("hm", 8 * blk, 8)                                                  # (8,128): u32 q = hmask bytes 4q..4q+3
    sc, dw = _kq_scales(rows, b, sub)
    d = jnp.broadcast_to(_f16_bits_to_f32(dw), (SUB, LANES))
    items = []
    for s_ in range(4):
        dl = d * ((sc[s_] & 255).astype(jnp.int32) - 32).astype(jnp.float32)
        for j in range(4):
            v = ((q >> (8 * j + 2 * s_)) & 3).astype(jnp.int32)
            hb = ((hm >> (8 * j + 4 * n + s_)) & 1).astype(jnp.int32)
            items.append((4 * s_ + j, dl * (v - 4 + 4 * hb).astype(jnp.float32)))
    return [(jnp.ones((SUB, LANES), jnp.float32), items)]


DECODE = {"IQ2_S": _decode_iq2_s, "IQ3_S": _decode_iq3_s, "IQ4_XS": _decode_iq4_xs, "Q2_K": _decode_q2_k,
          "Q3_K": _decode_q3_k}


def _table_arrays(qtype):
    if qtype == "IQ4_XS":
        return jnp.asarray(kv16_table())
    if qtype in ("Q2_K", "Q3_K"):
        return jnp.zeros((1, LANES), jnp.int32)                              # no codebook
    return jnp.asarray(code_table(qtype))


def _pick_rows(block, off, r0, n):
    """block (8, 128) value; rows off+r0 .. off+r0+n-1 where `off` is a traced scalar in [0, 8): select tree
    (Mosaic has no unaligned dynamic sublane loads)."""
    outs = []
    for r in range(n):
        want = off + r0 + r
        v = block[0:1]
        for q in range(1, SUB):
            v = jnp.where(want == q, block[q:q + 1], v)
        outs.append(v)
    return outs[0] if n == 1 else jnp.concatenate(outs, axis=0)


# ------------------------------------------------------------------------------------------------------- the kernel
def moe_matvec(planes, qtype, nblk, idx, X, *, interpret=False, lane_block=None):
    """planes: {k: u32 [E*rows_p, R] or [1, E*rows_p, R]}; idx int32 [Nk]; X f32 [Nk, W, C] -> f32 [Nk, R].

    Grid = (expert slot, lane block of `lane_block` matrix rows); every ref access uses static indices (Mosaic rejects
    dynamic sublane/lane offsets that are not provably tile aligned): planes with fewer than 8 rows per expert are
    fetched as their 8-row block and the expert's rows are picked with selects on the scalar-prefetched offset."""
    keys = PL.PLANE_KEYS[qtype]
    rows_p = PL.plane_rows(qtype, nblk)
    W = PL.WORDS_PER_BLOCK[qtype] * nblk
    C = PL.PLANES_PER_WORD[qtype]
    first = planes[keys[0]]
    lead = first.ndim == 3
    R = first.shape[-1]
    Nk = idx.shape[0]
    assert X.shape == (Nk, W, C), (X.shape, (Nk, W, C))
    assert R % LANES == 0 and W % SUB == 0, (R, W)
    LB = lane_block or min(R, 1024)
    assert R % LB == 0 and LB % LANES == 0, (R, LB)
    n_lb = R // LB
    n_vreg = LB // LANES
    tbl = _table_arrays(qtype)
    n_tbl = tbl.shape[0]
    blocks = {k: (rows_p[k] if rows_p[k] % SUB == 0 else SUB) for k in keys}
    decode = DECODE[qtype]

    def kernel(idx_ref, *refs):
        plane_refs = dict(zip(keys, refs[:len(keys)]))
        tbl_ref, x_ref, o_ref = refs[len(keys):]
        i = pl.program_id(0)
        e = idx_ref[i]
        offs = {k: (lax.rem(e * rows_p[k], SUB) if rows_p[k] % SUB else None) for k in keys}
        if qtype == "IQ4_XS":
            tb = [jnp.broadcast_to(tbl_ref[0:1, :], (SUB, LANES))]
        else:
            tb = [jnp.broadcast_to(tbl_ref[r:r + 1, :], (SUB, LANES)) for r in range(n_tbl)]
        for v in range(n_vreg):
            lanes = slice(v * LANES, (v + 1) * LANES)

            def rows(k, r0, n, lanes=lanes):
                if offs[k] is None:
                    return plane_refs[k][r0:r0 + n, lanes]
                return _pick_rows(plane_refs[k][0:SUB, lanes], offs[k], r0, n)

            out = jnp.zeros((SUB, LANES), jnp.float32)
            for b in range(W // SUB):
                xb = x_ref[SUB * b:SUB * (b + 1), :]                                    # (8, C)
                for scale, items in decode(b, rows, tb):
                    acc = jnp.zeros((SUB, LANES), jnp.float32)
                    for c, val in items:
                        acc = acc + val * jnp.broadcast_to(xb[:, c:c + 1], (SUB, LANES))
                    out = out + acc * scale
            o_ref[:, lanes] = jnp.sum(out, axis=0, keepdims=True)

    def plane_spec(k):
        B = blocks[k]
        if lead:
            return pl.BlockSpec((None, B, LB), lambda i, cb, idx_ref: (0, (idx_ref[i] * rows_p[k]) // B, cb))
        return pl.BlockSpec((B, LB), lambda i, cb, idx_ref: ((idx_ref[i] * rows_p[k]) // B, cb))

    in_specs = [plane_spec(k) for k in keys]
    in_specs += [pl.BlockSpec(tbl.shape, lambda i, cb, idx_ref: (0, 0)),
                 pl.BlockSpec((None, W, C), lambda i, cb, idx_ref: (i, 0, 0))]
    out_spec = pl.BlockSpec((None, 1, LB), lambda i, cb, idx_ref: (i, 0, cb))
    grid_spec = pltpu.PrefetchScalarGridSpec(num_scalar_prefetch=1, grid=(Nk, n_lb), in_specs=in_specs,
                                             out_specs=out_spec)
    fn = pl.pallas_call(kernel, grid_spec=grid_spec, out_shape=jax.ShapeDtypeStruct((Nk, 1, R), jnp.float32),
                        interpret=interpret,
                        compiler_params=pltpu.CompilerParams(dimension_semantics=("arbitrary", "arbitrary")))
    args = [planes[k] for k in keys] + [tbl, X]
    return fn(idx, *args).reshape(Nk, R)


def moe_matvec_ref(planes, qtype, nblk, idx, X):
    """Pure-JAX reference (gathers the experts, dequantizes with glm53.planes, einsum)."""
    keys = PL.PLANE_KEYS[qtype]
    rows_p = PL.plane_rows(qtype, nblk)
    sel = {}
    for k in keys:
        a = planes[k]
        a = a[0] if a.ndim == 3 else a
        R = a.shape[-1]
        a = a.reshape(-1, rows_p[k], R)
        sel[k] = jnp.take(a, idx, axis=0)                                    # [Nk, rows_p, R]
    V = PL.dequant_planes(sel, qtype, nblk)                                 # [Nk, C*W, R]
    x_pm = jnp.swapaxes(X, 1, 2).reshape(X.shape[0], -1)                   # [Nk, C*W]
    return jnp.einsum("ni,nir->nr", x_pm, V, precision="highest")


# ------------------------------------------------------------------------------------------ prefill (sweep) kernels
# Dequantize each expert ONCE into (K, 128) bf16 tiles of W^T in VMEM and MXU-matmul all T tokens against them.
# Tile rows are ordered (word block b, value plane c, word s): input(8b+s, c) -> row (b*C + c)*8 + s; `pm_mxu`
# permutes activations to that order. Expert slots come from scalar-prefetched `ids` (unique active experts first,
# inactive slots repeat the last active id so their planes are never re-DMA'd) and `n_active` gates the compute.
def pm_mxu(qtype, x):
    """x [..., n] natural -> [..., n] in the kernel tile order (b, c, s)."""
    X = PL.pm_x(qtype, x)                                                    # [..., W, C]
    lead = X.shape[:-2]
    W, C = X.shape[-2:]
    y = X.reshape(lead + (W // SUB, SUB, C))
    y = jnp.swapaxes(y, -1, -2)                                              # [..., W/8, C, 8]
    return y.reshape(lead + (W * C,))


def _tiles(decode, rows, tb, b0, nb, dtype=jnp.bfloat16):
    """(nb*C*8, 128) tile of scaled W^T rows for word blocks b0..b0+nb-1 at the current lanes, in `dtype`."""
    parts = []
    for b in range(b0, b0 + nb):
        items = []
        for scale, its in decode(b, rows, tb):
            items += [(c, v * scale) for c, v in its]
        items.sort(key=lambda t: t[0])
        parts += [v for _, v in items]
    return jnp.concatenate(parts, axis=0).astype(dtype)


def _dot(a, b):
    prec = lax.Precision.HIGHEST if a.dtype == jnp.float32 else None
    return jnp.dot(a, b, preferred_element_type=jnp.float32, precision=prec)


def active_slots(idx, E):
    """idx int32 [N, k] -> (ids int32 [E], n_active int32): unique routed experts first (ascending), the rest of the
    slots repeat the last active id."""
    hit = (jax.nn.one_hot(idx.reshape(-1), E, dtype=jnp.int32).sum(0) > 0)          # [E] bool
    order = jnp.argsort(jnp.where(hit, 0, 1) * E + jnp.arange(E))                     # active ids first, ascending
    n_active = hit.sum().astype(jnp.int32)
    last = order[jnp.maximum(n_active - 1, 0)]
    ids = jnp.where(jnp.arange(E) < n_active, order, last).astype(jnp.int32)
    return ids, n_active


def _k_group(qtype):
    """word blocks per MXU tile so that K >= 128."""
    return max(1, 128 // (PL.PLANES_PER_WORD[qtype] * SUB))


def _probe_tiles(probe, tiles_fn, K, dtype):
    """Cost-split probes: 'mxu' replaces the dequantized tile by a constant (matmul cost only), 'deq' keeps the
    dequant but returns None so the caller skips the dot (dequant cost only)."""
    if probe == "mxu":
        return jnp.full((K, LANES), 0.01, dtype)
    return tiles_fn()


def moe_sweep_gateup(planes_g, planes_u, qtype, nblk, ids, n_active, xk, limit, *, interpret=False, lane_block=None,
                     t_block=512, vmem_mb=48, probe=None):
    """planes_{g,u}: {k: u32 [1, E*rows_p, R]} (same qtype); ids [E] int32; n_active int32 scalar; xk [T, n_in] in
    pm_mxu order (bf16 on TPU; f32 -> f32 tiles + HIGHEST-precision dots for the CPU tests) ->
    h [E, T, R] (xk.dtype) = swiglu_clamped(x @ gate_e^T, x @ up_e^T) for active slots, 0 elsewhere.
    `probe` (timing only, wrong numbers): 'mxu' = constant tiles, 'deq' = dequant without the matmul."""
    dtype = xk.dtype
    keys = PL.PLANE_KEYS[qtype]
    rows_p = PL.plane_rows(qtype, nblk)
    W = PL.WORDS_PER_BLOCK[qtype] * nblk
    C = PL.PLANES_PER_WORD[qtype]
    E = ids.shape[0]
    T, n_in = xk.shape
    assert n_in == nblk * PL.QK, (n_in, nblk)
    first = planes_g[keys[0]]
    lead = first.ndim == 3
    R = first.shape[-1]
    LB = lane_block or min(R, 1024)
    TB = min(T, t_block)
    assert T % TB == 0 and R % LB == 0
    n_lb, n_vreg, n_tb = R // LB, LB // LANES, T // TB
    nbg = _k_group(qtype)
    K = nbg * C * SUB
    tbl = _table_arrays(qtype)
    n_tbl = tbl.shape[0]
    blocks = {k: (rows_p[k] if rows_p[k] % SUB == 0 else SUB) for k in keys}
    decode = DECODE[qtype]
    nk = len(keys)

    def kernel(ids_ref, nact_ref, *refs):
        g_refs = dict(zip(keys, refs[:nk]))
        u_refs = dict(zip(keys, refs[nk:2 * nk]))
        tbl_ref, x_ref, o_ref, acc_g, acc_u = refs[2 * nk:]
        e = pl.program_id(1)
        eid = ids_ref[e]
        offs = {k: (lax.rem(eid * rows_p[k], SUB) if rows_p[k] % SUB else None) for k in keys}
        if qtype == "IQ4_XS":
            tb = [jnp.broadcast_to(tbl_ref[0:1, :], (SUB, LANES))]
        else:
            tb = [jnp.broadcast_to(tbl_ref[r:r + 1, :], (SUB, LANES)) for r in range(n_tbl)]

        @pl.when(e >= nact_ref[0])
        def _inactive():
            o_ref[...] = jnp.zeros(o_ref.shape, o_ref.dtype)

        @pl.when(e < nact_ref[0])
        def _active():
            for v in range(n_vreg):
                lanes = slice(v * LANES, (v + 1) * LANES)

                def rows_of(prefs, lanes=lanes):
                    def rows(k, r0, n):
                        if offs[k] is None:
                            return prefs[k][r0:r0 + n, lanes]
                        return _pick_rows(prefs[k][0:SUB, lanes], offs[k], r0, n)
                    return rows

                rg, ru = rows_of(g_refs), rows_of(u_refs)
                ag = jnp.zeros((TB, LANES), jnp.float32)
                au = jnp.zeros((TB, LANES), jnp.float32)
                for b0 in range(0, W // SUB, nbg):
                    k0 = b0 * C * SUB
                    xs = x_ref[:, k0:k0 + K]                                              # (TB, K) bf16
                    tg = _probe_tiles(probe, lambda: _tiles(decode, rg, tb, b0, nbg, dtype), K, dtype)
                    tu = _probe_tiles(probe, lambda: _tiles(decode, ru, tb, b0, nbg, dtype), K, dtype)
                    if probe == "deq":
                        ag = ag + jnp.sum(tg, axis=0, keepdims=True).astype(jnp.float32)
                        au = au + jnp.sum(tu, axis=0, keepdims=True).astype(jnp.float32)
                        continue
                    ag = ag + _dot(xs, tg)
                    au = au + _dot(xs, tu)
                acc_g[:, lanes] = ag
                acc_u[:, lanes] = au
            o_ref[...] = M.swiglu_clamped(acc_g[...], acc_u[...], limit).astype(o_ref.dtype)

    def plane_spec(k):
        B = blocks[k]
        if lead:
            return pl.BlockSpec((None, B, LB), lambda t, e, cb, ids_ref, n_ref: (0, (ids_ref[e] * rows_p[k]) // B, cb))
        return pl.BlockSpec((B, LB), lambda t, e, cb, ids_ref, n_ref: ((ids_ref[e] * rows_p[k]) // B, cb))

    in_specs = [plane_spec(k) for k in keys] * 2
    in_specs += [pl.BlockSpec(tbl.shape, lambda t, e, cb, ids_ref, n_ref: (0, 0)),
                 pl.BlockSpec((TB, n_in), lambda t, e, cb, ids_ref, n_ref: (t, 0))]
    out_spec = pl.BlockSpec((None, TB, LB), lambda t, e, cb, ids_ref, n_ref: (e, t, cb))
    grid_spec = pltpu.PrefetchScalarGridSpec(num_scalar_prefetch=2, grid=(n_tb, E, n_lb), in_specs=in_specs,
                                             out_specs=out_spec,
                                             scratch_shapes=[pltpu.VMEM((TB, LB), jnp.float32)] * 2)
    fn = pl.pallas_call(kernel, grid_spec=grid_spec, out_shape=jax.ShapeDtypeStruct((E, T, R), dtype),
                        interpret=interpret,
                        compiler_params=pltpu.CompilerParams(dimension_semantics=("arbitrary",) * 3,
                                                             vmem_limit_bytes=vmem_mb << 20))
    args = [planes_g[k] for k in keys] + [planes_u[k] for k in keys] + [tbl, xk]
    return fn(ids, n_active.reshape(1), *args)


def moe_sweep_down(planes, qtype, nblk, ids, n_active, hk, *, interpret=False, lane_block=None, t_block=512,
                   vmem_mb=48, probe=None):
    """planes {k: u32 [1, E*rows_p, R]}; hk [E, T, n_in] (pm_mxu order, routing weights folded in; bf16 or f32) ->
    y f32 [T, R] = sum over active slots of hk[e] @ down_e^T."""
    dtype = hk.dtype
    keys = PL.PLANE_KEYS[qtype]
    rows_p = PL.plane_rows(qtype, nblk)
    W = PL.WORDS_PER_BLOCK[qtype] * nblk
    C = PL.PLANES_PER_WORD[qtype]
    E, T, n_in = hk.shape
    assert n_in == nblk * PL.QK and ids.shape == (E,)
    first = planes[keys[0]]
    lead = first.ndim == 3
    R = first.shape[-1]
    LB = lane_block or min(R, 1024)
    TB = min(T, t_block)
    assert T % TB == 0 and R % LB == 0
    n_lb, n_vreg, n_tb = R // LB, LB // LANES, T // TB
    nbg = _k_group(qtype)
    K = nbg * C * SUB
    tbl = _table_arrays(qtype)
    n_tbl = tbl.shape[0]
    blocks = {k: (rows_p[k] if rows_p[k] % SUB == 0 else SUB) for k in keys}
    decode = DECODE[qtype]
    nk = len(keys)

    def kernel(ids_ref, nact_ref, *refs):
        prefs = dict(zip(keys, refs[:nk]))
        tbl_ref, h_ref, o_ref = refs[nk:]
        e = pl.program_id(2)
        eid = ids_ref[e]
        offs = {k: (lax.rem(eid * rows_p[k], SUB) if rows_p[k] % SUB else None) for k in keys}
        if qtype == "IQ4_XS":
            tb = [jnp.broadcast_to(tbl_ref[0:1, :], (SUB, LANES))]
        else:
            tb = [jnp.broadcast_to(tbl_ref[r:r + 1, :], (SUB, LANES)) for r in range(n_tbl)]

        @pl.when(e == 0)
        def _init():
            o_ref[...] = jnp.zeros(o_ref.shape, o_ref.dtype)

        @pl.when(e < nact_ref[0])
        def _active():
            for v in range(n_vreg):
                lanes = slice(v * LANES, (v + 1) * LANES)

                def rows(k, r0, n, lanes=lanes):
                    if offs[k] is None:
                        return prefs[k][r0:r0 + n, lanes]
                    return _pick_rows(prefs[k][0:SUB, lanes], offs[k], r0, n)

                acc = o_ref[:, lanes]
                for b0 in range(0, W // SUB, nbg):
                    k0 = b0 * C * SUB
                    td = _probe_tiles(probe, lambda: _tiles(decode, rows, tb, b0, nbg, dtype), K, dtype)
                    if probe == "deq":
                        acc = acc + jnp.sum(td, axis=0, keepdims=True).astype(jnp.float32)
                        continue
                    acc = acc + _dot(h_ref[:, k0:k0 + K], td)
                o_ref[:, lanes] = acc

    def plane_spec(k):
        B = blocks[k]
        if lead:
            return pl.BlockSpec((None, B, LB), lambda t, cb, e, ids_ref, n_ref: (0, (ids_ref[e] * rows_p[k]) // B, cb))
        return pl.BlockSpec((B, LB), lambda t, cb, e, ids_ref, n_ref: ((ids_ref[e] * rows_p[k]) // B, cb))

    in_specs = [plane_spec(k) for k in keys]
    in_specs += [pl.BlockSpec(tbl.shape, lambda t, cb, e, ids_ref, n_ref: (0, 0)),
                 pl.BlockSpec((None, TB, n_in), lambda t, cb, e, ids_ref, n_ref: (e, t, 0))]
    out_spec = pl.BlockSpec((TB, LB), lambda t, cb, e, ids_ref, n_ref: (t, cb))
    grid_spec = pltpu.PrefetchScalarGridSpec(num_scalar_prefetch=2, grid=(n_tb, n_lb, E), in_specs=in_specs,
                                             out_specs=out_spec)
    fn = pl.pallas_call(kernel, grid_spec=grid_spec, out_shape=jax.ShapeDtypeStruct((T, R), jnp.float32),
                        interpret=interpret,
                        compiler_params=pltpu.CompilerParams(dimension_semantics=("arbitrary",) * 3,
                                                             vmem_limit_bytes=vmem_mb << 20))
    return fn(ids, n_active.reshape(1), *[planes[k] for k in keys], tbl, hk)


# ------------------------------------------------------------------------------------- grouped (ragged) prefill GEMM
# The dense sweep multiplies every 512-token chunk against every active expert (masked): on real text ~all 288 experts
# are active, so ~36x the useful MXU work, each expert dequantized once per chunk, and an [E, T, ml] intermediate.
# Here the (token, expert) slots are SORTED BY EXPERT into blocks of `tm` rows (each expert's rows start at a block
# boundary, zero-padded); the kernels walk the blocks in order, DMA + dequantize an expert's planes ONCE (when the
# block's expert changes; the dequantized W^T tiles stay in VMEM scratch) and multiply only that expert's rows.
def ragged_plan(idx, w, E, tm):
    """Slot layout for the grouped kernels. idx int32 [T, k], w f32 [T, k] -> dict with
    blk_expert int32 [G] (expert of row block g; G = T*k//tm + E blocks of `tm` rows, blocks >= n_blocks are padding
    and repeat the last expert), n_blocks int32 [1], tok_row int32 [Rp] (token of every sorted row, -1 = padding),
    w_row f32 [Rp] (its routing weight), row_slot int32 [T, k] (the row of every (token, slot))."""
    T, k = idx.shape
    S = T * k
    G = -(-S // tm) + E
    Rp = G * tm
    flat = idx.reshape(-1).astype(jnp.int32)
    order = jnp.argsort(flat)
    se = flat[order]
    counts = jnp.sum(jax.nn.one_hot(flat, E, dtype=jnp.int32), axis=0)                    # [E]
    padded = -(-counts // tm) * tm
    pstart = jnp.cumsum(padded) - padded
    ustart = jnp.cumsum(counts) - counts
    row_sorted = pstart[se] + (jnp.arange(S, dtype=jnp.int32) - ustart[se])              # row of sorted slot j
    tok_row = jnp.full((Rp,), -1, jnp.int32).at[row_sorted].set((order // k).astype(jnp.int32))
    w_row = jnp.zeros((Rp,), jnp.float32).at[row_sorted].set(w.reshape(-1)[order].astype(jnp.float32))
    row_slot = jnp.zeros((S,), jnp.int32).at[order].set(row_sorted).reshape(T, k)
    nb = padded // tm
    bstart = jnp.cumsum(nb) - nb
    n_blocks = jnp.sum(nb).astype(jnp.int32)
    g = jnp.arange(G, dtype=jnp.int32)
    be = jnp.sum((g[:, None] >= (bstart + nb)[None, :]).astype(jnp.int32), axis=1)        # experts ended at or before g
    be = jnp.minimum(be, E - 1)
    last = be[jnp.maximum(n_blocks - 1, 0)]
    be = jnp.where(g < n_blocks, be, last).astype(jnp.int32)
    return {"blk_expert": be, "n_blocks": n_blocks.reshape(1), "tok_row": tok_row, "w_row": w_row, "row_slot": row_slot}


def _ragged_call(kernel, planes_list, keys, rows_p, blocks, tbl, act, out_dtype, R, LB, tm, G, scratch, interpret, vmem_mb,
                 be, n_blocks):
    """Shared pallas_call plumbing of the two ragged kernels: grid (lane block, row block)."""
    lead = planes_list[0][keys[0]].ndim == 3
    n_lb = R // LB
    Rp, n_in = act.shape

    def plane_spec(k):
        B = blocks[k]
        if lead:
            return pl.BlockSpec((None, B, LB), lambda cb, g, be_ref, nb_ref: (0, (be_ref[g] * rows_p[k]) // B, cb))
        return pl.BlockSpec((B, LB), lambda cb, g, be_ref, nb_ref: ((be_ref[g] * rows_p[k]) // B, cb))

    in_specs = [plane_spec(k) for k in keys] * len(planes_list)
    in_specs += [pl.BlockSpec(tbl.shape, lambda cb, g, be_ref, nb_ref: (0, 0)),
                 pl.BlockSpec((tm, n_in), lambda cb, g, be_ref, nb_ref: (jnp.minimum(g, nb_ref[0] - 1), 0))]
    out_spec = pl.BlockSpec((tm, LB), lambda cb, g, be_ref, nb_ref: (g, cb))
    grid_spec = pltpu.PrefetchScalarGridSpec(num_scalar_prefetch=2, grid=(n_lb, G), in_specs=in_specs,
                                             out_specs=out_spec, scratch_shapes=scratch)
    fn = pl.pallas_call(kernel, grid_spec=grid_spec, out_shape=jax.ShapeDtypeStruct((Rp, R), out_dtype),
                        interpret=interpret,
                        compiler_params=pltpu.CompilerParams(dimension_semantics=("arbitrary", "arbitrary"),
                                                             vmem_limit_bytes=vmem_mb << 20))
    args = [p[k] for p in planes_list for k in keys] + [tbl, act]
    return fn(be, n_blocks, *args)


def moe_ragged_gateup(planes_g, planes_u, qtype, nblk, blk_expert, n_blocks, xs, limit, *, tm=32, interpret=False,
                      lane_block=None, vmem_mb=48):
    """Grouped prefill GEMM, gate + up. xs [Rp, n_in] in pm_mxu order, rows sorted by expert in blocks of `tm` rows
    (block g = rows g*tm.., expert blk_expert[g]; blocks >= n_blocks are padding) -> h [Rp, R] (xs.dtype) =
    swiglu_clamped(x @ gate_e^T, x @ up_e^T) per row with e = its block's expert; padding blocks are zero."""
    dtype = xs.dtype
    keys = PL.PLANE_KEYS[qtype]
    rows_p = PL.plane_rows(qtype, nblk)
    W = PL.WORDS_PER_BLOCK[qtype] * nblk
    C = PL.PLANES_PER_WORD[qtype]
    Rp, n_in = xs.shape
    assert n_in == nblk * PL.QK and n_in == W * C and Rp % tm == 0, (n_in, nblk, W, C, Rp, tm)
    G = Rp // tm
    assert blk_expert.shape == (G,), (blk_expert.shape, G)
    R = planes_g[keys[0]].shape[-1]
    LB = lane_block or min(R, 1024)
    assert R % LB == 0
    n_vreg = LB // LANES
    nbg = _k_group(qtype)
    K = nbg * C * SUB
    tbl = _table_arrays(qtype)
    n_tbl = tbl.shape[0]
    blocks = {k: (rows_p[k] if rows_p[k] % SUB == 0 else SUB) for k in keys}
    decode = DECODE[qtype]
    nk = len(keys)

    def kernel(be_ref, nb_ref, *refs):
        g_refs = dict(zip(keys, refs[:nk]))
        u_refs = dict(zip(keys, refs[nk:2 * nk]))
        tbl_ref, x_ref, o_ref, wg, wu = refs[2 * nk:]
        g = pl.program_id(1)
        eid = be_ref[g]
        prev = be_ref[jnp.maximum(g - 1, 0)]
        offs = {k: (lax.rem(eid * rows_p[k], SUB) if rows_p[k] % SUB else None) for k in keys}
        if qtype == "IQ4_XS":
            tb = [jnp.broadcast_to(tbl_ref[0:1, :], (SUB, LANES))]
        else:
            tb = [jnp.broadcast_to(tbl_ref[r:r + 1, :], (SUB, LANES)) for r in range(n_tbl)]

        @pl.when((g == 0) | (eid != prev))
        def _decode():                                     # this expert's W^T tiles for the lane block, once
            for v in range(n_vreg):
                lanes = slice(v * LANES, (v + 1) * LANES)

                def rows_of(prefs, lanes=lanes):
                    def rows(k, r0, n):
                        if offs[k] is None:
                            return prefs[k][r0:r0 + n, lanes]
                        return _pick_rows(prefs[k][0:SUB, lanes], offs[k], r0, n)
                    return rows

                rg, ru = rows_of(g_refs), rows_of(u_refs)
                for b0 in range(0, W // SUB, nbg):
                    k0 = b0 * C * SUB
                    wg[k0:k0 + K, lanes] = _tiles(decode, rg, tb, b0, nbg, dtype)
                    wu[k0:k0 + K, lanes] = _tiles(decode, ru, tb, b0, nbg, dtype)

        @pl.when(g < nb_ref[0])
        def _rows():
            x = x_ref[...]                                                                  # (tm, n_in)
            o_ref[...] = M.swiglu_clamped(_dot(x, wg[...]), _dot(x, wu[...]), limit).astype(o_ref.dtype)

        @pl.when(g >= nb_ref[0])
        def _pad():
            o_ref[...] = jnp.zeros(o_ref.shape, o_ref.dtype)

    scratch = [pltpu.VMEM((n_in, LB), dtype)] * 2
    return _ragged_call(kernel, [planes_g, planes_u], keys, rows_p, blocks, tbl, xs, dtype, R, LB, tm, G, scratch,
                        interpret, vmem_mb, blk_expert, n_blocks)


def moe_ragged_down(planes, qtype, nblk, blk_expert, n_blocks, hs, *, tm=32, interpret=False, lane_block=None,
                    vmem_mb=48, out_dtype=None):
    """Grouped prefill GEMM, down. hs [Rp, n_in] (pm_mxu order, sorted rows as in `moe_ragged_gateup`) ->
    y [Rp, R] (out_dtype, default hs.dtype) = h @ down_e^T per row; padding blocks are zero."""
    dtype = hs.dtype
    out_dtype = out_dtype or dtype
    keys = PL.PLANE_KEYS[qtype]
    rows_p = PL.plane_rows(qtype, nblk)
    W = PL.WORDS_PER_BLOCK[qtype] * nblk
    C = PL.PLANES_PER_WORD[qtype]
    Rp, n_in = hs.shape
    assert n_in == nblk * PL.QK and n_in == W * C and Rp % tm == 0, (n_in, nblk, W, C, Rp, tm)
    G = Rp // tm
    assert blk_expert.shape == (G,), (blk_expert.shape, G)
    R = planes[keys[0]].shape[-1]
    LB = lane_block or min(R, 1024)
    assert R % LB == 0
    n_vreg = LB // LANES
    nbg = _k_group(qtype)
    K = nbg * C * SUB
    tbl = _table_arrays(qtype)
    n_tbl = tbl.shape[0]
    blocks = {k: (rows_p[k] if rows_p[k] % SUB == 0 else SUB) for k in keys}
    decode = DECODE[qtype]
    nk = len(keys)

    def kernel(be_ref, nb_ref, *refs):
        prefs = dict(zip(keys, refs[:nk]))
        tbl_ref, h_ref, o_ref, wd = refs[nk:]
        g = pl.program_id(1)
        eid = be_ref[g]
        prev = be_ref[jnp.maximum(g - 1, 0)]
        offs = {k: (lax.rem(eid * rows_p[k], SUB) if rows_p[k] % SUB else None) for k in keys}
        if qtype == "IQ4_XS":
            tb = [jnp.broadcast_to(tbl_ref[0:1, :], (SUB, LANES))]
        else:
            tb = [jnp.broadcast_to(tbl_ref[r:r + 1, :], (SUB, LANES)) for r in range(n_tbl)]

        @pl.when((g == 0) | (eid != prev))
        def _decode():
            for v in range(n_vreg):
                lanes = slice(v * LANES, (v + 1) * LANES)

                def rows(k, r0, n, lanes=lanes):
                    if offs[k] is None:
                        return prefs[k][r0:r0 + n, lanes]
                    return _pick_rows(prefs[k][0:SUB, lanes], offs[k], r0, n)

                for b0 in range(0, W // SUB, nbg):
                    k0 = b0 * C * SUB
                    wd[k0:k0 + K, lanes] = _tiles(decode, rows, tb, b0, nbg, dtype)

        @pl.when(g < nb_ref[0])
        def _rows():
            o_ref[...] = _dot(h_ref[...], wd[...]).astype(o_ref.dtype)

        @pl.when(g >= nb_ref[0])
        def _pad():
            o_ref[...] = jnp.zeros(o_ref.shape, o_ref.dtype)

    scratch = [pltpu.VMEM((n_in, LB), dtype)]
    return _ragged_call(kernel, [planes], keys, rows_p, blocks, tbl, hs, out_dtype, R, LB, tm, G, scratch, interpret,
                        vmem_mb, blk_expert, n_blocks)
'''
FILES["glm53/planes.py"] = r'''"""Planar ("lane = matrix row") storage of codebook-quantized expert matrices for the resident engine.

The raw GGUF block layout (82/110/136 bytes per 256 weights) makes every field a byte slice at an odd offset, which
costs XLA one relayout copy + one "header parsing" fusion per field per step and is unusable from a Pallas kernel.
Here each block field becomes its own dense 32-bit plane, transposed so that the matrix ROW is the lane (minor) axis
and the 4-byte "word" of packed codes is the sublane axis:

    word w of a row = 4 consecutive qs bytes = 4 codebook groups (4w .. 4w+3); byte j of the word is group 4w+j and
    each group holds GROUP_W[qtype] weights (positions k). So one (8, 128) vreg of the qs plane covers 8 words x 128
    rows and the weight it describes at (byte j, position k) is input index INPUT(w, j, k) of those 128 rows.

Per expert the planes hold exactly the bytes of the GGUF blocks (no expansion) except `d`, which is stored as pairs
of f16 bit patterns in one u32 per two blocks. All planes of one table are E-merged: shape [E * rows_p, R] so that
every array is (8, 128)-tile dense whatever rows_p is (a [E, 1, 4096] array would pad 1 -> 8 sublanes = 8x waste).

Formats (W = words per row, C = value planes per word = bytes x positions, `input(w, c)`):
  IQ2_S : qs u32 [W, R]; sg u32 [W, R] (sign byte j of word w = group 4w+j, bit k = position k);
          qh u32 [W/4, R] (byte w%4 of u32 w//4: bits 2j..2j+1 = high index bits of group 4w+j);
          sc u32 [W/4, R] (byte w%4 of u32 w//4: nibble h = scale of groups 4w+2h, 4w+2h+1);
          d  u32 [ceil(nblk/2), R] (f16 bits of block b at half b%2 of u32 b//2; block b = w // 8).
          W = 8*nblk, C = 32, input = 32w + 8j + k.
  IQ3_S : qs u32 [W, R]; sg u32 [W/2, R] (half w%2 of u32 w//2: nibble j = group 4w+j, bit k = position k);
          qh u32 [W/8, R] (bits 4(w%8)+j of u32 w//8 = high bit of group 4w+j);
          sc u32 [W/16, R] (nibble (w//2)%8 of u32 w//16 = scale of words 2n, 2n+1); d as above, block b = w // 16.
          W = 16*nblk, C = 16, input = 16w + 4j + k.
  IQ4_XS: qs u32 [W, R] (byte j of word w: low nibble = position 0, high = position 1); sl u32 [nblk, R] (nibble i =
          low scale bits of sub-block i); sh u32 [nblk, R] (bits 2i..2i+1 = high scale bits); d as above.
          W = 32*nblk, C = 8 (c = 4*pos + j), input = 32*(w//4) + 16*pos + 4*(w%4) + j.

The XLA dequant returns each expert as [C*W, R] with input index c*W + w ("pm order"); `pm_x` permutes activations
to match: x_pm[c*W + w] = x[input(w, c)], and the per-word activation matrix used by the Pallas kernel is
X[w, c] = x[input(w, c)] (same data as x_pm reshaped [C, W] and transposed).
"""
import numpy as np
import jax
import jax.numpy as jnp

from glm53 import iqquant as Q

QK = Q.QK_K
U32 = np.dtype("<u4")

WORDS_PER_BLOCK = {"IQ2_S": 8, "IQ3_S": 16, "IQ4_XS": 32, "Q2_K": 16, "Q3_K": 16}
PLANES_PER_WORD = {"IQ2_S": 32, "IQ3_S": 16, "IQ4_XS": 8, "Q2_K": 16, "Q3_K": 16}   # C
PLANE_KEYS = {"IQ2_S": ("qs", "sg", "qh", "sc", "d"), "IQ3_S": ("qs", "sg", "qh", "sc", "d"),
              "IQ4_XS": ("qs", "sl", "sh", "d"), "Q2_K": ("qs", "sc", "d"), "Q3_K": ("qs", "hm", "sc", "d")}
# K-quants (the MTP layer's experts; no codebook): word w of a row = 4 qs bytes; block b = w // 16, half n = (w % 16)
# // 8, q = w % 8; byte j of the word is ggml byte l = 4 q + j of half n and holds 4 two-bit values (shift s):
#   input(w, c) = 256 b + 128 n + 32 s + 4 q + j with c = 4 s + j (C = 16).
#   Q2_K: sc u32 [4 nblk, R] (the block's 16 scale/min bytes: byte is = 8 n + 2 s + (q >= 4), low nibble scale,
#         high nibble min); d u32 [nblk, R] (f16 d in the low half, f16 dmin in the high half).
#   Q3_K: hm u32 [8 nblk, R] (u32 q of block b = hmask bytes 4 q .. 4 q + 3; bit 4 n + s of byte l = high bit);
#         sc u32 [4 nblk, R] (the 16 six-bit scales, pre-unpacked to bytes, same `is` indexing); d u32 [nblk, R].


def plane_rows(qtype, nblk):
    """rows_p per expert of every plane."""
    W = WORDS_PER_BLOCK[qtype] * nblk
    dd = (nblk + 1) // 2
    if qtype == "IQ2_S":
        return {"qs": W, "sg": W, "qh": W // 4, "sc": W // 4, "d": dd}
    if qtype == "IQ3_S":
        return {"qs": W, "sg": W // 2, "qh": W // 8, "sc": W // 16, "d": dd}
    if qtype == "IQ4_XS":
        return {"qs": W, "sl": nblk, "sh": nblk, "d": dd}
    if qtype == "Q2_K":
        return {"qs": W, "sc": 4 * nblk, "d": nblk}
    if qtype == "Q3_K":
        return {"qs": W, "hm": 8 * nblk, "sc": 4 * nblk, "d": nblk}
    raise ValueError(qtype)


def input_index(qtype, w, c):
    """Natural input index of value plane c of word w (numpy-vectorised over w, c)."""
    if qtype == "IQ2_S":
        return 32 * w + c                       # c = 8j + k
    if qtype == "IQ3_S":
        return 16 * w + c                       # c = 4j + k
    if qtype == "IQ4_XS":
        pos, j = c // 4, c % 4
        return 32 * (w // 4) + 16 * pos + 4 * (w % 4) + j
    if qtype in ("Q2_K", "Q3_K"):
        s_, j = c // 4, c % 4
        return 256 * (w // 16) + 128 * ((w % 16) // 8) + 32 * s_ + 4 * (w % 8) + j
    raise ValueError(qtype)


def pm_perm(qtype, n_inputs):
    """x_pm = x[perm]: perm[c*W + w] = input(w, c)."""
    W = n_inputs * WORDS_PER_BLOCK[qtype] // QK
    C = PLANES_PER_WORD[qtype]
    c, w = np.meshgrid(np.arange(C), np.arange(W), indexing="ij")
    return input_index(qtype, w, c).reshape(-1).astype(np.int32)


def pm_x(qtype, x):
    """x [..., n] natural -> X [..., W, C] with X[w, c] = x[input(w, c)] (reshape/transpose only, no gather)."""
    n = x.shape[-1]
    lead = x.shape[:-1]
    if qtype == "IQ2_S":
        return x.reshape(lead + (n // 32, 32))
    if qtype == "IQ3_S":
        return x.reshape(lead + (n // 16, 16))
    if qtype == "IQ4_XS":                                     # input = 32*sub + 16*pos + 4*wq + j -> [sub, pos, wq, j]
        y = x.reshape(lead + (n // 32, 2, 4, 4))
        y = jnp.swapaxes(y, -3, -2)                           # [sub, wq, pos, j]
        return y.reshape(lead + (n // 8, 8))                  # w = 4*sub + wq, c = 4*pos + j
    if qtype in ("Q2_K", "Q3_K"):                             # input = 256b + 128n + 32s + 4q + j -> [b, n, s, q, j]
        y = x.reshape(lead + (n // 256, 2, 4, 8, 4))
        y = jnp.swapaxes(y, -3, -2)                           # [b, n, q, s, j]
        return y.reshape(lead + (n // 16, 16))                # w = 16b + 8n + q, c = 4s + j
    raise ValueError(qtype)


def pm_flat(qtype, x):
    """x [..., n] natural -> x_pm [..., n] with x_pm[c*W + w] = x[input(w, c)]."""
    X = pm_x(qtype, x)
    return jnp.swapaxes(X, -1, -2).reshape(x.shape)


# ------------------------------------------------------------------------------------------------ numpy packing
def _u32(a):
    """[E, R, nblk, n] u8 -> [E, R, nblk*n/4] u32 (little-endian words, group order preserved)."""
    E, R = a.shape[:2]
    a = np.ascontiguousarray(a.reshape(E, R, -1))
    assert a.shape[-1] % 4 == 0, a.shape
    return a.view(U32)


def _merge(a):
    """[E, R, P] -> [E*P, R] (planes are stored expert-major, rows_p sublanes per expert, matrix rows on lanes)."""
    E, R, P = a.shape
    return np.ascontiguousarray(a.transpose(0, 2, 1)).reshape(E * P, R)


def pack_planes(t, qtype):
    """t uint8 [E, R, nblk, block_bytes] (one chip's slice of one expert table) -> {plane: u32 [E*rows_p, R]}."""
    E, R, nblk, bb = t.shape
    assert bb == Q.BLOCK_BYTES[qtype], (bb, qtype)
    f = lambda a, b: t[..., a:b]
    if qtype in ("Q2_K", "Q3_K"):
        if qtype == "Q2_K":
            out = {"qs": _u32(f(16, 80)), "sc": _u32(f(0, 16)), "d": _u32(f(80, 84))}
        else:
            sc16 = Q.q3k_scales(f(96, 108))                                              # [E, R, nblk, 16] u8
            d = np.concatenate([f(108, 110), np.zeros(t.shape[:3] + (2,), np.uint8)], -1)
            out = {"qs": _u32(f(32, 96)), "hm": _u32(f(0, 32)), "sc": _u32(sc16), "d": _u32(d)}
        rows = plane_rows(qtype, nblk)
        out = {k: _merge(out[k]) for k in PLANE_KEYS[qtype]}
        for k, a in out.items():
            assert a.shape == (E * rows[k], R), (k, a.shape, rows[k])
        return out
    d = t[..., 0:2].reshape(E, R, nblk * 2)
    if nblk % 2:
        d = np.concatenate([d, np.zeros((E, R, 2), np.uint8)], -1)
    out = {"d": d.copy().view(U32)}
    if qtype == "IQ2_S":
        out.update(qs=_u32(f(2, 34)), sg=_u32(f(34, 66)), qh=_u32(f(66, 74)), sc=_u32(f(74, 82)))
    elif qtype == "IQ3_S":
        out.update(qs=_u32(f(2, 66)), qh=_u32(f(66, 74)), sg=_u32(f(74, 106)), sc=_u32(f(106, 110)))
    elif qtype == "IQ4_XS":
        sh = np.concatenate([f(2, 4), np.zeros(t.shape[:3] + (2,), np.uint8)], -1)
        out.update(sh=_u32(sh), sl=_u32(f(4, 8)), qs=_u32(f(8, 136)))
    else:
        raise ValueError(qtype)
    rows = plane_rows(qtype, nblk)
    out = {k: _merge(out[k]) for k in PLANE_KEYS[qtype]}
    for k, a in out.items():
        assert a.shape == (E * rows[k], R), (k, a.shape, rows[k])
    return out


def planes_bytes(planes):
    return sum(int(a.nbytes) if hasattr(a, "nbytes") else int(np.prod(a.shape)) * 4 for a in planes.values())


# ------------------------------------------------------------------------------------------------ XLA dequant
def _f16_pairs(d, nblk):
    """d u32 [E, ceil(nblk/2), R] -> f32 [E, nblk, R] block scales."""
    E, _, R = d.shape
    halves = jnp.stack([d & 0xFFFF, d >> 16], axis=2).reshape(E, -1, R)[:, :nblk]        # [E, nblk, R] u32
    return jax.lax.bitcast_convert_type(halves.astype(jnp.uint16), jnp.float16).astype(jnp.float32)


def _bytes(a, n=4, bits=8):
    """u32 [E, P, R] -> [E, P*n, R] sub-fields of `bits` bits, field m of word p at index p*n + m."""
    E, P, R = a.shape
    mask = (1 << bits) - 1
    return jnp.stack([(a >> (bits * m)) & mask for m in range(n)], axis=2).reshape(E, P * n, R)


def _sgn(bits, k):
    return 1.0 - 2.0 * ((bits >> k) & 1).astype(jnp.float32)


def dequant_planes(planes, qtype, nblk):
    """planes {k: u32 [E, rows_p, R]} -> f32 [E, C*W, R] (pm order along axis 1)."""
    if qtype == "IQ2_S":
        return _deq_iq2_s(planes, nblk)
    if qtype == "IQ3_S":
        return _deq_iq3_s(planes, nblk)
    if qtype == "IQ4_XS":
        return _deq_iq4_xs(planes, nblk)
    if qtype in ("Q2_K", "Q3_K"):
        return _deq_kq(planes, qtype, nblk)
    raise ValueError(qtype)


def _f16_lo(u):
    return jax.lax.bitcast_convert_type((u & 0xFFFF).astype(jnp.uint16), jnp.float16).astype(jnp.float32)


def _kq_scale_index(W):
    """Scale byte of word w (into the block-major 16-byte scale lists): is = 8 n + 2 s + (q >= 4), block-offset 16 b."""
    w = np.arange(W)
    base = 16 * (w // 16) + 8 * ((w % 16) // 8) + ((w % 8) >= 4)
    return [base + 2 * s_ for s_ in range(4)]                                       # per shift s


def _deq_kq(p, qtype, nblk):
    qs = p["qs"].astype(jnp.int32)                                              # [E, W, R], W = 16 nblk
    E, W, R = qs.shape
    scb = _bytes(p["sc"]).astype(jnp.int32)                                     # [E, 16 nblk, R] scale bytes (`is` order)
    idx = _kq_scale_index(W)
    d = jnp.repeat(_f16_lo(p["d"]), 16, axis=1)                                 # [E, W, R]
    if qtype == "Q2_K":
        dmin = jnp.repeat(_f16_lo(p["d"] >> 16), 16, axis=1)
    else:
        hmb = _bytes(p["hm"]).astype(jnp.int32)                                 # [E, 32 nblk, R] hmask byte l of block b
        w = np.arange(W)
        hidx = [32 * (w // 16) + 4 * (w % 8) + j for j in range(4)]              # byte l = 4 q + j
        nhalf = (w % 16) // 8
    outs = []
    for c in range(16):
        s_, j = c // 4, c % 4
        v = ((qs >> (8 * j + 2 * s_)) & 3).astype(jnp.float32)
        sc = jnp.take(scb, jnp.asarray(idx[s_]), axis=1)                        # [E, W, R]
        if qtype == "Q2_K":
            outs.append(d * (sc & 15).astype(jnp.float32) * v - dmin * (sc >> 4).astype(jnp.float32))
        else:
            hb = (jnp.take(hmb, jnp.asarray(hidx[j]), axis=1) >> jnp.asarray(4 * nhalf + s_)[None, :, None]) & 1
            outs.append(d * (sc - 32).astype(jnp.float32) * (v - 4.0 * (1 - hb).astype(jnp.float32)))
    return jnp.stack(outs, axis=1).reshape(E, 16 * W, R)


def _deq_iq2_s(p, nblk):
    qs, sg = p["qs"].astype(jnp.int32), p["sg"].astype(jnp.int32)               # [E, W, R]
    E, W, R = qs.shape
    qhb = _bytes(p["qh"]).astype(jnp.int32)                                     # [E, W, R] qh byte of word w
    scb = _bytes(p["sc"]).astype(jnp.int32)                                     # [E, W, R]
    d = jnp.repeat(_f16_pairs(p["d"], nblk), 8, axis=1)                         # [E, W, R] block = w // 8
    scale = [d * (0.5 + ((scb >> (4 * h)) & 15).astype(jnp.float32)) * 0.25 for h in range(2)]
    outs = []
    for j in range(4):
        idx = ((qs >> (8 * j)) & 255) | (((qhb >> (2 * j)) & 3) << 8)
        sgn = (sg >> (8 * j)) & 255
        vals = Q._tree_pos("IQ2_S", idx)                                        # 8 int8 [E, W, R]
        outs += [scale[j // 2] * v.astype(jnp.float32) * _sgn(sgn, k) for k, v in enumerate(vals)]
    return jnp.stack(outs, axis=1).reshape(E, 32 * W, R)


def _deq_iq3_s(p, nblk):
    qs = p["qs"].astype(jnp.int32)                                              # [E, W, R]
    E, W, R = qs.shape
    sgh = _bytes(p["sg"], 2, 16).astype(jnp.int32)                              # [E, W, R] 16 sign bits of word w
    qhn = _bytes(p["qh"], 8, 4).astype(jnp.int32)                               # [E, W, R] 4 high bits of word w
    scn = jnp.repeat(_bytes(p["sc"], 8, 4), 2, axis=1).astype(jnp.float32)      # [E, W, R] scale nibble of word w
    d = jnp.repeat(_f16_pairs(p["d"], nblk), 16, axis=1)                        # [E, W, R] block = w // 16
    scale = d * (1.0 + 2.0 * scn)
    outs = []
    for j in range(4):
        idx = ((qs >> (8 * j)) & 255) | (((qhn >> j) & 1) << 8)
        sgn = (sgh >> (4 * j)) & 15
        vals = Q._tree_pos("IQ3_S", idx)                                        # 4 int8 [E, W, R]
        outs += [scale * v.astype(jnp.float32) * _sgn(sgn, k) for k, v in enumerate(vals)]
    return jnp.stack(outs, axis=1).reshape(E, 16 * W, R)


def _deq_iq4_xs(p, nblk):
    qs = p["qs"].astype(jnp.int32)                                              # [E, W, R], W = 32*nblk
    E, W, R = qs.shape
    sl = _bytes(p["sl"], 8, 4).astype(jnp.int32)                                # [E, 8*nblk, R] sub-block i
    sh = _bytes(p["sh"], 8, 2).astype(jnp.int32)                                # [E, 8*nblk, R]
    d = jnp.repeat(_f16_pairs(p["d"], nblk), 8, axis=1)                         # [E, 8*nblk, R]
    scale = jnp.repeat(d * ((sl | (sh << 4)) - 32).astype(jnp.float32), 4, axis=1)   # [E, W, R] sub-block = w // 4
    outs = []
    for pos in range(2):
        for j in range(4):
            nib = ((qs >> (8 * j + 4 * pos)) & 15)
            outs.append(scale * Q._kv16(nib))
    return jnp.stack(outs, axis=1).reshape(E, 8 * W, R)


def dequant_natural(planes, qtype, nblk):
    """Test helper: [E, C*W, R] pm order -> [E, R, n_inputs] natural order."""
    v = dequant_planes(planes, qtype, nblk)
    E, _, R = v.shape
    n = nblk * QK
    inv = np.argsort(pm_perm(qtype, n))
    return jnp.take(jnp.swapaxes(v, 1, 2), jnp.asarray(inv), axis=-1)
'''
FILES["glm53/quant8.py"] = r'''"""int8 storage for the big NON-expert matrices (attention projections, shared experts, dense MLPs): per-channel
absmax scales, dequantized to bf16 at the top of each layer program (XLA fuses the convert into the matmul operand).
Frees ~1.1 GB of HBM per chip on v5e-8 (18 GB bf16 -> 9 GB). Routers stay fp32; norms/biases/1-D/small arrays untouched.

A quantized leaf is the dict {"q": int8 [in, out], "s": scale} with the scale laid out so it shards like the weight:
weights sharded on axis 0 get per-row scales [in, 1]; everything else per-column scales [1, out]."""
import numpy as np
import jax
import jax.numpy as jnp
from jax.sharding import PartitionSpec as P

MIN_SIZE = 1 << 20
SKIP_KEYS = {"router_w", "router_bias", "fn", "base", "scale", "gate_up", "down_", "embed", "lm_head"}
Q8_KEYS = frozenset({"q", "s"})


def is_q8(x):
    return isinstance(x, dict) and set(x.keys()) == Q8_KEYS


def eligible(key, a, spec):
    return (key not in SKIP_KEYS and getattr(a, "ndim", 0) == 2 and a.size >= MIN_SIZE
            and isinstance(spec, P) and a.dtype in (jnp.bfloat16, jnp.float32, np.dtype("float32"), np.dtype("bfloat16")))


def quantize_array(w, axis):
    """w [in, out] -> {"q": int8, "s": fp32}; axis 0 = per-row scales [in,1], axis 1 = per-column [1,out]."""
    w32 = jnp.asarray(w).astype(jnp.float32)
    amax = jnp.max(jnp.abs(w32), axis=1 - axis, keepdims=True)
    s = jnp.maximum(amax, 1e-12) / 127.0
    q = jnp.clip(jnp.round(w32 / s), -127, 127).astype(jnp.int8)
    return {"q": q, "s": s}


def quantize_array_host(w, axis, rows=8192):
    """`quantize_array` in NumPy on the host, chunked over rows: for the 155k x 4096 embedding / lm_head (a jnp
    version materialises a 2.5 GB f32 copy on chip 0 and OOMs a full HBM)."""
    w = np.asarray(w)
    n = w.shape[0]
    if axis == 1:                                            # per-column scales need the whole column: two passes
        amax = np.zeros((1, w.shape[1]), np.float32)
        for a in range(0, n, rows):
            amax = np.maximum(amax, np.abs(w[a:a + rows].astype(np.float32)).max(axis=0, keepdims=True))
        s = np.maximum(amax, 1e-12) / 127.0
        q = np.empty(w.shape, np.int8)
        for a in range(0, n, rows):
            q[a:a + rows] = np.clip(np.round(w[a:a + rows].astype(np.float32) / s), -127, 127).astype(np.int8)
        return {"q": q, "s": s}
    q = np.empty(w.shape, np.int8)
    s = np.empty((n, 1), np.float32)
    for a in range(0, n, rows):
        w32 = w[a:a + rows].astype(np.float32)
        sa = np.maximum(np.abs(w32).max(axis=1, keepdims=True), 1e-12) / 127.0
        s[a:a + rows] = sa
        q[a:a + rows] = np.clip(np.round(w32 / sa), -127, 127).astype(np.int8)
    return {"q": q, "s": s}


def scale_axis(spec):
    return 0 if (len(spec) >= 1 and spec[0] is not None) else 1


def quantize_layer(p, spec, fn=quantize_array):
    """Recursively quantize eligible leaves of one layer's params; returns (params, specs) with matching structure."""
    if isinstance(p, dict):
        out_p, out_s = {}, {}
        for k, v in p.items():
            sp = spec[k] if isinstance(spec, dict) else spec
            if isinstance(v, dict):
                out_p[k], out_s[k] = quantize_layer(v, sp, fn)
            elif eligible(k, v, sp):
                out_p[k] = fn(v, scale_axis(sp))
                out_s[k] = {"q": sp, "s": sp}
            else:
                out_p[k], out_s[k] = v, sp
        return out_p, out_s
    return p, spec


def expand_specs(p, spec):
    """Specs for params that are already quantized (q8 dict nodes where the spec is a single PartitionSpec)."""
    if is_q8(p) and isinstance(spec, P):
        return {"q": spec, "s": spec}
    if isinstance(p, dict) and isinstance(spec, dict):
        return {k: expand_specs(p[k], spec[k]) for k in p}
    if isinstance(p, list) and isinstance(spec, list):
        return [expand_specs(a, b) for a, b in zip(p, spec)]
    return spec


def dequant_tree(p, dtype=jnp.bfloat16):
    """Replace q8 nodes with dequantized arrays (inside a jitted program)."""
    if is_q8(p):
        return (p["q"].astype(jnp.float32) * p["s"]).astype(dtype)
    if isinstance(p, dict):
        return {k: dequant_tree(v, dtype) for k, v in p.items()}
    if isinstance(p, list):
        return [dequant_tree(v, dtype) for v in p]
    return p


def count_bytes(p):
    if is_q8(p):
        return p["q"].size + p["s"].size * 4
    if isinstance(p, dict):
        return sum(count_bytes(v) for v in p.values())
    if isinstance(p, list):
        return sum(count_bytes(v) for v in p)
    return getattr(p, "nbytes", 0)
'''
FILES["glm53/resident.py"] = r'''"""HBM-resident codebook-quantized experts (Unsloth UD-IQ*/Q2_K_XL GGUF) for the fully-jitted `Engine`.

Storage per sparse layer (chip-major, `P(AXIS)` on axis 0 hands each chip its slice; TP over moe_inter): every table
(gate_q, up_q, down_q) is a dict of PLANAR arrays `{plane: u32 [n_dev, E * rows_p, R]}` (glm53.planes): the GGUF block
fields split into dense 32-bit planes with the matrix row on the lane axis, so a Pallas kernel can dequantize an
expert in VMEM without byte slicing and XLA needs no relayout copies. gate/up: R = ml rows (this chip's slice of
moe_inter), inputs D; down: R = D rows, inputs ml.

Tables live in `params["layers"][i]["mlp"]`; `M.moe_dense` calls `ResidentFetch.apply`:
  decode (N*k <= gather_max_rows): `glm53.pallas_moe.moe_matvec` per matrix (fused dequant + VPU matvec, the expert
  planes DMA'd by index), or the XLA path (dynamic slices + glm53.planes.dequant_planes + einsum) when use_pallas=False;
  prefill: sweep over all experts in chunks (XLA dequant + dense masked matmuls) so no per-token weight copies exist.
Activations are permuted to the planes' "pm" input order (planes.pm_x / pm_flat); outputs are in natural row order.
"""
import numpy as np
import jax
import jax.numpy as jnp
from jax import lax

from glm53 import iqquant as Q
from glm53 import model as M
from glm53 import planes as PL
from glm53 import pallas_moe as K

QK = Q.QK_K
KEYS = ("gate_q", "up_q", "down_q")


def pack_layer_from_gguf(gm, layer: int, n_dev: int, threads: int = 8):
    """Read one layer's routed experts from a `gguf_reader.GGUFModel` -> (tables dict, qtypes dict)."""
    out, qtypes = {}, {}
    for key, tname in (("gate_q", "gate"), ("up_q", "up"), ("down_q", "down")):
        name = f"blk.{layer}.ffn_{tname}_exps.weight"
        info = gm.info(name)
        ne0, ne1, E = info["dims"]                       # ne0 = input dim (blocks along it), ne1 = rows, E experts
        bb = Q.BLOCK_BYTES[info["type"]]
        raws = gm.read_experts(name, range(E), threads=threads)
        t = np.stack([np.frombuffer(r, np.uint8).reshape(ne1, ne0 // QK, bb) for r in raws])   # [E, rows, nblk, bb]
        if key == "down_q":                              # split blocks (the mi input dim) across chips
            t = t.reshape(E, ne1, n_dev, ne0 // QK // n_dev, bb).transpose(2, 0, 1, 3, 4)
        else:                                            # split rows (the mi output dim) across chips
            t = t.reshape(E, n_dev, ne1 // n_dev, ne0 // QK, bb).transpose(1, 0, 2, 3, 4)
        out[key] = pack_chip_planes(t, info["type"])
        qtypes[key] = info["type"]
    return out, qtypes


def pack_chip_planes(t, qtype, threads=None):
    """t uint8 [n_dev, E, rows, nblk, bb] -> {plane: u32 [n_dev, E*rows_p, rows]} (glm53.planes layout per chip).
    The chips' slices are packed in parallel threads (NumPy releases the GIL on the big slices; the packing, not the
    GGUF read, dominated the 9-minute build: the dataset mount serves ~650 MB/s to 8-16 readers, 2026-09-14)."""
    n = t.shape[0]
    threads = n if threads is None else max(1, int(threads))
    if threads > 1 and n > 1:
        from concurrent.futures import ThreadPoolExecutor
        with ThreadPoolExecutor(min(threads, n)) as ex:
            per_chip = list(ex.map(lambda c: PL.pack_planes(t[c], qtype), range(n)))
    else:
        per_chip = [PL.pack_planes(t[c], qtype) for c in range(n)]
    return {k: np.stack([pc[k] for pc in per_chip]) for k in per_chip[0]}


def hbm_bytes(tables: dict) -> int:
    """Bytes per chip of a layer's tables ({key: {plane: [n_dev, ...]}})."""
    return sum(int(a.nbytes) // int(a.shape[0]) for planes in tables.values() for a in planes.values())


class ResidentFetch:
    """Stored on the Engine; `M.moe_dense` calls `.apply(p, x, idx, w, layer, limit)` when present."""

    def __init__(self, qtypes: dict, D: int, ml: int, out_dtype=jnp.bfloat16, gather_max_rows: int = 64, chunk: int = 8,
                 use_pallas: bool = True, interpret: bool = False, sweep_mode: str = "ragged", tm: int = 32,
                 combine: str = "matmul"):
        self.qtypes = qtypes            # {layer: {"gate_q": "IQ2_S", "up_q": "IQ2_S", "down_q": "IQ3_S"}}
        self.rows = {"gate_q": ml, "up_q": ml, "down_q": D}     # logical rows per expert (this chip)
        self.nblk = {"gate_q": D // QK, "up_q": D // QK, "down_q": ml // QK}
        self.out_dtype = out_dtype
        self.gather_max_rows = gather_max_rows
        self.chunk = chunk
        self.use_pallas = use_pallas    # decode path: Pallas fused kernel (True) or XLA planes dequant + einsum
        self.interpret = interpret      # run the Pallas kernel in interpret mode (CPU tests)
        self.sweep_mode = sweep_mode    # prefill: "dense" = masked sweep over active experts, "ragged" = grouped GEMM
        self.tm = tm                    # ragged: rows per block (multiple of 16 for bf16)
        self.combine = combine          # ragged: per-token combine of slot outputs, "gather" (row gather) or "matmul"

    def _deq(self, layer, key, tbl, sel):
        """Dequantize the experts picked by `sel` from planes `tbl` -> [n, C*W, R] (pm order along inputs) in out_dtype."""
        qt = self.qtypes[layer][key]
        rows_p = PL.plane_rows(qt, self.nblk[key])
        picked = {k: sel(a, rows_p[k]).reshape(-1, rows_p[k], a.shape[-1]) for k, a in tbl.items()}
        return PL.dequant_planes(picked, qt, self.nblk[key]).astype(self.out_dtype)

    @staticmethod
    def _slice_experts(a, rows_p, flat):
        """a u32 [1, E*rows_p, R]; flat int32 [n] -> [n*rows_p, R] via n dynamic slices (DMA, never a generic gather
        and never a copy of the whole table)."""
        parts = [lax.dynamic_slice_in_dim(a, flat[j] * rows_p, rows_p, axis=1) for j in range(flat.shape[0])]
        return jnp.concatenate(parts, axis=1)[0]

    def apply(self, p, x, idx, w, layer, limit):
        """x [N,D]; idx [N,k] int32 into E; w [N,k] fp32 -> partial MoE output [N,D] (TP-reduced by the caller)."""
        gate_q, up_q, down_q = p["gate_q"], p["up_q"], p["down_q"]         # {plane: [1, E*rows_p, R]} on this chip
        N, k = idx.shape
        if N * k <= self.gather_max_rows:
            if self.use_pallas:
                return self._apply_kernel(x, idx, w, layer, gate_q, up_q, down_q, limit)
            return self._apply_gather(x, idx, w, layer, gate_q, up_q, down_q, limit)
        return self._apply_sweep(x, idx, w, layer, gate_q, up_q, down_q, limit)

    def _apply_kernel(self, x, idx, w, layer, gate_q, up_q, down_q, limit):
        N, k = idx.shape
        flat = idx.reshape(-1)
        qt_gu, qt_dn = self.qtypes[layer]["gate_q"], self.qtypes[layer]["down_q"]
        Xg = jnp.repeat(PL.pm_x(qt_gu, x.astype(jnp.float32)), k, axis=0)              # [Nk, W, C]
        mv = lambda planes, qt, key, X: K.moe_matvec(planes, qt, self.nblk[key], flat, X, interpret=self.interpret)
        g = mv(gate_q, qt_gu, "gate_q", Xg)                                             # [Nk, ml] f32
        u = mv(up_q, self.qtypes[layer]["up_q"], "up_q", Xg)
        h = M.swiglu_clamped(g, u, limit)                                               # [Nk, ml]
        y = mv(down_q, qt_dn, "down_q", PL.pm_x(qt_dn, h)).reshape(N, k, -1)            # [N, k, D]
        return jnp.einsum("nkd,nk->nd", y, w)

    def _apply_gather(self, x, idx, w, layer, gate_q, up_q, down_q, limit):
        """XLA decode path on the planar tables (no Pallas): per-expert dynamic slices + dequant + einsum."""
        N, k = idx.shape
        flat = idx.reshape(-1)
        qt_gu, qt_dn = self.qtypes[layer]["gate_q"], self.qtypes[layer]["down_q"]
        sel = lambda a, rows_p: self._slice_experts(a, rows_p, flat)
        g = self._deq(layer, "gate_q", gate_q, sel)                                     # [Nk, D_pm, ml]
        u = self._deq(layer, "up_q", up_q, sel)
        d = self._deq(layer, "down_q", down_q, sel)                                     # [Nk, ml_pm, D]
        xs = jnp.repeat(PL.pm_flat(qt_gu, x).astype(self.out_dtype), k, axis=0)         # [Nk, D_pm]
        h = M.swiglu_clamped(jnp.einsum("nd,ndm->nm", xs, g), jnp.einsum("nd,ndm->nm", xs, u), limit)   # [Nk, ml]
        h = PL.pm_flat(qt_dn, h).astype(d.dtype)
        y = jnp.einsum("nm,nmd->nd", h, d).astype(jnp.float32).reshape(N, k, -1)
        return jnp.einsum("nkd,nk->nd", y, w)

    def _apply_sweep(self, x, idx, w, layer, gate_q, up_q, down_q, limit):
        if self.use_pallas and self.sweep_mode == "ragged":
            return self._apply_ragged_kernel(x, idx, w, layer, gate_q, up_q, down_q, limit)
        if self.use_pallas:
            return self._apply_sweep_kernel(x, idx, w, layer, gate_q, up_q, down_q, limit)
        return self._apply_sweep_xla(x, idx, w, layer, gate_q, up_q, down_q, limit)

    def _apply_ragged_kernel(self, x, idx, w, layer, gate_q, up_q, down_q, limit):
        """Prefill through the grouped GEMM kernels: (token, expert) slots sorted by expert into `tm`-row blocks,
        each expert dequantized once and multiplied against its own rows only; the per-token combine gathers each
        token's k slot rows (or, `combine="matmul"`, multiplies by a [T, Rp] weight matrix)."""
        qt_gu, qt_dn = self.qtypes[layer]["gate_q"], self.qtypes[layer]["down_q"]
        E = gate_q["qs"].shape[1] // PL.plane_rows(qt_gu, self.nblk["gate_q"])["qs"]
        T, k = idx.shape
        plan = K.ragged_plan(idx, w, E, self.tm)
        tok = plan["tok_row"]
        xk = K.pm_mxu(qt_gu, x.astype(jnp.float32)).astype(self.out_dtype)                     # [T, D]
        xs = jnp.take(xk, jnp.maximum(tok, 0), axis=0, mode="clip")
        xs = jnp.where((tok >= 0)[:, None], xs, jnp.zeros((), xs.dtype))                       # [Rp, D]
        h = K.moe_ragged_gateup(gate_q, up_q, qt_gu, self.nblk["gate_q"], plan["blk_expert"], plan["n_blocks"], xs,
                                limit, tm=self.tm, interpret=self.interpret)                     # [Rp, ml]
        hk = K.pm_mxu(qt_dn, h.astype(jnp.float32)).astype(self.out_dtype)
        y = K.moe_ragged_down(down_q, qt_dn, self.nblk["down_q"], plan["blk_expert"], plan["n_blocks"], hk,
                              tm=self.tm, interpret=self.interpret)                              # [Rp, D]
        if self.combine == "matmul":
            wc = (tok[None, :] == jnp.arange(T, dtype=jnp.int32)[:, None]).astype(jnp.float32) * plan["w_row"][None, :]
            return jnp.dot(wc.astype(y.dtype), y, preferred_element_type=jnp.float32)
        ys = jnp.take(y, plan["row_slot"].reshape(-1), axis=0, mode="clip").reshape(T, k, -1).astype(jnp.float32)
        return jnp.einsum("tkd,tk->td", ys, w.astype(jnp.float32))

    SWEEP_TOKENS = 512     # token chunk per kernel launch (bounds the [E, T, ml] intermediate to ~75 MB per chip)

    def _apply_sweep_kernel(self, x, idx, w, layer, gate_q, up_q, down_q, limit):
        """Prefill through the Pallas sweep kernels: every routed expert is dequantized once into VMEM tiles and MXU-
        multiplied against all tokens; experts no token routes to are skipped (scalar-prefetched active slots)."""
        qt_gu, qt_dn = self.qtypes[layer]["gate_q"], self.qtypes[layer]["down_q"]
        E = gate_q["qs"].shape[1] // PL.plane_rows(qt_gu, self.nblk["gate_q"])["qs"]
        N = x.shape[0]
        ids, n_act = K.active_slots(idx, E)
        rw = jnp.einsum("nk,nke->ne", w.astype(jnp.float32),
                        (idx[:, :, None] == ids[None, None, :]).astype(jnp.float32))          # [N, E] per slot
        outs = []
        for t0 in range(0, N, self.SWEEP_TOKENS):
            xs = x[t0:t0 + self.SWEEP_TOKENS]
            xk = K.pm_mxu(qt_gu, xs.astype(jnp.float32)).astype(self.out_dtype)                # [T, D]
            h = K.moe_sweep_gateup(gate_q, up_q, qt_gu, self.nblk["gate_q"], ids, n_act, xk, limit,
                                   interpret=self.interpret)                                    # [E, T, ml] bf16
            hw = h.astype(jnp.float32) * jnp.swapaxes(rw[t0:t0 + self.SWEEP_TOKENS], 0, 1)[:, :, None]
            hk = K.pm_mxu(qt_dn, hw).astype(self.out_dtype)                                     # [E, T, ml]
            outs.append(K.moe_sweep_down(down_q, qt_dn, self.nblk["down_q"], ids, n_act, hk, interpret=self.interpret))
        return outs[0] if len(outs) == 1 else jnp.concatenate(outs, axis=0)

    def _apply_sweep_xla(self, x, idx, w, layer, gate_q, up_q, down_q, limit):
        E = gate_q["qs"].shape[1] // PL.plane_rows(self.qtypes[layer]["gate_q"], self.nblk["gate_q"])["qs"]
        CH = self.chunk
        assert E % CH == 0, (E, CH)
        qt_gu, qt_dn = self.qtypes[layer]["gate_q"], self.qtypes[layer]["down_q"]
        onehot = jax.nn.one_hot(idx, E, dtype=jnp.float32)                  # [N,k,E]
        rw = jnp.einsum("nk,nke->ne", w.astype(jnp.float32), onehot)        # [N,E] routing weight (0 if unused)
        xg = PL.pm_flat(qt_gu, x).astype(self.out_dtype)                    # [N, D_pm]

        def body(acc, c):
            c0 = c * CH
            sel = lambda a, rows_p: lax.dynamic_slice_in_dim(a, c0 * rows_p, CH * rows_p, axis=1)[0]
            g = self._deq(layer, "gate_q", gate_q, sel)                                          # [CH, D_pm, ml]
            u = self._deq(layer, "up_q", up_q, sel)
            d = self._deq(layer, "down_q", down_q, sel)                                          # [CH, ml_pm, D]
            h = M.swiglu_clamped(jnp.einsum("nd,edm->nem", xg, g), jnp.einsum("nd,edm->nem", xg, u), limit)  # [N,CH,ml]
            rws = lax.dynamic_slice_in_dim(rw, c0, CH, 1)                                        # [N, CH]
            h = PL.pm_flat(qt_dn, h.astype(jnp.float32) * rws[..., None]).astype(d.dtype)
            return acc + jnp.einsum("nem,emd->nd", h, d).astype(jnp.float32), None

        acc, _ = lax.scan(body, jnp.zeros((xg.shape[0], x.shape[1]), jnp.float32), jnp.arange(E // CH))
        return acc


# ------------------------------------------------------------------------------------- per-layer-kind engine
from glm53.engine import Engine, AXIS, R, shard_map, cache_specs, device_sample, DeviceSampler   # noqa: E402
from jax.sharding import NamedSharding, PartitionSpec as P           # noqa: E402


class _HostShards:
    """A sharded device array parked in host memory as its per-device shards (a pytree leaf)."""

    def __init__(self, shards, devices, shape, dtype, sharding):
        self.shards, self.devices, self.shape, self.dtype, self.sharding = shards, devices, shape, dtype, sharding
        self.nbytes = sum(int(x.nbytes) for x in shards)

    @classmethod
    def of(cls, a):
        sh = sorted(a.addressable_shards, key=lambda s: s.device.id)
        return cls([np.asarray(s.data) for s in sh], [s.device for s in sh], a.shape, a.dtype, a.sharding)

    def to_device(self):
        parts = [jax.device_put(x, d) for x, d in zip(self.shards, self.devices)]
        return jax.make_array_from_single_device_arrays(self.shape, self.sharding, parts)


class ResidentLayerEngine(Engine):
    """Resident experts with ONE jitted program per layer *kind* (attention type, MLP type, expert quant types,
    indexer presence) instead of one 45-layer program: the layer's params are program arguments, so ~6 programs cover
    all layers and the compile stays small (a single unrolled program with 42 codebook dequants got the XLA compiler
    SIGKILLed on the Kaggle host). Per step: embed program, 45 layer programs, head program — all device-resident,
    no host gathers.

    Prefill is CHUNKED: the caches are allocated up front (`alloc_caches`) and the prompt is fed in pieces of
    `prefill_piece` tokens (the last one padded to a bucket) through cache-carrying programs, so no temporary scales
    with the prompt length; `prefill(tokens, caches, pos0)` continues an existing context (prefix caching). Cache
    buffers are DONATED to the layer programs (in-place update; never reuse a cache object after passing it in —
    `copy_caches` for snapshots)."""

    def __init__(self, cfg, params, expert_fetch, devices=None, max_len: int = 4096, layers_per_program: int = 1,
                 layers_per_program_prefill: int = 1, int8_nonexpert: bool = False, seq_shard: bool = False,
                 q_block: int = 128, prefill_piece: int = 2048, cache_q8: bool = False):
        super().__init__(cfg, params, None, devices, max_len, expert_fetch=expert_fetch, int8_nonexpert=int8_nonexpert,
                         seq_shard=seq_shard, q_block=q_block, cache_q8=cache_q8)
        self._progs = {}
        self._allocs = {}
        self.launches = 0
        assert prefill_piece in self.PREFILL_BUCKETS, prefill_piece
        self.prefill_piece = min(prefill_piece, max_len)
        # consecutive layer groups; a group's program is keyed by the tuple of its layer kinds (dispatch ≈ 2 ms per
        # launch on the Kaggle TPU VM, so 45 single-layer launches cost ~100 ms/token). Prefill keeps 1 layer per
        # program by default: a 4-layer prefill program (expert sweep scans inside) did not finish compiling in 50 min.
        self.lpp = {"decode": layers_per_program, "prefill": layers_per_program_prefill}
        self.groups = {m: [list(range(a, min(a + k, cfg.n_layers))) for a in range(0, cfg.n_layers, k)]
                       for m, k in self.lpp.items()}

    def _kind(self, i):
        qt = tuple(sorted(self.expert_fetch.qtypes.get(i, {}).items())) if self.cfg.mlp_types[i] == "sparse" else ()
        return (self.cfg.layer_types[i], self.cfg.mlp_types[i], qt, "indexer" in self.params["layers"][i]["attn"])

    def _prog_embed(self, B, T, override=False):
        """Token embeddings broadcast to the hc residual streams [B,T,hc,D]; with override=True the extra arguments
        (vectors [B,T,D], mask [B,T] bool) replace the table rows where mask is set (image tokens: glm53.vision)."""
        key = ("embed", B, T, override)
        if key not in self._progs:
            def prog(embed, tokens, *ov):
                x = self._embed(embed, tokens).astype(self.lcfg.dtype)
                if override:
                    vec, mask = ov
                    x = jnp.where(mask[..., None], vec.astype(x.dtype), x)
                return jnp.broadcast_to(x[:, :, None, :], (B, T, self.cfg.hc, x.shape[-1]))
            in_specs = (self.specs["embed"], R) + ((R, R) if override else ())
            self._progs[key] = jax.jit(shard_map(prog, mesh=self.mesh, in_specs=in_specs, out_specs=R, check_vma=False))
        return self._progs[key]

    def _prog_head(self, B, T, sample=False):
        """Logits [B,V] of the last valid position; with sample=True also the device-sampled token ids [B]
        (extra args: temperature, top_p, key — `DeviceSampler.bind(mesh)`), returned as (ids, logits, next key)."""
        key = ("head", B, T, sample)
        if key not in self._progs:
            def prog(norm, lm_head, streams, length, *samp):
                h = M.rmsnorm(streams.mean(2).astype(self.lcfg.dtype), norm, self.cfg.eps)
                last = lax.dynamic_index_in_dim(h, length - 1, axis=1, keepdims=False)
                zl = self._logits_local(lm_head, last)
                logits = lax.all_gather(zl, AXIS, axis=-1, tiled=True)
                if not sample:
                    return logits
                temp, top_p, k = samp
                k1, k2 = jax.random.split(k)
                return device_sample(zl, temp, top_p, k1), logits, k2
            in_specs = (R, self.specs["lm_head"], R, R) + ((R, R, R) if sample else ())
            self._progs[key] = jax.jit(shard_map(prog, mesh=self.mesh, in_specs=in_specs,
                                                 out_specs=(R, R, R) if sample else R, check_vma=False))
        return self._progs[key]

    def _hist_spec(self, i):
        """Sharding of the per-token histories a speculative verify step returns for layer i (None: no history)."""
        if self.cfg.layer_types[i] == "linear_attention":
            return {"state_hist": P(None, None, AXIS), "conv_hist": P(None, None, None, AXIS)}
        if "indexer" in self.params["layers"][i]["attn"]:
            return {"tk_hist": R, "tg_hist": R}
        return None

    def _prog_group(self, layers, B, T, has_cache, use_recurrent, hist=False):
        kinds = tuple(self._kind(i) for i in layers)
        key = ("group", kinds, B, T, has_cache, use_recurrent, hist)
        if key not in self._progs:
            lts = [self.cfg.layer_types[i] for i in layers]
            mts = [self.cfg.mlp_types[i] for i in layers]
            css = [self._cache_spec(i) for i in layers]
            hss = [self._hist_spec(i) for i in layers]
            rep = list(layers)                     # representative layer ids (only used to look up expert qtypes)

            from glm53 import quant8 as Q8

            def prog(ps, streams, pos0, length, caches):
                new, hs = [], []
                for j, p in enumerate(ps):
                    p = Q8.dequant_tree(p, self.lcfg.dtype)          # int8 non-expert matrices -> bf16 (fused into dots)
                    fetch = self.expert_fetch if mts[j] == "sparse" else None
                    streams, nc = M.decoder_layer(p, streams, self.lcfg, lts[j], mts[j],
                                                  caches[j] if has_cache else None, pos0, use_recurrent, fetch, "auto",
                                                  layer=rep[j], cap=self.max_len, length=None if use_recurrent else length,
                                                  hist=hist)
                    nc, h = M.split_hist(nc)
                    new.append(nc); hs.append(h if hist else None)
                return (streams, new, hs) if hist else (streams, new)
            in_specs = ([self.specs["layers"][i] for i in layers], R, R, R, css if has_cache else None)
            out_specs = (R, css, hss) if hist else (R, css)
            sm = shard_map(prog, mesh=self.mesh, in_specs=in_specs, out_specs=out_specs, check_vma=False)
            self._progs[key] = jax.jit(sm, donate_argnums=(4,) if has_cache else ())
        return self._progs[key]

    def _prog_hidden(self, B, T):
        """Final-norm hidden states of every position [B,T,D] (MTP prefill input)."""
        key = ("hidden", B, T)
        if key not in self._progs:
            def prog(norm, streams):
                return M.rmsnorm(streams.mean(2).astype(self.lcfg.dtype), norm, self.cfg.eps)
            self._progs[key] = jax.jit(shard_map(prog, mesh=self.mesh, in_specs=(R, R), out_specs=R, check_vma=False))
        return self._progs[key]

    def _prog_head_all(self, B, T, sample=False):
        """Logits [B,T,V] and hidden [B,T,D] of every position (speculative verify; T small); with sample=True
        (extra args as in `_prog_head`) also the device-sampled ids [B,T]: (ids, logits, hidden, next key)."""
        key = ("head_all", B, T, sample)
        if key not in self._progs:
            def prog(norm, lm_head, streams, *samp):
                h = M.rmsnorm(streams.mean(2).astype(self.lcfg.dtype), norm, self.cfg.eps)
                zl = self._logits_local(lm_head, h)
                logits = lax.all_gather(zl, AXIS, axis=-1, tiled=True)
                if not sample:
                    return logits, h
                temp, top_p, k = samp
                k1, k2 = jax.random.split(k)
                ids = device_sample(zl.reshape(B * T, -1), temp, top_p, k1)
                return ids.reshape(B, T), logits, h, k2
            in_specs = (R, self.specs["lm_head"], R) + ((R, R, R) if sample else ())
            self._progs[key] = jax.jit(shard_map(prog, mesh=self.mesh, in_specs=in_specs,
                                                 out_specs=(R, R, R, R) if sample else (R, R), check_vma=False))
        return self._progs[key]

    def _prog_rollback(self, B, T):
        """per-token histories of a T-token verify + n_keep -> the small state entries (KDA state/conv, pool tails)
        as after the first n_keep tokens; spliced into the caches by the caller (the big positional arrays are
        not touched)."""
        key = ("rollback", B, T)
        if key not in self._progs:
            hss = [self._hist_spec(i) for i in range(self.cfg.n_layers)]
            oss = [None if h is None else {M.HIST_KEYS[k]: P(*v[1:]) for k, v in h.items()} for h in hss]   # drop the T axis
            dts = [{M.HIST_KEYS[k]: self.cfg.dtype if M.HIST_KEYS[k] != "state" else jnp.float32 for k in h} if h else None
                   for h in hss]
            sm = shard_map(lambda h, n: M.rollback_states(h, n, dts), mesh=self.mesh, in_specs=(hss, R), out_specs=oss,
                           check_vma=False)
            self._progs[key] = jax.jit(sm)
        return self._progs[key]

    mtp = None                       # MTP (NextN) layer params on device, or None (see set_mtp)

    def _cache_spec(self, i):
        if i == self.cfg.n_layers:                                             # the MTP layer's cache entry
            mla = P(None, AXIS) if self.lcfg.seq_shard > 1 else R
            spec = {"c": mla, "pk": mla, "tk": R, "tg": R, "h": R}
            if self.cfg.cache_q8:
                spec["cs"] = mla
            return spec
        return super()._cache_spec(i)

    def _is_kda(self, i):
        return i < self.cfg.n_layers and self.cfg.layer_types[i] == "linear_attention"

    def alloc_caches(self, B):
        """Zero caches for a new context at the engine's capacity, allocated directly on the chips (+ the MTP
        layer's entry, index n_layers, when an MTP layer is installed: its attention cache and the carried final
        hidden state "h" [B,1,D] of the last processed position)."""
        cfg = self.cfg
        out = []
        n_entries = cfg.n_layers + (1 if self.mtp is not None else 0)
        for i in range(n_entries):
            spec = self._cache_spec(i)
            if i == cfg.n_layers:
                shapes = {k: (sh, M.cache_dtype(cfg, k, cfg.dtype)) for k, sh in M.mla_cache_shapes(cfg, B, self.max_len, True).items()}
                shapes["h"] = ((B, 1, cfg.hidden), cfg.dtype)
            elif cfg.layer_types[i] == "linear_attention":
                shapes = {"conv": ((B, cfg.conv_k - 1, 3 * cfg.kda_heads * cfg.kda_hd), cfg.dtype),
                          "state": ((B, cfg.kda_heads, cfg.kda_hd, cfg.kda_hd), jnp.float32)}
            else:
                has_ix = "indexer" in self.params["layers"][i]["attn"]
                shapes = {k: (sh, M.cache_dtype(cfg, k, cfg.dtype)) for k, sh in M.mla_cache_shapes(cfg, B, self.max_len, has_ix).items()}
            c = {}
            for k, (shape, dtype) in shapes.items():
                key = (shape, jnp.dtype(dtype), spec[k])
                if key not in self._allocs:
                    self._allocs[key] = jax.jit(lambda shape=shape, dtype=dtype: jnp.zeros(shape, dtype),
                                                out_shardings=NamedSharding(self.mesh, spec[k]))
                c[k] = self._allocs[key]()
            out.append(c)
        return out

    @staticmethod
    def copy_caches(caches):
        """Device-side copy (the programs update caches in place)."""
        return jax.tree.map(jnp.copy, caches)

    def _prefix_rows(self, n_tokens):
        """Local cache rows that positions < n_tokens can occupy (interleaved layout: 4-token pools round-robin)."""
        kp, n = self.cfg.idx_kpool, self.lcfg.seq_shard
        return -(-n_tokens // (kp * n)) * kp if n > 1 else n_tokens

    def snapshot_prefix(self, caches, n_tokens, rows_bucket=1):
        """Compact snapshot of a context of n_tokens (the KDA states and only the used MLA/indexer cache rows), e.g.
        a shared system prompt: ~2 KB/token/chip instead of a full cache set. Restore with `restore_prefix`.
        `rows_bucket` rounds the row count up (bounds the number of slice/restore programs compiled for arbitrary
        context lengths; the extra rows are never read back)."""
        rows = self._prefix_rows(n_tokens)
        cap = self.max_len // max(self.lcfg.seq_shard, 1)
        rows = min(-(-rows // rows_bucket) * rows_bucket, cap)
        key = ("snap", rows)
        if key not in self._progs:
            self._progs[key] = {}
        out = []
        for i, c in enumerate(caches):
            if self._is_kda(i):
                out.append(jax.tree.map(jnp.copy, c)); continue
            spec = self._cache_spec(i)
            snap = {}
            for k, a in c.items():
                if k not in M.ROW_KEYS:                                       # tail buffers: whole copies
                    snap[k] = jnp.copy(a); continue
                rk = rows // self.cfg.idx_kpool if k == "pk" else rows
                pk = (k, a.shape, str(a.dtype))
                if pk not in self._progs[key]:
                    sp = spec[k]
                    self._progs[key][pk] = jax.jit(shard_map(lambda x, rk=rk: x[:, :rk], mesh=self.mesh, in_specs=(sp,),
                                                             out_specs=sp, check_vma=False))
                snap[k] = self._progs[key][pk](a)
            out.append(snap)
        return {"n_tokens": n_tokens, "rows": rows, "caches": out}

    def restore_prefix(self, snap, B=1):
        """Fresh caches at full capacity holding the snapshot's context (the snapshot stays valid)."""
        caches = self.alloc_caches(B)
        rows = snap.get("rows") or self._prefix_rows(snap["n_tokens"])
        key = ("restore", rows)
        if key not in self._progs:
            self._progs[key] = {}
        out = []
        for i, c in enumerate(caches):
            sc = snap["caches"][i]
            if self._is_kda(i):
                out.append(jax.tree.map(jnp.copy, sc)); continue
            spec = self._cache_spec(i)
            new = {}
            for k, a in c.items():
                if k not in M.ROW_KEYS:
                    new[k] = jnp.copy(sc[k]); continue
                pk = (k, a.shape, str(a.dtype))
                if pk not in self._progs[key]:
                    sp = spec[k]
                    self._progs[key][pk] = jax.jit(shard_map(lambda x, p: lax.dynamic_update_slice_in_dim(x, p, 0, axis=1),
                                                             mesh=self.mesh, in_specs=(sp, sp), out_specs=sp, check_vma=False),
                                                   donate_argnums=(0,))
                new[k] = self._progs[key][pk](a, sc[k])
            out.append(new)
        return out

    @staticmethod
    def snapshot_to_host(snap):
        """Move a compact snapshot into host memory so it holds no HBM; a parked context costs ~17 KB/token on the
        host. Every array is kept as its per-device shards (no host-side assembly of the 8 shards, which ran at
        ~1.5 GB/s); `snapshot_from_host` brings it back for `restore_prefix`."""
        host = jax.tree.map(_HostShards.of, snap["caches"])
        n_bytes = sum(h.nbytes for h in jax.tree.leaves(host, is_leaf=lambda x: isinstance(x, _HostShards)))
        return {"n_tokens": snap["n_tokens"], "rows": snap["rows"], "caches": host, "bytes": n_bytes}

    @staticmethod
    def snapshot_from_host(hsnap):
        """Device snapshot (as `snapshot_prefix` returns it) from a host snapshot; the host copy stays valid."""
        caches = jax.tree.map(lambda h: h.to_device(), hsnap["caches"], is_leaf=lambda x: isinstance(x, _HostShards))
        return {"n_tokens": hsnap["n_tokens"], "rows": hsnap["rows"], "caches": caches}

    def _run_layers(self, tokens, caches, pos0, use_recurrent, length, override=None):
        """embed + all layer groups -> (final residual streams, new caches). `override` = (vectors [B,T,D],
        mask [B,T]) replaces the token embeddings where mask is set."""
        B, T = tokens.shape
        if override is None:
            streams = self._prog_embed(B, T)(self.params["embed"], jnp.asarray(tokens))
        else:
            streams = self._prog_embed(B, T, True)(self.params["embed"], jnp.asarray(tokens), jnp.asarray(override[0]),
                                                   jnp.asarray(override[1]))
        new_caches = []
        has_cache = caches is not None
        pos0, length = jnp.int32(pos0), jnp.int32(length)
        for g in self.groups["decode" if use_recurrent else "prefill"]:
            prog = self._prog_group(g, B, T, has_cache, use_recurrent)
            streams, ncs = prog([self.params["layers"][i] for i in g], streams, pos0, length,
                                [caches[i] for i in g] if has_cache else None)
            new_caches.extend(ncs)
            self.launches += 1
        return streams, new_caches

    def _run(self, tokens, caches, pos0, use_recurrent, length=None, want_logits=True, want_streams=False, override=None):
        B, T = tokens.shape
        length = T if length is None else length
        streams, new_caches = self._run_layers(tokens, caches, pos0, use_recurrent, length, override)
        logits = None
        if want_logits:
            logits = self._prog_head(B, T)(self.params["norm"], self.params["lm_head"], streams, jnp.int32(length))
        return (logits, new_caches, streams) if want_streams else (logits, new_caches)

    def decode_sample(self, token, caches, pos, sampler: DeviceSampler):
        """`decode` with the next token sampled on the device: returns (ids [B] int32 device array, logits [B,V],
        caches, pos + 1). Only the ids need to reach the host (one small transfer instead of the [B,V] logits)."""
        B = token.shape[0]
        n = self.cfg.n_layers
        extra = caches[n:] if len(caches) > n else []
        streams, new = self._run_layers(np.asarray(token).reshape(B, 1), caches[:n], pos, True, 1)
        if getattr(self, "_one", None) is None:          # a replicated constant (no per-step scalar transfer); lazy so
            self._one = jax.device_put(jnp.int32(1), NamedSharding(self.mesh, P()))   # live engines survive a reload
        ids, logits, nk = self._prog_head(B, 1, True)(self.params["norm"], self.params["lm_head"], streams, self._one,
                                                      *sampler.bind(self.mesh))
        sampler.advance(nk)
        return ids, logits, new + extra, pos + 1

    # ---- batched decode of independent streams (continuous batching): one cache set per stream, per-row positions
    def _prog_embed_rows(self, B):
        """(embed, tokens [B] int32, pos [B] int32) -> (streams [B,1,hc,D], pos + 1): the positions stay on the
        device (no per-step host transfer), the program advances them."""
        key = ("embed_rows", B)
        if key not in self._progs:
            def prog(embed, tokens, pos):
                x = self._embed(embed, tokens[:, None]).astype(self.lcfg.dtype)
                return jnp.broadcast_to(x[:, :, None, :], (B, 1, self.cfg.hc, x.shape[-1])), pos + 1
            self._progs[key] = jax.jit(shard_map(prog, mesh=self.mesh, in_specs=(self.specs["embed"], R, R),
                                                 out_specs=(R, R), check_vma=False))
        return self._progs[key]

    def _prog_group_rows(self, layers, B):
        """Decode step of `layers` for B independent streams: (params, streams [B,1,hc,D], pos [B], caches) with
        caches[j][b] = layer j's cache of stream b (batch dim 1, donated) -> (streams, new caches[j][b])."""
        kinds = tuple(self._kind(i) for i in layers)
        key = ("group_rows", kinds, B)
        if key not in self._progs:
            lts = [self.cfg.layer_types[i] for i in layers]
            mts = [self.cfg.mlp_types[i] for i in layers]
            css = [[self._cache_spec(i)] * B for i in layers]
            rep = list(layers)
            from glm53 import quant8 as Q8

            def prog(ps, streams, pos, caches):
                pos_b = [pos[b] for b in range(B)]
                new = []
                for j, p in enumerate(ps):
                    p = Q8.dequant_tree(p, self.lcfg.dtype)
                    fetch = self.expert_fetch if mts[j] == "sparse" else None
                    streams, ncs = M.decoder_layer_rows(p, streams, self.lcfg, lts[j], mts[j], caches[j], pos_b, fetch,
                                                        "auto", layer=rep[j], cap=self.max_len)
                    new.append(ncs)
                return streams, new
            in_specs = ([self.specs["layers"][i] for i in layers], R, R, css)
            sm = shard_map(prog, mesh=self.mesh, in_specs=in_specs, out_specs=(R, css), check_vma=False)
            self._progs[key] = jax.jit(sm, donate_argnums=(3,))
        return self._progs[key]

    def device_positions(self, pos):
        """Host positions (ints) -> one replicated int32 [B] device array (the form `decode_rows` carries)."""
        return jax.device_put(np.asarray(pos, np.int32).reshape(-1), NamedSharding(self.mesh, P()))

    def decode_rows(self, tokens, cache_sets, pos, sampler: DeviceSampler | None = None):
        """One decode step of B independent streams in one batch: `tokens` [B] (their last tokens; a device array
        from the previous step or host ints), `cache_sets[b]` = stream b's own cache list (as `prefill` /
        `restore_prefix` return them, batch dim 1; consumed), `pos` [B] int32 replicated device array
        (`device_positions`) or host ints. Returns (ids [B] device int32 or None without a sampler, logits [B,V],
        new cache sets, pos + 1 on the device). Streams may sit at any positions; the MoE/mHC/projections run
        batched, the attention per row. Compiles one program set per B."""
        B = len(cache_sets)
        n = self.cfg.n_layers
        assert all(len(c) >= n for c in cache_sets), "every stream needs a full cache set"
        if not isinstance(pos, jax.Array):
            pos = self.device_positions(pos)
        if not (isinstance(tokens, jax.Array) and tokens.shape == (B,) and tokens.dtype == jnp.int32):
            tokens = jnp.asarray(np.asarray(tokens, np.int32).reshape(B))   # (a device array from the last step passes through)
        streams, pos_next = self._prog_embed_rows(B)(self.params["embed"], tokens, pos)
        new_sets = [[] for _ in range(B)]
        for g in self.groups["decode"]:
            prog = self._prog_group_rows(g, B)
            streams, ncs = prog([self.params["layers"][i] for i in g], streams, pos,
                                [[cache_sets[b][i] for b in range(B)] for i in g])
            for j in range(len(g)):
                for b in range(B):
                    new_sets[b].append(ncs[j][b])
            self.launches += 1
        for b in range(B):                                   # extra entries (an MTP cache) pass through untouched
            new_sets[b].extend(cache_sets[b][n:])
        if sampler is None:
            logits = self._prog_head(B, 1)(self.params["norm"], self.params["lm_head"], streams, jnp.int32(1))
            return None, logits, new_sets, pos_next
        if getattr(self, "_one", None) is None:
            self._one = jax.device_put(jnp.int32(1), NamedSharding(self.mesh, P()))
        ids, logits, nk = self._prog_head(B, 1, True)(self.params["norm"], self.params["lm_head"], streams, self._one,
                                                      *sampler.bind(self.mesh))
        sampler.advance(nk)
        return ids, logits, new_sets, pos_next

    def prefill(self, tokens, caches=None, pos0=0, embeds=None):
        """Prefill `tokens` [B,T] into a fresh context (caches=None) or continue one at position pos0 (the caches
        are consumed). Returns (logits of the last token, caches, pos0 + T). With an MTP layer installed the MTP
        cache (entry n_layers) is prefilled too: MTP position p takes (token p+1, final hidden p), so a piece
        covers positions [pos0-1, pos0+L-1) (or [0, L-1) for the first piece) and the last hidden is carried.
        `embeds` = (idx [M] int, vec [M, D]) replaces the embeddings of tokens[0, idx] (B = 1; image tokens,
        glm53.vision) — the MTP layer's own embedding input keeps the table rows (drafts only; verify is exact)."""
        tokens = np.asarray(tokens)
        B, T = tokens.shape
        assert T >= 1 and pos0 + T <= self.max_len, (pos0, T, self.max_len)
        if caches is None:
            caches = self.alloc_caches(B)
        if embeds is not None:
            assert B == 1
            e_idx, e_vec = np.asarray(embeds[0], np.int64), np.asarray(embeds[1])
        logits = None
        main, mc = (caches[:self.cfg.n_layers], caches[self.cfg.n_layers]) if self.mtp is not None else (caches, None)
        for a in range(0, T, self.prefill_piece):
            seg = tokens[:, a:a + self.prefill_piece]
            L = seg.shape[1]
            Tp = self._bucket(L)
            padded = np.zeros((B, Tp), np.int32)
            padded[:, :L] = seg
            ov = None
            if embeds is not None:
                sel = (e_idx >= a) & (e_idx < a + L)
                if sel.any():
                    vec = np.zeros((B, Tp, e_vec.shape[-1]), e_vec.dtype)
                    mask = np.zeros((B, Tp), bool)
                    vec[0, e_idx[sel] - a] = e_vec[sel]
                    mask[0, e_idx[sel] - a] = True
                    ov = (vec, mask)
            if mc is None:
                logits, main = self._run(padded, main, pos0 + a, False, length=L, want_logits=a + L >= T, override=ov)
                continue
            logits, main, streams = self._run(padded, main, pos0 + a, False, length=L, want_logits=a + L >= T,
                                              want_streams=True, override=ov)
            hidden = self._prog_hidden(B, Tp)(self.params["norm"], streams)                 # [B,Tp,D]
            p0 = pos0 + a
            if p0 == 0:                                        # rows 0..L-2: token j+1 with hidden j
                mt = np.zeros((B, Tp), np.int32); mt[:, :L - 1] = seg[:, 1:]
                hp = jnp.concatenate([hidden[:, :Tp - 1], jnp.zeros_like(hidden[:, :1])], axis=1)
                start, length = 0, L - 1
            else:                                              # rows p0-1..p0+L-2: token p0+j with hidden p0+j-1
                mt = padded
                hp = jnp.concatenate([mc["h"], hidden[:, :Tp - 1]], axis=1)
                start, length = p0 - 1, L
            mcache = {k: v for k, v in mc.items() if k != "h"}
            if length > 0:
                _, _, mcache = self._prog_mtp(B, Tp, True)(self.mtp, self.params["embed"], self.params["lm_head"],
                                                           jnp.asarray(mt), hp, jnp.int32(start), jnp.int32(length), mcache)
            mc = {**mcache, "h": lax.dynamic_index_in_dim(hidden, L - 1, axis=1, keepdims=True)}
        return logits, (main + [mc] if mc is not None else main), pos0 + T

    # ---- speculative decoding with the MTP (NextN) layer
    def set_mtp(self, params, qtypes, k=1):
        """Install the MTP layer: `params` from checkpoint.load_mtp_layer (2-D matrices bf16) plus its expert planes
        (pack_layer_from_gguf) under mlp["gate_q"/"up_q"/"down_q"]; `qtypes` their formats; `k` drafts per step.
        The layer's attention cache + carried hidden state become cache entry n_layers (alloc_caches)."""
        col, row = P(None, AXIS), P(AXIS, None)
        attn = {"q_a": R, "q_a_norm": R, "q_b": col, "kv_a": R, "kv_a_norm": R, "kv_b": col, "o": row,
                "indexer": {kk: R for kk in params["attn"]["indexer"]}}
        mlp = {"router_w": R, "router_bias": R, "shared": {"gate": col, "up": col, "down": row}}
        for kk in ("gate_q", "up_q", "down_q"):
            mlp[kk] = {pk: P(AXIS) for pk in params["mlp"][kk]}
        spec = {"ln1": R, "ln2": R, "attn": attn, "mlp": mlp, "enorm": R, "hnorm": R, "eh_proj": R, "head_norm": R}
        self.expert_fetch.qtypes[self.cfg.n_layers] = qtypes
        self.mtp = jax.tree.map(lambda a, sp: a if hasattr(a, "sharding") and a.sharding == NamedSharding(self.mesh, sp)
                                else jax.device_put(a, NamedSharding(self.mesh, sp)), params, spec)
        self.mtp_spec, self.mtp_k = spec, k

    def _prog_mtp(self, B, T, has_cache=True):
        """(mtp params, embed, lm_head, tokens [B,T] = the token AFTER each MTP position, h_prev [B,T,D], pos0,
        length, cache) -> (logits of the last real position [B,V], hidden [B,T,D], new cache)."""
        key = ("mtp", B, T, has_cache)
        if key not in self._progs:
            from glm53 import quant8 as Q8
            cs = {k: v for k, v in self._cache_spec(self.cfg.n_layers).items() if k != "h"}

            def prog(mp, embed, lm_head, tokens, h_prev, pos0, length, cache):
                mp = Q8.dequant_tree(mp, self.lcfg.dtype)
                e = self._embed(embed, tokens).astype(self.lcfg.dtype)
                e = jnp.where((pos0 + jnp.arange(T))[None, :, None] == 0, jnp.zeros((), e.dtype), e)   # no position -1
                h, nc = M.mtp_layer(mp, e, h_prev, self.lcfg, cache, pos0, self.expert_fetch, "auto", self.max_len,
                                    length, layer=self.cfg.n_layers)
                last = lax.dynamic_index_in_dim(h, length - 1, axis=1, keepdims=False)
                return self._logits(lm_head, last), h, nc
            in_specs = (self.mtp_spec, self.specs["embed"], self.specs["lm_head"], R, R, R, R, cs)
            sm = shard_map(prog, mesh=self.mesh, in_specs=in_specs, out_specs=(R, R, cs), check_vma=False)
            self._progs[key] = jax.jit(sm, donate_argnums=(7,))
        return self._progs[key]

    def _run_verify(self, tokens, caches, pos0, sampler=None):
        """Recurrent T-token step returning (logits [B,T,V], caches, per-layer histories, hidden [B,T,D], ids):
        ids = the device-sampled token of every position [B,T] when a DeviceSampler is given, else None."""
        B, T = tokens.shape
        streams = self._prog_embed(B, T)(self.params["embed"], jnp.asarray(tokens))
        new, hists = [], []
        pos0 = jnp.int32(pos0)
        for g in self.groups["decode"]:
            prog = self._prog_group(g, B, T, True, True, hist=True)
            streams, ncs, hs = prog([self.params["layers"][i] for i in g], streams, pos0, jnp.int32(T), [caches[i] for i in g])
            new.extend(ncs); hists.extend(hs)
            self.launches += 1
        if sampler is None:
            logits, hidden = self._prog_head_all(B, T)(self.params["norm"], self.params["lm_head"], streams)
            return logits, new, hists, hidden, None
        ids, logits, hidden, nk = self._prog_head_all(B, T, True)(self.params["norm"], self.params["lm_head"], streams,
                                                                  *sampler.bind(self.mesh))
        sampler.advance(nk)
        return logits, new, hists, hidden, ids

    def spec_decode(self, token, caches, pos, k=None, sampler=None, draft_override=None, stop_ids=()):
        """One speculative step from the committed token `token` [B] at position `pos` (B = 1): the MTP layer
        drafts k tokens, the main model verifies them in one (k+1)-token recurrent step, the longest accepted prefix
        is kept (recurrent states and pool tails rolled back to it). `sampler` decides the target's token per
        position: None = argmax, a host callable `sampler(logits_row) -> int`, or a `DeviceSampler` (sampled inside
        the head program; only the ids cross to the host). A draft is accepted iff it equals the target's own choice,
        which is exact for greedy and a valid, if conservative, scheme for sampling. A draft in `stop_ids` ends the
        accepted run (it is emitted as the last, unfed token). Returns (emitted tokens [1..k+1] — all but the last
        were fed to the model, logits of the last emitted position [B,V], caches, new position)."""
        k = self.mtp_k if k is None else k
        n = self.cfg.n_layers
        B = token.shape[0]
        assert B == 1 and self.mtp is not None
        mc = caches[n]
        mcache = {kk: v for kk, v in mc.items() if kk != "h"}
        hp = mc["h"]
        tok = jnp.asarray(np.asarray(token, np.int32).reshape(B, 1))
        drafts, tails = [], []                                  # the draft chain stays on the device (one host sync per step)
        for j in range(k):
            tails.append((mcache["tk"], mcache["tg"]))          # before draft j (donated: keep the objects, not copies)
            lg, hd, mcache = self._prog_mtp(B, 1, True)(self.mtp, self.params["embed"], self.params["lm_head"],
                                                        tok, hp, jnp.int32(pos - 1 + j), jnp.int32(1),
                                                        {**mcache, "tk": jnp.copy(mcache["tk"]), "tg": jnp.copy(mcache["tg"])})
            d = jnp.asarray([[int(draft_override[j])]], jnp.int32) if draft_override is not None else jnp.argmax(lg, axis=-1)[:, None].astype(jnp.int32)
            drafts.append(d); tok = d; hp = hd
        seq = jnp.concatenate([jnp.asarray(np.asarray(token, np.int32).reshape(B, 1))] + drafts, axis=1)   # [B,k+1]
        dev = isinstance(sampler, DeviceSampler)
        logits, new, hists, hidden, ids = self._run_verify(seq, caches[:n], pos, sampler if dev else None)
        seq_h = np.asarray(seq)[0]
        drafts_h = [int(x) for x in seq_h[1:]]
        if dev:                                                 # only k+1 ids cross to the host, not [k+1, V] logits
            preds = [int(x) for x in np.asarray(ids[0])]
        else:
            lg_h = np.asarray(logits[0])
            preds = [int(np.argmax(lg_h[j])) if sampler is None else int(sampler(lg_h[j])) for j in range(k + 1)]
        a = 0                                                   # a stop token is never fed: it ends the accepted run
        while a < k and preds[a] == drafts_h[a] and drafts_h[a] not in stop_ids:
            a += 1
        if a < k:
            rolled = self._prog_rollback(B, k + 1)(hists, jnp.int32(a + 1))
            new = [({**c, **r} if r else c) for c, r in zip(new, rolled)]
            if a + 1 < k:                                       # drafts a+1.. were fed rejected tokens: tail as before draft a+1
                mcache = {**mcache, "tk": tails[a + 1][0], "tg": tails[a + 1][1]}
        emitted = drafts_h[:a] + [preds[a]]
        h_next = lax.dynamic_index_in_dim(hidden, a, axis=1, keepdims=True)
        return emitted, logits[:, a], new + [{**mcache, "h": h_next}], pos + a + 1

    def decode(self, token, caches, pos):
        B = token.shape[0]
        n = self.cfg.n_layers
        extra = caches[n:] if len(caches) > n else []
        logits, caches = self._run(np.asarray(token).reshape(B, 1), caches[:n], pos, True)
        return logits, caches + extra, pos + 1
'''
FILES["glm53/scheduler.py"] = r'''"""Continuous batching for `ResidentLayerEngine.decode_rows`: independent streams decoded together, admitted between
steps, prefilled piece by piece with decode steps of the running streams in between, each keeping its own cache set.

One engine thread (`Scheduler.run`) owns every JAX call. Client threads `submit` a `Request` and wait on its
`done` event (tokens arrive through `on_token`, called on the engine thread). Cache sets on the chips = the active
streams' + finished contexts kept LIVE for the next turn (LRU), at most `max_sets` in total; beyond that, finished
contexts are parked in host memory (`SnapStore`) and resumed when a later prompt extends them. Between steps only the
sampled ids cross to the host: token ids, positions and the sampler state stay on the device.
"""
import collections
import threading
import time

import numpy as np
import jax
import jax.numpy as jnp

from glm53.engine import DeviceSampler


def match_prefix(live_ids, pos, prompt, quiet=False):
    """Default matcher: (k, fed) when the context (live_ids[:pos]) is a prefix of `prompt`: prompt[k:] must still be
    prefilled and `fed` = the ids the engine will have seen after that; (0, None) otherwise."""
    if pos == 0 or len(prompt) < pos:
        return 0, None
    a = np.asarray(live_ids[:pos]); b = np.asarray(prompt[:pos])
    if a.shape != b.shape or not np.array_equal(a, b):
        return 0, None
    return pos, list(live_ids[:pos]) + list(prompt[pos:])


class Request:
    """A generation request. `prompt`: token ids (the server's signature ids: image tokens negative, `imgs` maps
    them; the scheduler's `feed` turns slices into real ids + embedding overrides). Results: `out` (emitted tokens,
    the stop token included when hit), `stop_reason` ("stop" | "length" | "cancelled" | "error"), timings."""

    def __init__(self, prompt, max_new, temperature=1.0, top_p=1.0, imgs=None, on_token=None, rid=None, on_done=None,
                 budget=None):
        self.prompt = [int(t) for t in prompt]
        self.max_new, self.temperature, self.top_p = int(max_new), float(temperature), float(top_p)
        self.imgs, self.on_token, self.rid, self.on_done = imgs, on_token, rid, on_done
        self.budget = budget            # (n, end_id, forced_ids): unless `end_id` was emitted within the first n output
                                        # tokens, the next tokens are forced to `forced_ids` (a thinking budget)
        self.out = []
        self.done = threading.Event()
        self.error = None
        self.cancelled = False
        self.stop_reason = None
        self.reused = 0
        self.prefill_s = 0.0
        self.t_submit, self.t_first, self.t_end = time.time(), None, None

    def cancel(self):
        self.cancelled = True

    @property
    def decode_s(self):
        return (self.t_end or time.time()) - (self.t_first or self.t_submit)


class _Stream:
    """An admitted request: its own cache set, its position, the ids fed so far and the last (unfed) token."""

    def __init__(self, req, caches, pos, fed, tok, logits):
        self.req, self.caches, self.pos, self.fed, self.tok = req, caches, pos, list(fed), tok
        self.seen = []                                   # tokens fed beyond `fed`
        self.logits = logits                             # logits [1,V] after the last fed token (predict `tok`)
        self.max_new = req.max_new
        self.open = req.budget is not None               # the budgeted block (thinking) is still open
        self.force = []                                  # tokens to feed instead of the sampled ones (budget reached)


class SnapStore:
    """Host-memory LRU of compact context snapshots keyed by their token ids (see `ResidentLayerEngine.snapshot_prefix`).
    `park` moves a context off the chips, `lookup` finds the entry a prompt extends (the matcher handles dropped
    thinking blocks and boundary drift when the server passes its own), `restore` brings it back into fresh
    full-capacity caches. A parked conversation keeps its two latest turn boundaries; older strict prefixes are
    dropped unless pinned (pinned = a conversation's system section, the branch point new sessions start from)."""

    def __init__(self, eng, max_bytes, rows_bucket=1024, match_len=match_prefix, log=print, state=None):
        self.eng, self.max_bytes, self.rows_bucket = eng, max_bytes, rows_bucket
        self.match_len, self.log = match_len, log
        self.state = state if state is not None else {}
        self.entries = collections.OrderedDict()                     # tuple(ids) -> entry, oldest first
        self.bytes = 0

    def park(self, ids, caches, pos, logits, pinned=False):
        """Snapshot `caches` (a context of `pos` tokens = `ids`) into host memory; the caches stay valid."""
        key = tuple(int(t) for t in ids)
        if key in self.entries:
            e = self.entries[key]
            e["pinned"] = e["pinned"] or pinned
            if logits is not None and e["logits"] is None:
                e["logits"] = np.asarray(logits)
            self.entries.move_to_end(key)
            return e
        t = time.time()
        snap = self.eng.snapshot_prefix(caches, pos, rows_bucket=self.rows_bucket)
        host = self.eng.snapshot_to_host(snap)
        del snap
        e = {"ids": list(key), "n": pos, "snap": host, "logits": None if logits is None else np.asarray(logits),
             "bytes": host["bytes"], "pinned": pinned}
        if not pinned:                                                # supersede our own earlier turns but the latest
            older = sorted((k for k, o in self.entries.items() if not o["pinned"] and len(k) < len(key) and key[:len(k)] == k),
                           key=len)
            for k in older[:-1]:
                self._drop(k)
        self.entries[key] = e
        self.bytes += e["bytes"]
        while self.bytes > self.max_bytes and len(self.entries) > 1:
            victims = [k for k, o in self.entries.items() if not o["pinned"]] or list(self.entries)
            self._drop(victims[0])
        st = self.state
        st["snap_parks" if not pinned else "snap_pins"] = st.get("snap_parks" if not pinned else "snap_pins", 0) + 1
        st["snap_entries"], st["snap_bytes"] = len(self.entries), self.bytes
        st["snap_s"] = st.get("snap_s", 0.0) + time.time() - t
        self.log(f"parked {pos} tokens ({e['bytes'] / 1e6:.0f} MB, {'pinned' if pinned else 'lru'}) in {time.time() - t:.2f}s; "
                 f"store {len(self.entries)} entries, {self.bytes / 1e9:.2f} GB")
        return e

    def _drop(self, key):
        e = self.entries.pop(key)
        self.bytes -= e["bytes"]

    def lookup(self, prompt):
        """-> (k, fed, entry) for the entry that covers most of `prompt`, or (0, None, None)."""
        best = (0, None, None)
        for key, e in list(self.entries.items()):
            if e["n"] <= best[0]:
                continue
            k, fed = self.match_len(e["ids"], e["n"], prompt, quiet=True)
            if k > best[0] and not (k == len(prompt) and e["logits"] is None):
                best = (k, fed, e)
        if best[2] is not None:
            self.entries.move_to_end(tuple(best[2]["ids"]))
        return best

    def restore(self, e):
        t = time.time()
        caches = self.eng.restore_prefix(self.eng.snapshot_from_host(e["snap"]))
        self.state["snap_hits"] = self.state.get("snap_hits", 0) + 1
        self.state["snap_s"] = self.state.get("snap_s", 0.0) + time.time() - t
        return caches


def _argmax_first(logits_row, temperature, top_p):
    return int(np.argmax(logits_row))


class Scheduler:
    """See the module docstring. `feed(req, a, b) -> (ids [n] int32, embeds or None)` renders req.prompt[a:b] for
    the engine (the server attaches its image data to the request); `match_len(live_ids, pos, prompt, quiet=False) -> (k, fed)`; `system_end(prompt)` = length of
    the system section worth pinning (0 = none); `first_sample(logits_row, temperature, top_p) -> int` samples the
    first token of a request on the host (the rest come from the device sampler)."""

    def __init__(self, eng, stop_ids, max_streams=4, max_sets=None, feed=None, match_len=match_prefix,
                 system_end=None, first_sample=_argmax_first, snaps=None, log=print, state=None,
                 base_min=512, snap_min=256, piece=None, seed=None, min_free_gb=0.65, max_wait_s=90.0):
        self.eng, self.stop_ids = eng, set(int(t) for t in stop_ids)
        self.max_streams = max(1, int(max_streams))
        self.max_sets = max(self.max_streams, int(max_sets if max_sets is not None else max_streams + 1))
        self.feed = feed or (lambda req, a, b: (np.asarray(req.prompt[a:b], np.int32), None))
        self.match_len, self.system_end = match_len, (system_end or (lambda prompt: 0))
        self.first_sample = first_sample
        self.snaps = snaps if snaps is not None else SnapStore(eng, 0, 1, match_len, log, state)
        self.log, self.state = log, (state if state is not None else {})
        self.base_min, self.snap_min = base_min, snap_min
        self.piece = piece or eng.prefill_piece
        self.min_free_gb = min_free_gb                  # HBM headroom an admission needs (prefill temporaries); 0 = off
        self.max_wait_s = max_wait_s                    # a request queued longer than this fails ("queue_timeout")
        self.live = collections.OrderedDict()            # finished contexts on the chips (LRU): key -> ctx dict
        self._live_n = 0
        self.active = []
        self.pending = collections.deque()
        self.cond = threading.Condition()
        self.sampler = DeviceSampler(1.0, 1.0, seed=seed)
        self._members = None                             # ids of the streams the device state below belongs to
        self._toks = self._pos = None
        self._stop = False
        self._paused, self._idle = False, threading.Event()
        self._waits = 0
        self._t_step = None
        self.thread = None
        self.steps = 0

    # ---- client side
    def submit(self, req: Request):
        if len(req.prompt) + 2 > self.eng.max_len:
            self._fail(req, ValueError(f"prompt of {len(req.prompt)} tokens exceeds the context capacity {self.eng.max_len}"))
            return req
        with self.cond:
            self.pending.append(req)
            self.cond.notify()
        return req

    def start(self):
        self.thread = threading.Thread(target=self.run, daemon=True, name="glm-scheduler")
        self.thread.start()
        return self.thread

    def stop(self):
        with self.cond:
            self._stop = True
            self.cond.notify_all()
        if self.thread is not None:
            self.thread.join(timeout=60)
        for ctx in self.live.values():
            self._free_set(ctx["caches"])
        for s in self.active:
            self._free_set(s.caches)
        self.live.clear(); self.active.clear(); self._toks = self._pos = None

    @staticmethod
    def _free_set(caches):
        """Release a dropped cache set's HBM now (`Array.delete`), whatever else still references the arrays: a
        dropped set kept alive by a stray reference (seen once after a concurrent burst, 2026-09-13) cost a stream
        slot for the rest of the session — refcounts alone are not a guarantee."""
        for x in jax.tree.leaves(caches):
            try:
                x.delete()
            except Exception:  # noqa: BLE001
                pass

    @property
    def n_sets(self):
        return len(self.active) + len(self.live)

    def free_gb(self):
        """Free HBM on chip 0 in GB, or None when the backend reports no memory statistics (CPU tests)."""
        st = self.eng.mesh.devices.flat[0].memory_stats() or {}
        return (st["bytes_limit"] - st["bytes_in_use"]) / 1e9 if "bytes_limit" in st else None

    def _headroom(self):
        """True when an admission can allocate a cache set and run its prefill: parks live contexts (LRU) until the
        set budget and the HBM margin allow it; False when only active streams hold the chips (wait for one)."""
        self._make_room()
        f = self.free_gb()
        return f is None or not self.min_free_gb or f >= self.min_free_gb

    # ---- engine thread
    def pause(self, timeout=60.0):
        """Stop the engine thread between steps (running streams stall, nothing is admitted) until `resume`; returns
        once the thread is idle. For probe jobs that need the chips."""
        with self.cond:
            self._paused = True
            self.cond.notify_all()
        return self._idle.wait(timeout)

    def resume(self):
        with self.cond:
            self._paused = False
            self.cond.notify_all()

    def run(self):
        while not self._stop:
            with self.cond:
                while not self._stop and (self._paused or (not self.pending and not self.active)):
                    if self._paused:
                        self._idle.set()
                    self._t_step = None
                    self.cond.wait(timeout=1.0)
                self._idle.clear()
                if self._stop:
                    break
                req = None
                while self.pending and self.max_wait_s and time.time() - self.pending[0].t_submit > self.max_wait_s:
                    old = self.pending.popleft()                        # queued too long: fail it (the client can retry)
                    self._fail(old, TimeoutError(f"queued for {time.time() - old.t_submit:.0f}s without a free cache set"), "queue_timeout")
                if self.pending and len(self.active) < self.max_streams and self._headroom():
                    req = self.pending.popleft()
                elif self.pending and not self.active:                  # nothing running and still no room
                    if self._waits % 10 == 0:
                        self.log(f"admission waits: {len(self.live)} live contexts, free HBM {self.free_gb()} GB")
                    self._waits += 1
                    self.cond.wait(timeout=1.0)
            if req is not None:
                if req.cancelled:
                    self._fail(req, None, "cancelled"); continue
                try:
                    self._admit(req)
                except Exception as e:  # noqa: BLE001
                    import gc, traceback
                    self.log("admission failed:", traceback.format_exc()[-1500:])
                    self._fail(req, RuntimeError(f"{type(e).__name__}: {str(e)[:800]}"))   # no traceback: its frames
                    del e                                                                    # would pin the half-built caches
                    gc.collect()
                continue
            if self.active:
                try:
                    self._step()
                except Exception as e:  # noqa: BLE001
                    import gc, traceback
                    self.log("decode step failed:", traceback.format_exc()[-1500:])
                    err = RuntimeError(f"{type(e).__name__}: {str(e)[:800]}")
                    del e
                    for s in list(self.active):
                        s.req.error = err
                        self._finish(s, "error", keep=False)
                    self.active = []; self._members = None; self._toks = self._pos = None
                    gc.collect()

    def _fail(self, req, error, reason="error"):
        """Finish a request that never became a stream (capacity error, admission failure, cancelled while queued)."""
        req.error, req.stop_reason, req.t_end = error, reason, time.time()
        if req.on_done is not None:
            try:
                req.on_done()
            except Exception:  # noqa: BLE001
                pass
        req.done.set()

    def _make_room(self):
        """Park LRU live contexts until a cache set can be allocated within the set budget AND the HBM margin
        (`min_free_gb`, prefill temporaries); a parked set is freed (donated caches)."""
        while self.live:                                                 # (no gc.collect() here: a full collection
            f = self.free_gb()                                           #  costs ~14 s in a process with hundreds of
            if self.n_sets < self.max_sets and (f is None or not self.min_free_gb or f >= self.min_free_gb):
                break                                                    #  compiled programs; refcounts free the set)
            key, ctx = self.live.popitem(last=False)
            if ctx["pos"] >= self.snap_min:
                self.snaps.park(ctx["ids"], ctx["caches"], ctx["pos"], ctx["logits"])
            self._free_set(ctx["caches"])
            del ctx

    def _prefill(self, req, a, b, caches, pos):
        """prompt[a:b] into `caches` at `pos`, one piece at a time with a decode step of the running streams between
        pieces (chunked admission). Returns (logits, caches, pos)."""
        logits = None
        for x in range(a, b, self.piece):
            y = min(b, x + self.piece)
            ids, emb = self.feed(req, x, y)
            ids = np.asarray(ids, np.int32).reshape(1, -1)
            logits, caches, pos = self.eng.prefill(ids, caches, pos, embeds=emb)
            if y < b and self.active:
                self._step()
        return logits, caches, pos

    def _admit(self, req):
        prompt = req.prompt
        t0 = time.time()
        st = self.state
        # 1. a finished context still on the chips that the prompt extends
        best = (0, None, None)
        for key, c in self.live.items():
            k, fed = self.match_len(c["ids"], c["pos"], prompt, quiet=True)
            if k > best[0] and not (k == len(prompt) and c["logits"] is None):
                best = (k, fed, key)
        c = None                                                         # (keep no reference to a context _make_room may drop)
        if best[2] is None and self.live:
            self.log(f"no live context matched ({len(self.live)} live, {len(self.snaps.entries)} parked)")
        if best[2] is not None:
            k, fed, key = best
            ctx = self.live.pop(key)
            st["prefix_hits"] = st.get("prefix_hits", 0) + 1
            st["prefix_tokens_reused"] = st.get("prefix_tokens_reused", 0) + k
            if k == len(prompt):
                logits, caches, pos = ctx["logits"], ctx["caches"], ctx["pos"]
            else:
                logits, caches, pos = self._prefill(req, k, len(prompt), ctx["caches"], ctx["pos"])
            req.reused = k
        else:
            self._make_room()
            k, fed, entry = self.snaps.lookup(prompt)
            if k > 0:                                                    # 2. a parked context the prompt extends
                st["prefix_tokens_reused"] = st.get("prefix_tokens_reused", 0) + k
                caches = self.snaps.restore(entry)
                if k == len(prompt):
                    logits, pos = entry["logits"], entry["n"]
                else:
                    logits, caches, pos = self._prefill(req, k, len(prompt), caches, entry["n"])
                req.reused = k
            else:                                                        # 3. from scratch (pin the system section)
                fed = list(prompt)
                n_sys = self.system_end(prompt)
                if n_sys >= self.base_min and n_sys < len(prompt):
                    _, caches, pos = self._prefill(req, 0, n_sys, None, 0)
                    self.snaps.park(prompt[:n_sys], caches, pos, None, pinned=True)
                    logits, caches, pos = self._prefill(req, n_sys, len(prompt), caches, pos)
                else:
                    logits, caches, pos = self._prefill(req, 0, len(prompt), None, 0)
        jax.block_until_ready(logits)
        self._t_step = None
        req.prefill_s = time.time() - t0
        st["prefill_s"] = st.get("prefill_s", 0.0) + req.prefill_s
        tok = self.first_sample(np.asarray(logits)[0], req.temperature, req.top_p)
        s = _Stream(req, caches, pos, fed, tok, logits)
        s.max_new = max(1, min(req.max_new, self.eng.max_len - pos - 1))
        req.t_first = time.time()
        self.active.append(s); self._members = None
        self._emit(s, tok)

    def _emit(self, s, tok):
        req = s.req
        req.out.append(tok)
        if s.open and tok == req.budget[1]:
            s.open = False
        if req.on_token is not None and not req.cancelled:
            try:
                req.on_token(tok)
            except Exception as e:  # noqa: BLE001
                self.log(f"on_token failed ({e!r}): cancelling {req.rid}")
                req.cancelled = True
        if tok in self.stop_ids:
            self._finish(s, "stop")
        elif len(req.out) >= s.max_new or s.pos + 1 >= self.eng.max_len:
            self._finish(s, "length")
        elif req.cancelled:
            self._finish(s, "cancelled")

    def _finish(self, s, reason, keep=True):
        """The stream's context becomes a live (on-chip) context: ids = everything the engine has seen (the last
        emitted token is never fed), logits = the prediction after them."""
        req = s.req
        if s in self.active:
            self.active.remove(s); self._members = None
        if not keep:
            self._free_set(s.caches)
        if keep:
            self._live_n += 1
            lg = s.logits[0][s.logits[1]:s.logits[1] + 1] if isinstance(s.logits, tuple) else s.logits
            self.live[self._live_n] = {"ids": s.fed + s.seen, "caches": s.caches, "pos": s.pos, "logits": lg}
        req.stop_reason = req.stop_reason or reason
        req.t_end = time.time()
        self.state["tokens"] = self.state.get("tokens", 0) + len(req.out)
        if req.on_done is not None:
            try:
                req.on_done()
            except Exception as e:  # noqa: BLE001
                self.log(f"on_done failed ({e!r}) for {req.rid}")
        req.done.set()

    def _step(self):
        """One batched decode step of the active streams."""
        for s in list(self.active):
            if s.req.cancelled:
                self._finish(s, "cancelled")
        act = list(self.active)                                         # (streams finishing below leave self.active)
        if not act:
            return
        B = len(act)
        members = tuple(id(s) for s in act)
        if members != self._members:                                    # membership changed: re-place the small state
            self._pos = self.eng.device_positions([s.pos for s in act])
            self._toks = jnp.asarray([s.tok for s in act], jnp.int32)
            self.sampler.set([s.req.temperature for s in act], [s.req.top_p for s in act])
            self._members = members
        t0 = time.perf_counter()
        ids, logits, sets, pos_next = self.eng.decode_rows(self._toks, [s.caches for s in act], self._pos, self.sampler)
        ids_h = np.asarray(ids)                                         # the one device -> host sync per step
        forced = False
        for r, s in enumerate(act):                                     # a budget reached: feed its forced tokens
            if s.open and not s.force and len(s.req.out) >= s.req.budget[0]:
                s.force = list(s.req.budget[2])
            if s.force:
                if not forced:
                    ids_h, forced = np.array(ids_h), True
                ids_h[r] = s.force.pop(0)
        if forced:
            ids = jnp.asarray(ids_h, jnp.int32)
        t1 = time.perf_counter()
        self._toks, self._pos = ids, pos_next
        self.steps += 1
        st = self.state
        st["steps"] = self.steps
        st["step_tokens"] = st.get("step_tokens", 0) + B
        st["decode_s"] = st.get("decode_s", 0.0) + (t1 - t0)          # inside the engine (dispatch + device + sync)
        if self._t_step is not None:                                    # back-to-back steps: the loop's own overhead
            st["step_s"] = st.get("step_s", 0.0) + (t1 - self._t_step)
            st["steps_busy"] = st.get("steps_busy", 0) + 1
        self._t_step = t1
        for r, s in enumerate(act):                                     # every row takes its new caches first
            s.caches, s.pos = sets[r], s.pos + 1
            s.seen.append(s.tok)
            s.tok = int(ids_h[r])
            s.logits = (logits, r)                                      # sliced only when the stream finishes
        for s in act:
            self._emit(s, s.tok)

    # ---- warm-up
    def warmup(self, max_B=None, token=0):
        """Compile the batched decode programs for every batch size up to max_streams (zero caches, position 0)."""
        max_B = max_B or self.max_streams
        for B in range(1, max_B + 1):
            t = time.time()
            sets = [self.eng.alloc_caches(1) for _ in range(B)]
            samp = DeviceSampler([1.0] * B, [0.95] * B, seed=0)
            ids, lg, sets, pos = self.eng.decode_rows(np.full((B,), token, np.int32), sets, [0] * B, samp)
            jax.block_until_ready(ids)
            del sets
            self.log(f"warm-up batched decode B={B}: {time.time() - t:.0f}s")
'''
FILES["glm53/vision.py"] = r'''"""GLM-5.3-Flash vision tower (HF `Glm5NextVisionModel`, weights `model.visual.*`, bf16, 1.13 GB) in pure JAX, plus the
image preprocessing of `Glm5NextImageProcessor` (PIL): dynamic-resolution resize to a 28-pixel grid, 14x14 patches
duplicated over the temporal axis (2), 2x2 spatial merge -> one language-model token per 28x28 pixels.

Forward: patch embed (Conv3d == linear over (c, t, ph, pw) = 1176 inputs) -> 24 pre-norm blocks (RMSNorm; attention
with q/k RMSNorm over head_dim 64, 2-D rotary over (h, w) patch positions, full attention within each image; SwiGLU
MLP with the ±10 clamps, all with biases) -> post RMSNorm -> the 2x2 downsample conv (== linear over (c, i, j)) to
4096 -> merger (proj, LayerNorm, GELU, clamped SwiGLU to 10240 and back). Patches are laid out block-major (the 4
patches of a merge block consecutive), so consecutive groups of 4 form one output token; output tokens are the
merge blocks in row-major order, i.e. the order of the `<|image|>` tokens in the prompt.
"""
import math
import numpy as np
import jax
import jax.numpy as jnp

PATCH, MERGE, TEMPORAL = 14, 2, 2
FACTOR = PATCH * MERGE                                                  # 28 pixels per token side
MEAN = np.array([0.48145466, 0.4578275, 0.40821073], np.float32)          # OPENAI_CLIP_MEAN / STD
STD = np.array([0.26862954, 0.26130258, 0.27577711], np.float32)
EPS = 1e-5


# ----------------------------------------------------------------------------- preprocessing (port of the HF processor)
def smart_resize(height, width, min_tokens=16, max_tokens=8000, factor=FACTOR):
    """The HF `smart_resize` for one image (num_frames = temporal_factor = 2): the padded canvas (H, W), both multiples
    of 28, holding between min_tokens and max_tokens 28x28 tokens."""
    ppt = TEMPORAL * factor ** 2
    min_px, max_px = min_tokens * ppt, max_tokens * ppt
    align = lambda v: math.ceil(v / factor) * factor  # noqa: E731
    frames = TEMPORAL
    ah, aw = align(height), align(width)
    if frames * ah * aw < min_px:
        s = math.sqrt(min_px / (TEMPORAL * height * width))
        ah, aw = align(max(1, math.ceil(height * s))), align(max(1, math.ceil(width * s)))
    if frames * ah * aw > max_px:
        lo, hi = 1, height
        bh, bw = factor, factor
        while lo <= hi:
            ch = (lo + hi) // 2
            cw = max(1, math.floor(width * ch / height))
            H, W = align(ch), align(cw)
            if frames * H * W <= max_px:
                bh, bw = H, W
                lo = ch + 1
            else:
                hi = ch - 1
        ah, aw = bh, bw
    return ah, aw


def preprocess(img, min_tokens=16, max_tokens=8000):
    """PIL image -> (patches float32 [gh*gw, 3*2*14*14] in the model's block-major order, (1, gh, gw)). Mirrors the
    HF processor: RGB, bicubic resize of the content to fit the smart_resize canvas (never upscaled unless the image
    is below min_tokens), zero (black) padding right/bottom, rescale 1/255, CLIP mean/std, patchify."""
    from PIL import Image
    img = img.convert("RGB")
    w, h = img.size
    H, W = smart_resize(h, w, min_tokens, max_tokens)
    ppt = TEMPORAL * FACTOR ** 2
    s = min(H / h, W / w)
    if TEMPORAL * h * w >= ppt * min_tokens:
        s = min(1.0, s)
    ch, cw = max(1, min(H, math.floor(h * s))), max(1, min(W, math.floor(w * s)))
    if (ch, cw) != (h, w):
        img = img.resize((cw, ch), Image.BICUBIC)
    x = np.zeros((H, W, 3), np.uint8)
    x[:ch, :cw] = np.asarray(img, np.uint8)
    x = ((x.astype(np.float32) / 255.0 - MEAN) / STD).transpose(2, 0, 1)             # [3, H, W]
    gh, gw = H // PATCH, W // PATCH
    p = x.reshape(3, gh // MERGE, MERGE, PATCH, gw // MERGE, MERGE, PATCH)
    p = p.transpose(1, 4, 2, 5, 0, 3, 6)                                              # [bh, bw, mi, mj, c, ph, pw]
    p = p.reshape(gh * gw, 3, 1, PATCH, PATCH)
    p = np.broadcast_to(p, (gh * gw, 3, TEMPORAL, PATCH, PATCH)).reshape(gh * gw, 3 * TEMPORAL * PATCH * PATCH)
    return np.ascontiguousarray(p), (1, gh, gw)


def n_tokens(grid):
    t, gh, gw = grid
    return t * gh * gw // (MERGE * MERGE)


def position_ids(grids):
    """[N, 2] (h, w) patch positions in the block-major order, for all images concatenated."""
    out = []
    for t, gh, gw in grids:
        hh, ww = np.meshgrid(np.arange(gh), np.arange(gw), indexing="ij")
        shp = (gh // MERGE, MERGE, gw // MERGE, MERGE)
        hh = hh.reshape(shp).transpose(0, 2, 1, 3).reshape(-1)
        ww = ww.reshape(shp).transpose(0, 2, 1, 3).reshape(-1)
        out.append(np.tile(np.stack([hh, ww], -1), (t, 1)))
    return np.concatenate(out, 0).astype(np.int32)


def segment_ids(grids):
    """[N] image index of every patch (attention stays within an image)."""
    return np.concatenate([np.full(t * gh * gw, i, np.int32) for i, (t, gh, gw) in enumerate(grids)])


# ----------------------------------------------------------------------------- parameters
def from_state_dict(get, depth, hidden, dtype=np.float32):
    """`get(name)` -> np array for the HF names without the `model.visual.` prefix -> our params (matrices
    transposed to [in, out]; the two convs flattened to matrices)."""
    t = lambda n: np.ascontiguousarray(np.asarray(get(n), dtype).T)  # noqa: E731
    v = lambda n: np.asarray(get(n), dtype)  # noqa: E731
    blocks = []
    for i in range(depth):
        b = f"blocks.{i}."
        blocks.append({"n1": v(b + "norm1.weight"), "n2": v(b + "norm2.weight"),
                       "qkv": t(b + "attn.qkv.weight"), "qkv_b": v(b + "attn.qkv.bias"),
                       "proj": t(b + "attn.proj.weight"), "proj_b": v(b + "attn.proj.bias"),
                       "qn": v(b + "attn.q_norm.weight"), "kn": v(b + "attn.k_norm.weight"),
                       "gate": t(b + "mlp.gate_proj.weight"), "gate_b": v(b + "mlp.gate_proj.bias"),
                       "up": t(b + "mlp.up_proj.weight"), "up_b": v(b + "mlp.up_proj.bias"),
                       "down": t(b + "mlp.down_proj.weight"), "down_b": v(b + "mlp.down_proj.bias")})
    pe = np.asarray(get("patch_embed.proj.weight"), dtype)                              # [C, 3, 2, 14, 14]
    ds = np.asarray(get("downsample.weight"), dtype)                                    # [O, C, 2, 2]
    return {"patch": np.ascontiguousarray(pe.reshape(pe.shape[0], -1).T), "patch_b": v("patch_embed.proj.bias"),
            "blocks": blocks, "post_ln": v("post_layernorm.weight"),
            "ds": np.ascontiguousarray(ds.reshape(ds.shape[0], -1).T), "ds_b": v("downsample.bias"),
            "m_proj": t("merger.proj.weight"), "m_ln": v("merger.post_projection_norm.weight"),
            "m_ln_b": v("merger.post_projection_norm.bias"), "m_gate": t("merger.gate_proj.weight"),
            "m_up": t("merger.up_proj.weight"), "m_down": t("merger.down_proj.weight"),
            "n_heads": None}


def from_hf_module(m, dtype=np.float32):
    sd = {k: v.detach().float().cpu().numpy() for k, v in m.state_dict().items()}
    p = from_state_dict(sd.__getitem__, m.config.depth, m.config.hidden_size, dtype)
    p["n_heads"] = m.config.num_heads
    return p


def load_vision(reader, dtype=np.float32, depth=24, hidden=1024, n_heads=16):
    """From a checkpoint `ShardReader` (names `model.visual.*`, all in the last shard)."""
    p = from_state_dict(lambda n: reader.get("model.visual." + n, dtype), depth, hidden, dtype)
    p["n_heads"] = n_heads
    return p


# ----------------------------------------------------------------------------- forward
def _rms(x, w, eps=EPS):
    xf = x.astype(jnp.float32)
    return (xf * jax.lax.rsqrt(jnp.mean(xf * xf, -1, keepdims=True) + eps) * w.astype(jnp.float32)).astype(x.dtype)


def _rope(pos, head_dim, theta=10000.0):
    d = head_dim // 2                                                    # rotary dim per axis pair = 32 -> 16 freqs
    inv = 1.0 / (theta ** (np.arange(0, d, 2, dtype=np.float32) / d))
    f = (pos.astype(np.float32)[:, :, None] * inv[None, None, :]).reshape(pos.shape[0], -1)   # [N, 32] (h freqs, w freqs)
    emb = np.concatenate([f, f], -1)                                     # [N, 64]
    return jnp.asarray(np.cos(emb)), jnp.asarray(np.sin(emb))


def _rot_half(x):
    a, b = jnp.split(x, 2, axis=-1)
    return jnp.concatenate([-b, a], -1)


def _attention(p, x, cos, sin, same, n_heads, dtype):
    N, C = x.shape
    hd = C // n_heads
    qkv = (jnp.dot(x, p["qkv"].astype(dtype), preferred_element_type=jnp.float32) + p["qkv_b"]).reshape(N, 3, n_heads, hd)
    q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]
    q, k = _rms(q, p["qn"]), _rms(k, p["kn"])
    c, s = cos[:, None, :], sin[:, None, :]
    q = q * c + _rot_half(q) * s
    k = k * c + _rot_half(k) * s
    sc = jnp.einsum("nhd,mhd->hnm", q, k) * (hd ** -0.5)
    sc = jnp.where(same[None], sc, -jnp.inf)
    a = jax.nn.softmax(sc, axis=-1)
    o = jnp.einsum("hnm,mhd->nhd", a, v).reshape(N, C).astype(dtype)
    return jnp.dot(o, p["proj"].astype(dtype), preferred_element_type=jnp.float32) + p["proj_b"]


def _swiglu(g, u, limit=10.0):
    g = jnp.minimum(g, limit)
    u = jnp.clip(u, -limit, limit)
    return jax.nn.silu(g) * u


def forward(p, patches, grids, dtype=jnp.float32):
    """patches [N, 1176] (all images concatenated), grids [(t, gh, gw), ...] -> [sum n_tokens, out_hidden] f32.
    Shapes are static per call (one compile per (N, layout)); pad N to a bucket for the TPU."""
    hd = p["blocks"][0]["qn"].shape[0]                                   # static shapes (works under jit; the
    pos, seg = position_ids(grids), segment_ids(grids)                    # n_heads entry may be a traced leaf)
    same = jnp.asarray(seg[:, None] == seg[None, :])
    x = jnp.dot(jnp.asarray(patches, dtype), p["patch"].astype(dtype), preferred_element_type=jnp.float32) + p["patch_b"]
    x = x.astype(dtype)
    n_heads = x.shape[-1] // hd
    cos, sin = _rope(pos, hd)
    for b in p["blocks"]:
        x = x + _attention(b, _rms(x, b["n1"]), cos, sin, same, n_heads, dtype).astype(dtype)
        h = _rms(x, b["n2"])
        g = jnp.dot(h, b["gate"].astype(dtype), preferred_element_type=jnp.float32) + b["gate_b"]
        u = jnp.dot(h, b["up"].astype(dtype), preferred_element_type=jnp.float32) + b["up_b"]
        x = x + (jnp.dot(_swiglu(g, u).astype(dtype), b["down"].astype(dtype), preferred_element_type=jnp.float32) + b["down_b"]).astype(dtype)
    x = _rms(x, p["post_ln"])
    N, C = x.shape
    x4 = x.reshape(N // 4, 2, 2, C).transpose(0, 3, 1, 2).reshape(N // 4, C * 4)   # (c, i, j) like the conv weight
    y = jnp.dot(x4, p["ds"].astype(dtype), preferred_element_type=jnp.float32) + p["ds_b"]
    y = jnp.dot(y.astype(dtype), p["m_proj"].astype(dtype), preferred_element_type=jnp.float32)
    mu = y.mean(-1, keepdims=True)
    var = ((y - mu) ** 2).mean(-1, keepdims=True)
    y = (y - mu) * jax.lax.rsqrt(var + EPS) * p["m_ln"] + p["m_ln_b"]
    y = jax.nn.gelu(y, approximate=False).astype(dtype)
    g = jnp.dot(y, p["m_gate"].astype(dtype), preferred_element_type=jnp.float32)
    u = jnp.dot(y, p["m_up"].astype(dtype), preferred_element_type=jnp.float32)
    return jnp.dot(_swiglu(g, u).astype(dtype), p["m_down"].astype(dtype), preferred_element_type=jnp.float32)


# ----------------------------------------------------------------------------- TP-sharded forward (8 chips)
def shard_specs(p):
    """PartitionSpecs for `to_sharded(p)`: attention heads over AXIS (qkv columns / proj rows regrouped per head),
    MLP and merger intermediate columns over AXIS (down rows), the downsample columns and m_proj rows over AXIS;
    small vectors replicated. ~140 MB per chip in bf16 for the real tower."""
    from jax.sharding import PartitionSpec as P
    from glm53.engine import AXIS
    R = P()
    blk = {"n1": R, "n2": R, "qkv": P(None, None, AXIS, None), "qkv_b": P(None, AXIS, None),
           "proj": P(AXIS, None, None), "proj_b": R, "qn": R, "kn": R,
           "gate": P(None, AXIS), "gate_b": P(AXIS), "up": P(None, AXIS), "up_b": P(AXIS), "down": P(AXIS, None), "down_b": R}
    blk = {k: P(None, *v) for k, v in blk.items()}                  # blocks are STACKED on a leading axis (lax.scan)
    return {"patch": R, "patch_b": R, "blocks": blk, "post_ln": R,
            "ds": P(None, AXIS), "ds_b": P(AXIS), "m_proj": P(AXIS, None), "m_ln": R, "m_ln_b": R,
            "m_gate": P(None, AXIS), "m_up": P(None, AXIS), "m_down": P(AXIS, None), "n_heads": None}


def to_sharded(p, mesh, dtype=jnp.bfloat16):
    """Regroup the head-sharded matrices ([in, 3, H, hd] qkv / [H, hd, out] proj) and place every array on the mesh."""
    from jax.sharding import NamedSharding
    H = p["n_heads"]
    q = dict(p)
    blocks = []
    for b in p["blocks"]:
        C = b["qkv"].shape[0]
        hd = C // H
        blocks.append({**b, "qkv": np.asarray(b["qkv"]).reshape(C, 3, H, hd), "qkv_b": np.asarray(b["qkv_b"]).reshape(3, H, hd),
                       "proj": np.asarray(b["proj"]).reshape(H, hd, C)})
    q["blocks"] = {k: np.stack([np.asarray(b[k]) for b in blocks]) for k in blocks[0]}   # [depth, ...] for lax.scan
    specs = shard_specs(p)

    def place(a, sp):
        if sp is None or a is None:
            return a
        a = np.asarray(a)
        a = a.astype(np.float32) if a.dtype == np.float32 else a
        return jax.device_put(jnp.asarray(a, dtype if a.ndim >= 2 else jnp.float32), NamedSharding(mesh, sp))
    out = jax.tree.map(place, q, specs, is_leaf=lambda x: x is None or not isinstance(x, (dict, list)))
    out["n_heads"] = H
    return out


def make_forward_sharded(p_sharded, mesh, dtype=jnp.bfloat16, q_chunk=1024):
    """-> f(patches [N,1176] np, grids tuple) -> [n_tokens, out] f32, one jit per (N, grids). Per chip: its heads
    (attention scores chunked over q_chunk queries), its MLP/merger columns; two psums per block. The 24 blocks run
    as a `lax.scan` over the stacked parameters (one block body per executable: TPU executables live in HBM, and
    five unrolled buckets held ~0.4 GB)."""
    from glm53.engine import AXIS, shard_map
    from jax import lax
    from jax.sharding import PartitionSpec as P
    R = P()
    n_heads = p_sharded["n_heads"]
    specs = shard_specs(p_sharded)
    del specs["n_heads"]
    pp = {k: v for k, v in p_sharded.items() if k != "n_heads"}
    n_dev = mesh.devices.size
    Hl = n_heads // n_dev
    cache = {}

    def build(N):
        """One program per patch count N: positions / segments arrive as arrays, so images of any grid share it."""
        def attn(b, x, cos, sin, same):
            C = x.shape[-1]
            hd = C // n_heads
            qkv = jnp.einsum("nc,cthd->nthd", x, b["qkv"].astype(dtype), preferred_element_type=jnp.float32) + b["qkv_b"]
            q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]                                 # [N, Hl, hd]
            q, k = _rms(q, b["qn"]), _rms(k, b["kn"])
            c, s_ = cos[:, None, :], sin[:, None, :]
            q = q * c + _rot_half(q) * s_
            k = k * c + _rot_half(k) * s_
            outs = []
            for a in range(0, N, q_chunk):
                sc = jnp.einsum("nhd,mhd->hnm", q[a:a + q_chunk], k) * (hd ** -0.5)
                sc = jnp.where(same[a:a + q_chunk][None], sc, -jnp.inf)
                outs.append(jnp.einsum("hnm,mhd->nhd", jax.nn.softmax(sc, axis=-1), v))
            o = jnp.concatenate(outs, 0).astype(dtype)                                # [N, Hl, hd]
            y = jnp.einsum("nhd,hdc->nc", o, b["proj"].astype(dtype), preferred_element_type=jnp.float32)
            return lax.psum(y, AXIS) + b["proj_b"]

        def prog(p, patches, cos, sin, same):
            x = jnp.dot(patches.astype(dtype), p["patch"].astype(dtype), preferred_element_type=jnp.float32) + p["patch_b"]
            x = x.astype(dtype)

            def block(x, b):
                x = x + attn(b, _rms(x, b["n1"]), cos, sin, same).astype(dtype)
                h = _rms(x, b["n2"])
                g = jnp.dot(h, b["gate"].astype(dtype), preferred_element_type=jnp.float32) + b["gate_b"]
                u = jnp.dot(h, b["up"].astype(dtype), preferred_element_type=jnp.float32) + b["up_b"]
                y = jnp.dot(_swiglu(g, u).astype(dtype), b["down"].astype(dtype), preferred_element_type=jnp.float32)
                return x + (lax.psum(y, AXIS) + b["down_b"]).astype(dtype), None
            x, _ = lax.scan(block, x, p["blocks"])
            x = _rms(x, p["post_ln"])
            C = x.shape[-1]
            x4 = x.reshape(N // 4, 2, 2, C).transpose(0, 3, 1, 2).reshape(N // 4, C * 4)
            y = jnp.dot(x4, p["ds"].astype(dtype), preferred_element_type=jnp.float32) + p["ds_b"]   # local columns
            y = lax.psum(jnp.dot(y.astype(dtype), p["m_proj"].astype(dtype), preferred_element_type=jnp.float32), AXIS)
            mu = y.mean(-1, keepdims=True)
            var = ((y - mu) ** 2).mean(-1, keepdims=True)
            y = (y - mu) * lax.rsqrt(var + EPS) * p["m_ln"] + p["m_ln_b"]
            y = jax.nn.gelu(y, approximate=False).astype(dtype)
            g = jnp.dot(y, p["m_gate"].astype(dtype), preferred_element_type=jnp.float32)
            u = jnp.dot(y, p["m_up"].astype(dtype), preferred_element_type=jnp.float32)
            return lax.psum(jnp.dot(_swiglu(g, u).astype(dtype), p["m_down"].astype(dtype), preferred_element_type=jnp.float32), AXIS)
        return jax.jit(shard_map(prog, mesh=mesh, in_specs=(specs, R, R, R, R), out_specs=R, check_vma=False))

    hd = pp["blocks"]["qn"].shape[-1]

    def run(patches, grids):
        grids = tuple(tuple(int(v) for v in g) for g in grids)
        N = patches.shape[0]
        if N not in cache:
            cache[N] = build(N)
        pos, seg = position_ids(grids), segment_ids(grids)
        cos, sin = _rope(pos, hd)
        same = jnp.asarray(seg[:, None] == seg[None, :])
        return cache[N](pp, jnp.asarray(patches, dtype), cos, sin, same)
    return run
'''
for path, src in FILES.items():
    open(path, "w").write(src)
print(f"wrote {len(FILES)} files of the glm53 package")


### The serving script
The next cell writes the serving script (the same one `launch.py` pushes; the source and docs are in the [kaggle-tpu-lab repo](https://github.com/ARahim3/kaggle-tpu-lab)).


In [ ]:
%%writefile serve_glm53.py
"""
Serve GLM-5.3-Flash (320B MoE, 18B active) on a Kaggle TPU v5e-8 with our own JAX engine.

This script is pushed to Kaggle as a script kernel by ../../launch.py, which fills in the CFG line below and embeds
the engine package. It also runs standalone (pasted into a Kaggle notebook next to a `glm53/` folder written by the
previous cell) — then it prints instead of using ntfy.

Steps (each one is announced in the log):
  1/6  runtime  — pre-flight (datasets attached, Internet on, a real TPU: ~20 s, before anything slow), pinned libtpu
                  + a few pip packages (~1 min), cloudflared, the engine package
  2/6  weights  — the routed experts (3-bit codebook tables, Unsloth's UD-IQ3_XXS GGUF) and the non-expert weights
                  (int8) straight onto the eight chips (~6 min); the non-expert weights come from the serve dataset's
                  base store when it is attached, else from the FP8 checkpoint datasets
  3/6  vision   — the vision tower, sharded over the chips
  4/6  warm-up  — the prefill buckets, the batched decode programs and the snapshot programs (~9 min with the serve
                  dataset's compile cache, ~14 min cold: the rest is JAX tracing, which no cache skips)
  5/6  tunnel   — a public cloudflared URL (three attempts; a URL that never resolves is replaced once)
  6/6  ready    — READY banner + self-test, then keep serving until keepalive_min elapses

A cold run leaves `jax_cache/` (everything it compiled) and `base/` (the non-expert weights, vision tower and
tokenizer, ~11 GB) in /kaggle/working, so its output can be turned into the serve dataset ("New dataset" from the
kernel output) that later runs attach instead of the FP8 datasets.
"""
import base64, collections, glob, hashlib, io, json, os, queue, re, secrets, shutil, subprocess, sys, tarfile, threading, time, urllib.request, uuid
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

CFG = None  # __LAUNCHER_CONFIG__  (launch.py replaces this line)

DEFAULTS = {
    "expert_datasets": ["rahim3/glm53-flash-iq3xxs-1", "rahim3/glm53-flash-iq3xxs-2"],   # Unsloth UD-IQ3_XXS GGUF, mirrored
    "base_datasets": ["rahim3/glm53-flash-fp8-1", "rahim3/glm53-flash-fp8-2",            # HF FP8 checkpoint, mirrored:
                      "rahim3/glm53-flash-fp8-3", "rahim3/glm53-flash-fp8-4"],           # non-expert weights, vision, tokenizer
    "serve_dataset": "rahim3/glm53-flash-serve",   # base/ (non-expert weights, vision, tokenizer) + jax_cache/ (compiled
                                     # programs); attached: the FP8 datasets are not needed and the warm-up is ~9 min instead of ~14
    "dump_base": True,               # a cold run writes base/ into its output (so the serve dataset can be made from it)
    "libtpu": "0.0.42.*",            # the image's runtime is 140x slower on gathers and cannot run Pallas kernels
    "max_len": 262144,               # context capacity (tokens); a multiple of 32
    "streams": 4,                    # requests decoded together (one program set per batch size, ~2.5 min compile each)
    "sets": 4,                       # cache sets on the chips: running streams + finished contexts kept for their next turn
    "piece": 512,                    # prefill piece (tokens): 512 keeps the prefill temporaries small enough for three
    "sched_piece": 512,              # 262k streams next to the engine (1024 is ~13 % faster prefill but fits two or three)
    "q_block": 32,                   # queries per attention block during prefill
    "cache_q8": True,                # int8 latent cache (a 262k set costs ~0.2 GB/chip instead of 0.3)
    "bucket_min": 256,               # smallest prefill bucket compiled (short prompts pad to it)
    "min_free_gb": 0.55,             # HBM headroom an admission needs (a cache set + the prefill temporaries of one piece)
    "max_queue": 8,                  # waiting requests beyond the streams before a 429
    "max_wait_s": 90.0,              # a request waiting longer than this gets a 503 (clients retry)
    "keepalive_s": 15,               # SSE ping when nothing was streamed for this long (a buffered tool call, a queued
                                     # request): Cloudflare drops a response that is silent for ~100 s
    "think_budget_default": 0,       # thinking tokens before </think> is forced, when the request sets no budget (0 = unlimited)
    "vision": True,                  # load the vision tower (images in both APIs); False saves ~1 min and 0.14 GB/chip
    "vision_max_tokens": 1024,       # 28x28-pixel tokens per image
    "reasoning_effort_default": "low",   # server-side default: low | high (anything else = the template's Max)
    "temperature": 1.0, "top_p": 0.95,   # generation_config defaults
    "max_new_default": 4096,
    "snap_host_gb": 48,              # host RAM for parked contexts (agent sessions that interleave)
    "snap_rows": 1024,               # snapshot row bucket (1024 rows = 8192 tokens sharded)
    "snap_min": 256,                 # contexts shorter than this are re-prefilled instead of parked
    "base_min": 512,                 # a system section at least this long is pinned for new sessions
    "snap_warm_tokens": 32768,       # warm the snapshot programs for contexts up to this many tokens
    "keepalive_min": 480,            # auto-shutdown guard (Kaggle TPU caps at 9h anyway)
    "api_key": "",                   # generated if empty
    "ntfy_topic": "",                # optional: publish progress to ntfy.sh/<topic> (launch.py watches it)
    "served_model_name": "glm-5.3-flash",
    "tunnel": True,                  # False: no cloudflared (local testing)
    "skip_runtime": False,           # True: no pip installs (local testing)
    "port": 8000,
}
CFG = {**DEFAULTS, **(CFG or {}), **globals().get("CFG_PRESET", {})}
_cfg_file = Path("serve_config.json")            # notebook flow: overrides next to this script
if _cfg_file.exists():
    CFG.update(json.loads(_cfg_file.read_text()))
if not CFG["api_key"]:
    CFG["api_key"] = "glm-" + secrets.token_hex(12)

ENGINE_B64 = ""  # __ENGINE__  (launch.py embeds the glm53 package here; the notebook writes the files instead)

PORT = int(CFG["port"])
WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/tmp")
CACHE_DIR = WORK / "jax_cache"
CLOUDFLARED = Path("/tmp/cloudflared")
T0 = time.time()
LOG_LINES = []


def log(*parts):
    line = time.strftime("[%H:%M:%S] ") + " ".join(str(p) for p in parts)
    LOG_LINES.append(line)
    print(line, flush=True)


def elapsed():
    return f"{int(time.time() - T0) // 60} min {int(time.time() - T0) % 60:02d} s"


def banner(step, title, note=""):
    log("")
    log("=" * 70)
    log(f" STEP {step}/6  {title}" + (f"   ({note})" if note else "") + f"   [{elapsed()} so far]")
    log("=" * 70)


def publish(phase, **extra):
    """Progress event: always logged; also pushed to ntfy if a topic is set."""
    log(f"PHASE {phase}", json.dumps(extra) if extra else "")
    if not CFG["ntfy_topic"]:
        return
    try:
        body = {"topic": CFG["ntfy_topic"], "title": f"kaggle-tpu-lab {phase}", "message": json.dumps({"phase": phase, **extra})}
        req = urllib.request.Request("https://ntfy.sh", data=json.dumps(body).encode(), headers={"Content-Type": "application/json"})
        urllib.request.urlopen(req, timeout=10)
    except Exception as e:  # noqa: BLE001
        log(f"(ntfy publish failed: {e})")


def sh(cmd, tag):
    t = time.time()
    r = subprocess.run(cmd, capture_output=True, text=True)
    log(f"   {tag}: rc {r.returncode} in {time.time() - t:.0f}s" + (f" | {(r.stderr or '')[-200:].strip()}" if r.returncode else ""))
    return r.returncode


def mounts_of(names):
    """Dataset names ('owner/slug') -> mounted paths, in the given order (missing ones dropped)."""
    out = []
    for n in names:
        slug = n.split("/")[-1]
        hits = glob.glob(f"/kaggle/input/datasets/{n}") + glob.glob(f"/kaggle/input/{slug}")
        if hits:
            out.append(hits[0])
    return out


def hbm():
    st = jax.devices()[0].memory_stats() or {}
    return st.get("bytes_in_use", 0) / 1e9, (st.get("bytes_limit") or 0) / 1e9


def fail(step, msg):
    """Stop with a plain message (the launcher prints `step` and `tail` of a "failed" phase)."""
    log("   " + msg)
    publish("failed", step=step, tail=msg)
    sys.exit(1)


def preflight():
    """Look before the slow steps (~20 s): the datasets attached, Internet on (pip, cloudflared) and a real TPU present.
    Kaggle sometimes starts a "TPU" session with no TPU (a CPU-only container, most often on new or not-yet-verified
    accounts): jax then sees one CPU device and the build dies minutes later with a sharding error."""
    need = list(CFG["expert_datasets"]) + ([] if CFG["serve_dataset"] and mounts_of([CFG["serve_dataset"]]) else
                                           ([CFG["serve_dataset"]] if CFG["serve_dataset"] and not mounts_of(CFG["base_datasets"]) else list(CFG["base_datasets"])))
    missing = [n for n in need if not mounts_of([n])]
    if missing:
        fail("datasets", f"datasets not attached: {missing}. In the right sidebar, Add Input -> search each name -> attach, then run again.")
    try:
        urllib.request.urlopen("https://pypi.org/simple/pip/", timeout=20).read(1)
    except Exception as e:  # noqa: BLE001
        fail("no-internet", f"no Internet from this session ({str(e)[:120]}): Session options -> Internet ON (a phone-verified "
                            "Kaggle account is needed for that), then run again. The pip packages and the tunnel need it.")
    code = ("import jax\n"
            "try:\n"
            "    d = jax.devices()\n"
            "    print('TPU_CHECK', len(d), d[0].platform, getattr(d[0], 'device_kind', ''))\n"
            "except Exception as e:\n"
            "    print('TPU_CHECK 0 none', str(e).replace(chr(10), ' ')[:200])\n")
    try:
        r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True, timeout=180)
        m = re.search(r"TPU_CHECK (\d+) (\S+)(.*)", (r.stdout or "") + (r.stderr or ""))
    except Exception as e:  # noqa: BLE001
        log(f"   (TPU check skipped: {e})"); return
    if m is None:
        log("   (TPU check inconclusive: the image's jax did not answer; continuing)"); return
    n, platform, rest = int(m.group(1)), m.group(2), m.group(3).strip()
    if n == 8 and platform == "tpu":
        log(f"   pre-flight OK: datasets attached, Internet on, 8 TPU chips ({rest})"); return
    if n == 0 and not re.search(r"jellyfish|TPU initialization failed|initialize backend 'tpu'|No TPU|vfio", rest, re.I):
        log(f"   (TPU check inconclusive, continuing: {rest[:160]})"); return
    fail("no-tpu", f"this session has no working TPU: jax sees {n} {platform} device(s) {rest}. Kaggle sometimes starts a TPU "
                   "session without one (most often on new or not-yet-verified accounts); nothing in this notebook can fix "
                   "that. Stop the session and start it again; `import jax; print(jax.device_count())` in a fresh cell must "
                   "print 8 before this script is worth running.")


# ----------------------------------------------------------------------------- 1. runtime
if not CFG["skip_runtime"]:
    banner(1, "Runtime", "pre-flight, pinned libtpu + pip packages, cloudflared, the engine")
    preflight()
    sh([sys.executable, "-m", "pip", "install", "-q", "safetensors", "huggingface_hub", "transformers>=5.16", "pillow",
        "torch", "--index-url", "https://download.pytorch.org/whl/cpu", "--extra-index-url", "https://pypi.org/simple"], "pip packages")
    if CFG["libtpu"]:
        sh([sys.executable, "-m", "pip", "install", "-q", f"libtpu=={CFG['libtpu']}"], f"libtpu=={CFG['libtpu']}")
    if ENGINE_B64 and not ENGINE_B64.startswith("__"):
        with tarfile.open(fileobj=io.BytesIO(base64.b64decode(ENGINE_B64)), mode="r:gz") as tf:
            tf.extractall(WORK)
        sys.path.insert(0, str(WORK))
        log(f"   engine package extracted to {WORK}/glm53")
    elif not Path("glm53").is_dir():
        sys.exit("no glm53/ package next to this script and nothing embedded — run the notebook's engine cell first")

if CFG["tunnel"] and not CLOUDFLARED.exists():
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CLOUDFLARED)
    CLOUDFLARED.chmod(0o755)
    log("   cloudflared downloaded")

import numpy as np                      # noqa: E402  (after the runtime pins: libtpu must be installed before jax loads)
import jax, jax.numpy as jnp            # noqa: E402
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P   # noqa: E402

SERVE_MOUNT = (mounts_of([CFG["serve_dataset"]]) or [None])[0] if CFG["serve_dataset"] else None
BASE_DIR = os.path.join(SERVE_MOUNT, "base") if SERVE_MOUNT and os.path.isdir(os.path.join(SERVE_MOUNT, "base")) else None
if "eng" not in globals():              # (a test harness may pre-set eng/tok/ids/eos/cfg and skip the build)
    cache_src = os.path.join(SERVE_MOUNT, "jax_cache") if SERVE_MOUNT and os.path.isdir(os.path.join(SERVE_MOUNT, "jax_cache")) else None
    if cache_src and not CACHE_DIR.exists():
        t = time.time()
        shutil.copytree(cache_src, CACHE_DIR)
        log(f"   compile cache restored from {cache_src} in {time.time() - t:.0f}s")
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    jax.config.update("jax_compilation_cache_dir", str(CACHE_DIR))
    try:                                 # a process that already initialised the cache elsewhere keeps writing there
        from jax._src import compilation_cache as _cc
        _cc.reset_cache()
    except Exception as e:  # noqa: BLE001
        log(f"   (compile cache reset skipped: {e!r})")
    log(f"   jax {jax.__version__}, {jax.device_count()} devices, compile cache at {CACHE_DIR}")

# ----------------------------------------------------------------------------- 2. weights -> the chips
if "eng" not in globals():
    banner(2, "Weights", "3-bit codebook experts + int8 non-expert weights onto the eight chips, ~9 min")
    from glm53 import basestore as BS
    from glm53 import checkpoint as C
    from glm53 import model as M
    from glm53.engine import AXIS
    from glm53 import gguf_reader as G
    from glm53.resident import ResidentFetch, ResidentLayerEngine, pack_layer_from_gguf, hbm_bytes
    from transformers import AutoConfig, AutoTokenizer
    gguf_mounts = mounts_of(CFG["expert_datasets"])
    fp8_mounts = [] if BASE_DIR else mounts_of(CFG["base_datasets"])
    if len(gguf_mounts) < len(CFG["expert_datasets"]) or (not BASE_DIR and len(fp8_mounts) < len(CFG["base_datasets"])):
        fail("datasets", f"datasets missing: found gguf {gguf_mounts}, fp8 {fp8_mounts}, serve {SERVE_MOUNT}; attach "
                         f"{CFG['expert_datasets']} + {CFG['serve_dataset'] or CFG['base_datasets']}")
    hf_dir = BASE_DIR or fp8_mounts[0]
    log(f"   experts from {gguf_mounts}; the rest from {'the base store ' + BASE_DIR if BASE_DIR else 'the FP8 checkpoint'}")
    publish("loading", note="weights")
    hf_cfg = AutoConfig.from_pretrained(hf_dir)
    tc = hf_cfg.text_config
    N_LAYERS = tc.num_hidden_layers
    cfg = M.Cfg.from_hf(tc, dtype=jnp.bfloat16)
    N_DEV = jax.device_count()
    mesh = Mesh(np.array(jax.devices()), (AXIS,))
    tbl_sh = NamedSharding(mesh, P(AXIS))
    gm = G.GGUFModel(G.open_mirror(gguf_mounts))
    t_load = time.time()
    if BASE_DIR:
        store = BS.load(BASE_DIR)
        params = {**store["top"], "layers": []}
        log(f"   base store read in {time.time() - t_load:.0f}s ({store['manifest'].get('n_layers')} layers)")
        r = None
    else:
        r = C.RawShardReader(":".join(fp8_mounts))
        top = C.load_top(r, np.float32)
        params = {"embed": top["embed"].astype(jnp.bfloat16), "norm": top["norm"], "lm_head": top["lm_head"].astype(jnp.bfloat16), "layers": []}
        del top
    qtypes, hbm_total = {}, 0
    for i in range(N_LAYERS):
        ti = time.time()
        if BASE_DIR:
            p = store["layers"][i]
        else:
            p = C.load_layer(r, i, tc.layer_types[i], tc.mlp_layer_types[i], np.float32)
            p = jax.tree.map(lambda a: a.astype(jnp.bfloat16) if a.ndim >= 2 else a, p)
        if tc.mlp_layer_types[i] == "sparse":
            tables, qt = pack_layer_from_gguf(gm, i, N_DEV, threads=16)
            qtypes[i] = qt
            hbm_total += hbm_bytes(tables) * N_DEV
            for k, t in tables.items():                    # each table straight onto the chips: host RAM holds one layer
                p["mlp"][k] = jax.tree.map(lambda a: jax.device_put(a, tbl_sh), t)
            jax.block_until_ready([p["mlp"][k] for k in tables])
            del tables
        params["layers"].append(p)
        if r is not None:
            for f in C.layer_shards(":".join(fp8_mounts), i):
                r.release(f)
        if i % 5 == 0 or i == N_LAYERS - 1:
            log(f"   layer {i + 1}/{N_LAYERS} ({time.time() - ti:.0f}s) | HBM chip0 {hbm()[0]:.2f} GB")
    log(f"   expert tables {hbm_total / 1e9:.1f} GB across {N_DEV} chips; weights read in {(time.time() - t_load) / 60:.1f} min")
    fetch = ResidentFetch(qtypes, tc.hidden_size, tc.moe_intermediate_size // N_DEV, out_dtype=jnp.bfloat16, gather_max_rows=64, chunk=8,
                          use_pallas=True, sweep_mode="ragged", tm=32, combine="matmul")
    eng = ResidentLayerEngine(cfg, params, fetch, max_len=CFG["max_len"], layers_per_program=4, layers_per_program_prefill=1,
                              int8_nonexpert=False if BASE_DIR else "all", seq_shard=True, q_block=CFG["q_block"],
                              prefill_piece=CFG["piece"], cache_q8=CFG["cache_q8"])
    del params
    use, lim = hbm()
    log(f"   ENGINE READY in {(time.time() - t_load) / 60:.1f} min: HBM chip0 {use:.2f} / {lim:.2f} GB, context {CFG['max_len']}")
    publish("loaded", hbm_gb=round(use, 2), minutes=round((time.time() - t_load) / 60, 1))
    tok = AutoTokenizer.from_pretrained(hf_dir)
    ids = np.asarray(tok(tok.apply_chat_template([{"role": "user", "content": "Write a haiku about tensor processing units."}],
                                                 tokenize=False, add_generation_prompt=True, reasoning_effort="low"),
                         add_special_tokens=False).input_ids).reshape(1, -1)
    eos = set(json.load(open(os.path.join(hf_dir, "config.json")))["text_config"].get("eos_token_id", [tok.eos_token_id]))

# ----------------------------------------------------------------------------- 3. vision tower
VISION, VISION_FWD = globals().get("VISION"), globals().get("VISION_FWD")
_vp = None
if CFG["vision"] and VISION_FWD is None and "hf_dir" in globals():
    banner(3, "Vision", "the vision tower sharded over the chips")
    from glm53 import vision as VIS
    t = time.time()
    if BASE_DIR and store.get("vision") is not None:
        _vp = store["vision"]
    else:
        _vp = VIS.load_vision(C.RawShardReader(":".join(fp8_mounts)), dtype=np.float32)
        _vp = jax.tree.map(lambda a: a.astype(jnp.bfloat16) if getattr(a, "dtype", None) == np.float32 else a, _vp)
    VISION = VIS.to_sharded(_vp, eng.mesh, dtype=jnp.bfloat16)
    VISION_FWD = VIS.make_forward_sharded(VISION, eng.mesh, dtype=jnp.bfloat16, q_chunk=1024)
    from PIL import Image
    for target in [b for b in (256, 512, 1024, 2048, 4096) if b <= 4 * CFG["vision_max_tokens"]]:
        side = int(target ** 0.5) * 14 - 14
        img = Image.fromarray(np.random.default_rng(0).integers(0, 256, (side, side, 3), dtype=np.uint8))
        patches, grid = VIS.preprocess(img, max_tokens=CFG["vision_max_tokens"])
        n = patches.shape[0]
        bucket = max(256, 1 << (n - 1).bit_length())
        grids = (grid,) if bucket == n else (grid, (1, 2, (bucket - n) // 2))
        if bucket > n:
            patches = np.concatenate([patches, np.zeros((bucket - n, patches.shape[1]), patches.dtype)], 0)
        jax.block_until_ready(VISION_FWD(patches, grids))
    log(f"   vision tower ready in {time.time() - t:.0f}s; HBM chip0 {hbm()[0]:.2f} GB")
if CFG["dump_base"] and not BASE_DIR and "hf_dir" in globals():
    t = time.time()
    BS.dump(str(WORK / "base"), eng.params, vision=_vp, hf_dir=hf_dir, log=log,
            meta={"model": CFG["served_model_name"], "int8_nonexpert": "all", "experts": "resident (not stored)"})
    log(f"   base store dumped in {time.time() - t:.0f}s (this run's output can become the serve dataset)")
del _vp

# ----------------------------------------------------------------------------- 4. the server
from glm53.resident import ResidentLayerEngine as _RLE   # noqa: E402
from glm53.engine import DeviceSampler                    # noqa: E402
from glm53.scheduler import Request, Scheduler, SnapStore # noqa: E402
from glm53 import vision as VIS                           # noqa: E402

API_KEY = CFG["api_key"]
MAX_NEW_DEFAULT = CFG["max_new_default"]
KEEPALIVE_S = float(CFG["keepalive_s"] or 0)
THINK_BUDGET_DEFAULT = int(CFG["think_budget_default"] or 0)
MAX_STREAMS, MAX_SETS = CFG["streams"], CFG["sets"]
SCHED_PIECE = CFG["sched_piece"]
MIN_FREE_GB, MAX_WAIT_S, MAX_QUEUE = CFG["min_free_gb"], CFG["max_wait_s"], CFG["max_queue"]
VISION_MAX_TOKENS = CFG["vision_max_tokens"]
DEFAULT_EFFORT = CFG["reasoning_effort_default"]
DEFAULT_TEMP, DEFAULT_TOP_P = CFG["temperature"], CFG["top_p"]
MODEL = CFG["served_model_name"]
BUCKET_MIN = CFG["bucket_min"]
if BUCKET_MIN:
    eng.PREFILL_BUCKETS = tuple(b for b in type(eng).PREFILL_BUCKETS if b >= BUCKET_MIN)
STATE = {"url": None, "requests": 0, "tokens": 0, "prefix_hits": 0, "prefix_tokens_reused": 0, "prefill_s": 0.0,
         "snap_hits": 0, "snap_parks": 0, "snap_pins": 0, "snap_entries": 0, "snap_bytes": 0, "snap_s": 0.0,
         "steps": 0, "step_tokens": 0}
T = {k: tok.convert_tokens_to_ids(k) for k in ("<think>", "</think>", "<tool_call>", "</tool_call>", "<arg_key>", "</arg_key>",
                                                "<arg_value>", "</arg_value>", "<|user|>", "<|observation|>", "<|assistant|>",
                                                "<|image|>", "<|begin_of_image|>", "<|end_of_image|>")}
STOP_IDS = set(eos) | {T["<|user|>"], T["<|observation|>"]}

# ---- images
IMG_CACHE = collections.OrderedDict()                  # sha256 -> (n_tokens, embeddings f32 [n, D]); LRU
IMG_CACHE_MAX = 64


def embed_image(img_bytes):
    """Image bytes -> (n_tokens, embeddings [n, D] f32) through the vision tower; cached by content hash. The patch
    count is padded to a power of two with a dummy image segment (its rows are dropped) so few shapes compile."""
    from PIL import Image
    if VISION_FWD is None:
        raise ValueError("this server has no vision tower loaded (config: vision)")
    h = hashlib.sha256(img_bytes).hexdigest()
    if h in IMG_CACHE:
        IMG_CACHE.move_to_end(h)
        return IMG_CACHE[h]
    patches, grid = VIS.preprocess(Image.open(io.BytesIO(img_bytes)), max_tokens=VISION_MAX_TOKENS)
    n = patches.shape[0]
    bucket = max(256, 1 << (n - 1).bit_length())
    grids = (grid,) if bucket == n else (grid, (1, 2, (bucket - n) // 2))
    if bucket > n:
        patches = np.concatenate([patches, np.zeros((bucket - n, patches.shape[1]), patches.dtype)], 0)
    t = time.time()
    out = np.asarray(VISION_FWD(patches, grids), np.float32)[:VIS.n_tokens(grid)]
    STATE["vision_s"] = STATE.get("vision_s", 0.0) + time.time() - t
    STATE["images"] = STATE.get("images", 0) + 1
    IMG_CACHE[h] = (out.shape[0], out)
    while len(IMG_CACHE) > IMG_CACHE_MAX:
        IMG_CACHE.popitem(last=False)
    return IMG_CACHE[h]


def _image_bytes(block):
    """Anthropic image block or OpenAI image_url part -> raw bytes (base64 or data: URL inline; http(s) fetched)."""
    if block.get("type") == "image":
        src = block.get("source", {})
        if src.get("type") == "base64":
            return base64.b64decode(src["data"])
        url = src.get("url", "")
    else:
        u = block.get("image_url", "")
        url = u.get("url", "") if isinstance(u, dict) else u
    if url.startswith("data:"):
        return base64.b64decode(url.split(",", 1)[1])
    if url.startswith("http://") or url.startswith("https://"):
        return urllib.request.urlopen(url, timeout=30).read()
    raise ValueError("unsupported image source")


def _img_sig(img_bytes):
    """Negative pseudo token id identifying an image in prompt signatures (all of its tokens carry it)."""
    return -(1 + int(hashlib.sha256(img_bytes).hexdigest()[:8], 16) % (1 << 30))


def _dec(ids):
    return tok.decode([T["<|image|>"] if int(i) < 0 else int(i) for i in ids])


def _feed(sig, imgs):
    """Signature ids (image tokens negative) -> (real ids int32 [n], embeds (idx, vec) or None) for the engine."""
    sig_a, off = sig
    ids = np.where(sig_a < 0, T["<|image|>"], sig_a).astype(np.int32)
    idx = np.nonzero(sig_a < 0)[0]
    if len(idx) == 0:
        return ids, None
    vec = np.concatenate([imgs[int(sig_a[i])][1][off[i]:off[i] + 1] for i in idx], 0)
    return ids, (idx, vec)


def _run_offsets(sig):
    """Per position: index within its run of equal negative ids (0 for text)."""
    sig = np.asarray(sig)
    off = np.zeros(len(sig), np.int64)
    for i in range(1, len(sig)):
        if sig[i] < 0 and sig[i] == sig[i - 1]:
            off[i] = off[i - 1] + 1
    return off


# ---- sampling of the first token (the rest are sampled on the device)
def sample(logits, temperature, top_p, rng, n_cand=2048):
    z = np.asarray(logits, dtype=np.float32).reshape(-1)
    if temperature <= 0:
        return int(z.argmax())
    n_cand = min(n_cand, z.size - 1)
    cand = np.argpartition(-z, n_cand)[:n_cand]
    zc = z[cand] / temperature
    zc = zc - zc.max()
    p = np.exp(zc); p /= p.sum()
    if top_p < 1.0:
        order = np.argsort(-p); cum = np.cumsum(p[order])
        keep = order[: max(1, int((cum <= top_p).sum()) + 1)]
        q = np.zeros_like(p); q[keep] = p[keep]; p = q / q.sum()
    return int(cand[rng.choice(n_cand, p=p)])


BASE_MIN, SNAP_HOST_GB = CFG["base_min"], CFG["snap_host_gb"]
SNAP_ROWS, SNAP_MIN, SNAP_WARM = CFG["snap_rows"], CFG["snap_min"], CFG["snap_warm_tokens"]


def system_end(prompt):
    """Index of the first <|user|> token = end of the rendered system section (system prompt + tools)."""
    try:
        return prompt.index(T["<|user|>"])
    except ValueError:
        return 0


def _match_len(live_ids, pos, prompt, quiet=False):
    """Longest reuse of the live context (ids[:pos]) for `prompt`: (k, fed) where prompt[k:] must still be prefilled and
    `fed` = the ids the engine will have seen after that, or (0, None). Handles a dropped thinking block (the template
    renders `<think></think>` where the live context holds the reasoning) and re-tokenisation drift near the boundary."""
    THINK, END = T["<|assistant|>"], T["</think>"]
    i = j = 0
    while True:
        n = min(pos - i, len(prompt) - j)
        if n > 0:
            neq = np.asarray(live_ids[i:i + n]) != np.asarray(prompt[j:j + n])
            d = int(np.argmax(neq)) if neq.any() else n
            i += d; j += d
        if i >= pos:
            return (j, list(live_ids[:pos]) + list(prompt[j:])) if j <= len(prompt) else (0, None)
        if j >= len(prompt):
            return 0, None
        if i > 0 and live_ids[i - 1] == T["<think>"] and prompt[j] == END and END in live_ids[i:pos]:
            i = live_ids.index(END, i) + 1; j += 1
            continue
        if pos - i > 256:
            return 0, None
        a = max(0, i - 32); ja = j - (i - a)
        t_live = _dec(live_ids[a:pos])
        for k in range(max(ja + 1, j - 24), min(len(prompt), j + (pos - i) + 24) + 1):
            if _dec(prompt[ja:k]) == t_live:
                return k, list(live_ids[:pos]) + list(prompt[k:])
        return 0, None


RNG = np.random.default_rng()


def sched_feed(req, a, b):
    return _feed((req.sig[a:b], req.off[a:b]), req.imgs or {})


_drop = [k for k in eng._progs if isinstance(k, tuple) and k[0] == "group" and k[3] == 1 and k[5]]   # batch-1 loop programs: unused here
for k in _drop:
    del eng._progs[k]
SNAPS = SnapStore(eng, int(SNAP_HOST_GB * 1e9), SNAP_ROWS, match_len=_match_len, log=log, state=STATE)
SCHED = Scheduler(eng, STOP_IDS, MAX_STREAMS, MAX_SETS, feed=sched_feed, match_len=_match_len, system_end=system_end,
                  first_sample=lambda z, t, p: sample(z, t, p, RNG), snaps=SNAPS, log=log, state=STATE,
                  base_min=BASE_MIN, snap_min=SNAP_MIN, piece=SCHED_PIECE, min_free_gb=MIN_FREE_GB, max_wait_s=MAX_WAIT_S)


class QueueFull(Exception):
    """Too many requests in flight (-> 429)."""


class ClientGone(Exception):
    """The client closed the connection mid-response."""


_IDLE = object()


def generate(prompt, max_new, temperature, top_p, on_token=None, imgs=None, rid=None, on_idle=None, budget=None):
    """One request through the scheduler -> (out ids, prefill_s, decode_s, reused, reason) with reason "stop" | "length"
    | "stop_sequence" (`on_token` returned True: cancelled there) | "cancelled". Tokens reach `on_token` on THIS thread;
    `on_idle` is called when no token arrived for KEEPALIVE_S (a queued request); `budget` = Request.budget."""
    prompt = [int(t) for t in prompt]
    if MAX_QUEUE and len(SCHED.pending) + len(SCHED.active) >= MAX_STREAMS + MAX_QUEUE:
        raise QueueFull(f"{len(SCHED.active)} requests running and {len(SCHED.pending)} waiting; retry later")
    q = queue.Queue()
    req = Request(prompt, max_new, temperature, top_p, imgs=imgs, rid=rid, budget=budget,
                  on_token=q.put if on_token else None, on_done=(lambda: q.put(None)) if on_token else None)
    req.sig = np.asarray(prompt, np.int64)
    req.off = _run_offsets(req.sig) if imgs else np.zeros(len(prompt), np.int64)
    SCHED.submit(req)
    stopped = False
    if on_token:
        while True:
            try:
                t = q.get(timeout=KEEPALIVE_S or None)
            except queue.Empty:
                t = _IDLE
            if t is None:
                break
            if stopped:
                continue
            try:
                if t is _IDLE:
                    if on_idle is not None:
                        on_idle()
                elif on_token(t):
                    stopped = True
                    req.cancel()
            except Exception:
                req.cancel()
                raise
    req.done.wait()
    if req.error is not None:
        raise req.error
    return req.out, req.prefill_s, req.decode_s, req.reused, ("stop_sequence" if stopped else req.stop_reason)


# ---- output parsing (token level)
class StopFilter:
    """`stop_sequences` on the TEXT events: `feed(text) -> (emit, hit)` holds back a tail that could begin a stop
    sequence; at a match returns the text before it and the sequence; `flush()` releases the held tail."""

    def __init__(self, stops):
        self.stops = [s for s in (stops or []) if s]
        self.buf, self.hit = "", None

    def feed(self, text):
        if not self.stops:
            return text, None
        self.buf += text
        best = None
        for s in self.stops:
            i = self.buf.find(s)
            if i >= 0 and (best is None or i < best[0]):
                best = (i, s)
        if best is not None:
            out, self.buf, self.hit = self.buf[:best[0]], "", best[1]
            return out, best[1]
        keep = 0
        for s in self.stops:
            for k in range(min(len(s) - 1, len(self.buf)), keep, -1):
                if self.buf.endswith(s[:k]):
                    keep = k; break
        cut = len(self.buf) - keep
        out, self.buf = self.buf[:cut], self.buf[cut:]
        return out, None

    def flush(self):
        out, self.buf = self.buf, ""
        return out


def parse_tool_call(body, tools):
    """body = tokens between <tool_call> and </tool_call>."""
    name_end = body.index(T["<arg_key>"]) if T["<arg_key>"] in body else len(body)
    name = tok.decode(body[:name_end]).strip()
    schema = {}
    for t in tools or []:
        f = t.get("function", t)
        if f.get("name") == name:
            schema = (f.get("parameters") or f.get("input_schema") or {}).get("properties", {}) or {}
    args, i = {}, name_end
    while i < len(body) and T["<arg_key>"] in body[i:]:
        a = body.index(T["<arg_key>"], i) + 1
        b = body.index(T["</arg_key>"], a) if T["</arg_key>"] in body[a:] else len(body)
        key = tok.decode(body[a:b]).strip()
        c = body.index(T["<arg_value>"], b) + 1 if T["<arg_value>"] in body[b:] else len(body)
        d = body.index(T["</arg_value>"], c) if T["</arg_value>"] in body[c:] else len(body)
        raw = tok.decode(body[c:d])
        typ = schema.get(key, {}).get("type")
        if typ == "string":
            val = raw
        else:
            try:
                val = json.loads(raw)
            except Exception:  # noqa: BLE001
                val = raw
        args[key] = val
        i = d + 1
    return {"id": "call_" + uuid.uuid4().hex[:12], "name": name, "input": args}


class TokenStream:
    """Incremental token -> (kind, text/tool) events: kind in {"thinking", "text", "tool"}; partial multibyte text
    is held back until it decodes cleanly; tool-call tokens are buffered until </tool_call>."""

    def __init__(self, tools, mode="thinking"):
        self.tools, self.mode, self.buf, self.tool_buf = tools, mode, [], []

    def feed(self, t):
        if t in STOP_IDS:
            return self.flush()
        if self.mode == "tool":
            if t == T["</tool_call>"]:
                self.mode = "text"; call = parse_tool_call(self.tool_buf, self.tools); self.tool_buf = []
                return [("tool", call)]
            self.tool_buf.append(t); return []
        if t == T["</think>"] and self.mode == "thinking":
            ev = self.flush(); self.mode = "text"; return ev
        if t == T["<tool_call>"] and self.mode == "text":
            ev = self.flush(); self.mode = "tool"; return ev
        self.buf.append(t)
        piece = tok.decode(self.buf)
        if piece.endswith("�"):
            return []
        self.buf = []
        return [(self.mode, piece)] if piece else []

    def flush(self):
        if not self.buf:
            return []
        piece = tok.decode(self.buf); self.buf = []
        return [(self.mode, piece)] if piece else []


THINK_WRAP = tok("\n\nMy thinking budget is used up, so I will give the answer directly now.\n", add_special_tokens=False).input_ids + [T["</think>"]]


def run_request(ids, max_new, temperature, top_p, imgs, rid, tools, stops=None, prefix=(), on_event=None, raw=False,
                think_budget=None):
    """One request end to end: `prefix` ids (a forced tool call) are fed to the parser first, then the generated tokens;
    events (kind, value) go to `on_event` as they are parsed (stop sequences applied to the text), plus ("ping", None)
    when nothing was emitted for KEEPALIVE_S (a tool call is buffered until it closes; a queued request waits): the
    tunnel drops a response that stays silent for ~100 s. `think_budget`: thinking tokens after which `</think>` is
    forced (a short wrap-up line first) so the answer always comes. Returns
    {reasoning, text, calls, out_len, stop, stop_sequence, tp, tg, reused}."""
    ts, sf = TokenStream(tools, "text" if raw else "thinking"), StopFilter(stops)
    res = {"reasoning": "", "text": "", "calls": [], "n": 0}
    last = [time.time()]

    def tick():
        if on_event is not None and KEEPALIVE_S and time.time() - last[0] >= KEEPALIVE_S:
            last[0] = time.time()
            on_event("ping", None)

    def emit(kind, val):
        last[0] = time.time()
        if kind == "thinking":
            res["reasoning"] += val
        elif kind == "text":
            res["text"] += val
        else:
            res["calls"].append(val)
        if on_event is not None:
            on_event(kind, val)

    def handle(t):
        for kind, val in ts.feed(t):
            if kind == "text":
                out, hit = sf.feed(val)
                if out:
                    emit("text", out)
                if hit is not None:
                    return True
            else:
                held = sf.flush()
                if held:
                    emit("text", held)
                emit(kind, val)
        return False

    for t in prefix:
        handle(t)

    def on_token(t):
        res["n"] += 1
        if handle(t):
            return True
        tick()
        return False
    full = list(ids) + list(prefix)
    budget = (int(think_budget), T["</think>"], THINK_WRAP) if think_budget and not raw and full and full[-1] == T["<think>"] else None
    try:
        out, tp, tg, reused, reason = generate(full, max_new, temperature, top_p, on_token, imgs, rid, on_idle=tick, budget=budget)
    except ClientGone as e:
        e.n = res["n"]
        raise
    if reason != "stop_sequence":
        held = sf.flush()
        if held:
            emit("text", held)
    return {"reasoning": res["reasoning"], "text": res["text"], "calls": res["calls"], "out_len": res["n"],
            "stop": reason, "stop_sequence": sf.hit, "tp": tp, "tg": tg, "reused": reused}


def forced_prefix(choice):
    """tool_choice -> generation prefix ids: "any" forces a tool call, a name forces that tool (the template's own
    rendering of a tool turn: `<think></think><tool_call>name...`, so the next turn's prompt matches the live context)."""
    if choice in (None, "none"):
        return []
    ids = [T["</think>"], T["<tool_call>"]]
    if choice != "any":
        ids += tok(choice, add_special_tokens=False).input_ids
    return ids


def tool_choice_of(req, api):
    """-> None (auto) | "none" | "any" | tool name, from an Anthropic or OpenAI request."""
    tc = req.get("tool_choice")
    if tc is None or not req.get("tools"):
        return None
    if api == "anthropic":
        kind = tc.get("type", "auto") if isinstance(tc, dict) else str(tc)
        return {"auto": None, "none": "none", "any": "any"}.get(kind, tc.get("name") if isinstance(tc, dict) else None)
    if isinstance(tc, str):
        return {"none": "none", "required": "any"}.get(tc)
    return (tc.get("function") or {}).get("name") or None


def stops_of(req):
    s = req.get("stop_sequences", req.get("stop"))
    if not s:
        return []
    return [s] if isinstance(s, str) else [str(x) for x in s if x]


# ---- request conversion
STRIP_PATTERNS = [re.compile(p) for p in [
    r"\s*<total_tokens>[^<]*</total_tokens>\s*",            # Claude Code's per-request token budget reminder
    r"x-anthropic-billing-header:[^\n]*\n?",                # Claude Code's per-session billing metadata (system role)
]]


def clean(text):
    for pat in STRIP_PATTERNS:
        text = pat.sub("", text)
    return text


def anthropic_to_messages(req):
    """Anthropic Messages request -> (chat-template messages, tools in OpenAI function form)."""
    msgs = []
    sysm = req.get("system")
    if sysm:
        text = sysm if isinstance(sysm, str) else "\n".join(b.get("text", "") for b in sysm if b.get("type") == "text")
        msgs.append({"role": "system", "content": clean(text)})
    for m in req.get("messages", []):
        c = m.get("content")
        if isinstance(c, str):
            c = clean(c)
            if c.strip() or m["role"] != "system":
                msgs.append({"role": m["role"], "content": c})
            continue
        if m["role"] == "system":
            text = clean("\n".join(b.get("text", "") for b in c if b.get("type") == "text"))
            if text.strip():
                msgs.append({"role": "system", "content": text})
            continue
        if m["role"] == "user":
            items, results = [], []
            for b in c:
                if b.get("type") == "text":
                    if clean(b["text"]).strip():
                        items.append({"type": "text", "text": clean(b["text"])})
                elif b.get("type") == "image":
                    items.append({"type": "image", "_img": _image_bytes(b)})
                elif b.get("type") == "tool_result":
                    rc = b.get("content", "")
                    if isinstance(rc, list):
                        parts = [{"type": "text", "text": x.get("text", "")} if x.get("type") == "text"
                                 else {"type": "image", "_img": _image_bytes(x)} for x in rc if x.get("type") in ("text", "image")]
                        rc = parts if any(p["type"] == "image" for p in parts) else "\n".join(p["text"] for p in parts)
                    results.append({"role": "tool", "content": rc if isinstance(rc, list) else str(rc),
                                    "tool_call_id": b.get("tool_use_id", "")})
            msgs.extend(results)
            if items:
                msgs.append({"role": "user", "content": _content(items)})
        else:
            texts, thinks, calls = [], [], []
            for b in c:
                if b.get("type") == "text":
                    texts.append(b["text"])
                elif b.get("type") == "thinking":
                    thinks.append(b.get("thinking", ""))
                elif b.get("type") == "tool_use":
                    calls.append({"id": b.get("id"), "type": "function",
                                  "function": {"name": b["name"], "arguments": b.get("input", {})}})
            am = {"role": "assistant", "content": "\n".join(texts)}
            if thinks:
                am["reasoning_content"] = "\n".join(thinks)
            if calls:
                am["tool_calls"] = calls
            msgs.append(am)
    tools = [{"type": "function", "function": {"name": t["name"], "description": t.get("description", ""),
                                               "parameters": t.get("input_schema", {"type": "object", "properties": {}})}}
             for t in req.get("tools", []) if t.get("name")]
    return msgs, tools


def openai_messages(req):
    msgs = []
    for m in req["messages"]:
        m = dict(m)
        if isinstance(m.get("content"), list):
            items = [{"type": "text", "text": p.get("text", "")} if p.get("type") == "text"
                     else {"type": "image", "_img": _image_bytes(p)} for p in m["content"] if p.get("type") in ("text", "image_url")]
            m["content"] = _content(items)
        if m.get("content") is None:
            m["content"] = ""
        for tc in m.get("tool_calls") or []:
            fn = tc.get("function", {})
            if isinstance(fn.get("arguments"), str):
                try:
                    fn["arguments"] = json.loads(fn["arguments"])
                except Exception:  # noqa: BLE001
                    fn["arguments"] = {"_raw": fn["arguments"]}
        msgs.append(m)
    return msgs, req.get("tools") or None


def _content(items):
    if any(it["type"] == "image" for it in items):
        return items
    return "\n".join(it["text"] for it in items)


def _images_of(msgs):
    out = []
    for m in msgs:
        if isinstance(m.get("content"), list):
            out += [it["_img"] for it in m["content"] if it.get("type") == "image"]
    return out


def render(msgs, tools, effort):
    """-> (signature ids, imgs): token ids with every <|image|> expanded to the image's token count, those tokens
    replaced by the image's negative signature id; imgs maps signature -> (n_tokens, embeddings)."""
    text = tok.apply_chat_template(msgs, tools=tools or None, tokenize=False, add_generation_prompt=True, reasoning_effort=effort)
    ids = tok(text, add_special_tokens=False).input_ids
    images = _images_of(msgs)
    if not images:
        return ids, {}
    imgs, out, n_img = {}, [], 0
    for t in ids:
        if t != T["<|image|>"]:
            out.append(t); continue
        if n_img >= len(images):
            raise ValueError("more <|image|> tokens than images")
        b = images[n_img]; n_img += 1
        v = _img_sig(b)
        imgs[v] = embed_image(b)
        out += [v] * imgs[v][0]
    if n_img != len(images):
        raise ValueError("fewer <|image|> tokens than images")
    return out, imgs


def think_budget_of(req):
    """Thinking tokens before `</think>` is forced: Anthropic `thinking.budget_tokens`, OpenAI-style `thinking_budget`
    or `reasoning.max_tokens`, else the server default; None = unlimited."""
    th = req.get("thinking") or {}
    b = th.get("budget_tokens") if th.get("type") == "enabled" else None
    if b is None:
        b = req.get("thinking_budget") or (req.get("reasoning") or {}).get("max_tokens")
    b = int(b or THINK_BUDGET_DEFAULT or 0)
    return b if b > 0 else None


def effort_of(req, default=DEFAULT_EFFORT):
    e = (req.get("chat_template_kwargs") or {}).get("reasoning_effort") or req.get("reasoning_effort")
    th = req.get("thinking") or {}
    if not e and th.get("type") == "enabled":
        e = "high" if int(th.get("budget_tokens", 0) or 0) >= 8192 else "low"
    if not e and th.get("type") == "adaptive":
        e = default
    if not e and th.get("type") == "disabled":
        e = "low"
    return e or default


# ---- HTTP
class H(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.0"      # one request per connection: SSE responses carry no framing for keep-alive

    def log_message(self, *a):
        pass

    def _json(self, code, obj, headers=()):
        body = json.dumps(obj).encode()
        self.send_response(code); self.send_header("Content-Type", "application/json")
        for k, v in headers:
            self.send_header(k, v)
        self.send_header("Content-Length", str(len(body))); self.end_headers(); self._raw(body)

    def _raw(self, b):
        try:
            self.wfile.write(b if isinstance(b, bytes) else b.encode()); self.wfile.flush()
        except (BrokenPipeError, ConnectionResetError):
            raise ClientGone() from None

    def _auth(self):
        if not API_KEY:
            return True
        h = self.headers.get("Authorization", ""); k = self.headers.get("x-api-key", "")
        return h == f"Bearer {API_KEY}" or k == API_KEY

    def _sse(self, obj, event=None):
        self._raw((f"event: {event}\n" if event else "") + f"data: {json.dumps(obj)}\n\n")

    def do_GET(self):
        if self.path.startswith("/health"):
            return self._json(200, {"status": "ok", "model": MODEL, "layers": cfg.n_layers, "max_len": eng.max_len, "max_streams": MAX_STREAMS,
                                    "max_sets": MAX_SETS, "max_queue": MAX_QUEUE, "piece": SCHED.piece, "buckets": list(eng.PREFILL_BUCKETS),
                                    "active": len(SCHED.active), "pending": len(SCHED.pending),
                                    "live_contexts": len(SCHED.live), "hbm_free_gb": SCHED.free_gb(), **STATE})
        if self.path.startswith("/v1/models"):
            return self._json(200, {"object": "list", "data": [{"id": MODEL, "object": "model"}]})
        self._json(404, {"error": "not found"})

    def do_POST(self):
        if not self._auth():
            return self._json(401, {"error": {"type": "authentication_error", "message": "bad api key"}})
        n = int(self.headers.get("Content-Length", 0))
        try:
            req = json.loads(self.rfile.read(n) or b"{}")
        except Exception as e:  # noqa: BLE001
            return self._json(400, {"error": str(e)})
        try:
            if self.path.startswith("/v1/messages/count_tokens"):
                msgs, tools = anthropic_to_messages(req)
                return self._json(200, {"input_tokens": len(render(msgs, tools, effort_of(req))[0])})
            if self.path.startswith("/v1/messages"):
                return self.anthropic(req)
            if self.path.startswith("/v1/chat/completions"):
                return self.openai_chat(req)
            if self.path.startswith("/v1/completions"):
                return self.openai_text(req)
        except Exception as e:  # noqa: BLE001
            if isinstance(e, ClientGone):
                return log(f"client disconnected: {getattr(self, 'rid', '?')} after {getattr(e, 'n', '?')} tokens (stream cancelled)")
            if isinstance(e, QueueFull):
                STATE["rejected"] = STATE.get("rejected", 0) + 1
                return self._json(429, {"error": {"type": "rate_limit_error", "message": str(e)}}, [("Retry-After", "5")])
            import traceback; log("request failed:", traceback.format_exc()[-1500:])
            code, kind = (503, "overloaded_error") if isinstance(e, TimeoutError) else \
                         (400, "invalid_request_error") if isinstance(e, ValueError) else (500, "api_error")
            try:
                return self._json(code, {"error": {"type": kind, "message": str(e)}})
            except Exception:  # noqa: BLE001
                return
        self._json(404, {"error": "not found"})

    # ---- OpenAI
    def openai_chat(self, req):
        msgs, tools = openai_messages(req)
        choice = tool_choice_of(req, "openai")
        ids, imgs = render(msgs, None if choice == "none" else tools, effort_of(req))
        max_new = int(req.get("max_tokens") or req.get("max_completion_tokens") or MAX_NEW_DEFAULT)
        temperature = float(req.get("temperature", DEFAULT_TEMP)); top_p = float(req.get("top_p", DEFAULT_TOP_P))
        rid = self.rid = "chatcmpl-" + uuid.uuid4().hex[:12]
        STATE["requests"] += 1
        kw = dict(tools=tools, stops=stops_of(req), prefix=forced_prefix(choice), think_budget=think_budget_of(req))
        finish = lambda r: "tool_calls" if r["calls"] else ("length" if r["stop"] == "length" else "stop")
        if req.get("stream"):
            self.send_response(200); self.send_header("Content-Type", "text/event-stream")
            self.send_header("Cache-Control", "no-cache"); self.end_headers()
            nt = [0]

            def chunk(delta, finish=None):
                self._sse({"id": rid, "object": "chat.completion.chunk", "model": MODEL,
                           "choices": [{"index": 0, "delta": delta, "finish_reason": finish}]})

            def on_event(kind, val):
                if kind == "ping":
                    self._raw(": keep-alive\n\n")               # an SSE comment: OpenAI clients skip it
                elif kind == "thinking":
                    chunk({"reasoning_content": val})
                elif kind == "text":
                    chunk({"content": val})
                else:
                    chunk({"tool_calls": [{"index": nt[0], "id": val["id"], "type": "function",
                                           "function": {"name": val["name"], "arguments": json.dumps(val["input"])}}]})
                    nt[0] += 1
            r = run_request(ids, max_new, temperature, top_p, imgs, rid, on_event=on_event, **kw)
            chunk({}, finish(r))
            self._raw(b"data: [DONE]\n\n")
        else:
            r = run_request(ids, max_new, temperature, top_p, imgs, rid, **kw)
            msg = {"role": "assistant", "content": r["text"]}
            if r["reasoning"]:
                msg["reasoning_content"] = r["reasoning"]
            if r["calls"]:
                msg["tool_calls"] = [{"id": c["id"], "type": "function",
                                      "function": {"name": c["name"], "arguments": json.dumps(c["input"])}} for c in r["calls"]]
            self._json(200, {"id": rid, "object": "chat.completion", "model": MODEL,
                             "choices": [{"index": 0, "message": msg, "finish_reason": finish(r)}],
                             "usage": {"prompt_tokens": len(ids), "completion_tokens": r["out_len"],
                                       "prefill_s": round(r["tp"], 2), "decode_tok_s": round(r["out_len"] / max(r["tg"], 1e-6), 2)}})
        self._done(rid, ids, r)

    def openai_text(self, req):
        ids = tok(req.get("prompt", ""), add_special_tokens=False).input_ids
        max_new = int(req.get("max_tokens") or MAX_NEW_DEFAULT)
        temperature = float(req.get("temperature", DEFAULT_TEMP)); top_p = float(req.get("top_p", DEFAULT_TOP_P))
        rid = self.rid = "cmpl-" + uuid.uuid4().hex[:12]
        STATE["requests"] += 1
        r = run_request(ids, max_new, temperature, top_p, None, rid, None, stops=stops_of(req), raw=True)
        self._json(200, {"id": rid, "object": "text_completion", "model": MODEL,
                         "choices": [{"index": 0, "text": r["text"], "finish_reason": "length" if r["stop"] == "length" else "stop"}],
                         "usage": {"prompt_tokens": len(ids), "completion_tokens": r["out_len"]}})
        self._done(rid, ids, r)

    # ---- Anthropic
    def anthropic(self, req):
        msgs, tools = anthropic_to_messages(req)
        choice = tool_choice_of(req, "anthropic")
        ids, imgs = render(msgs, None if choice == "none" else tools, effort_of(req))
        max_new = int(req.get("max_tokens") or MAX_NEW_DEFAULT)
        temperature = float(req.get("temperature", DEFAULT_TEMP)); top_p = float(req.get("top_p", DEFAULT_TOP_P))
        rid = self.rid = "msg_" + uuid.uuid4().hex[:16]
        STATE["requests"] += 1
        kw = dict(tools=tools, stops=stops_of(req), prefix=forced_prefix(choice), think_budget=think_budget_of(req))
        stop_of = lambda r: "tool_use" if r["calls"] else {"length": "max_tokens", "stop_sequence": "stop_sequence"}.get(r["stop"], "end_turn")
        if req.get("stream"):
            self.send_response(200); self.send_header("Content-Type", "text/event-stream")
            self.send_header("Cache-Control", "no-cache"); self.end_headers()
            self._sse({"type": "message_start", "message": {"id": rid, "type": "message", "role": "assistant", "model": MODEL,
                                                             "content": [], "stop_reason": None, "stop_sequence": None,
                                                             "usage": {"input_tokens": len(ids), "output_tokens": 0}}}, "message_start")
            st = {"idx": -1, "open": None}

            def open_block(kind):
                st["idx"] += 1; st["open"] = kind
                blk = {"type": "thinking", "thinking": ""} if kind == "thinking" else {"type": "text", "text": ""}
                self._sse({"type": "content_block_start", "index": st["idx"], "content_block": blk}, "content_block_start")

            def close_block():
                if st["open"] is not None:
                    self._sse({"type": "content_block_stop", "index": st["idx"]}, "content_block_stop"); st["open"] = None

            def on_event(kind, val):
                if kind == "ping":
                    self._sse({"type": "ping"}, "ping")            # the Anthropic API's own keep-alive event
                elif kind in ("thinking", "text"):
                    if st["open"] != kind:
                        close_block(); open_block(kind)
                    d = {"type": "thinking_delta", "thinking": val} if kind == "thinking" else {"type": "text_delta", "text": val}
                    self._sse({"type": "content_block_delta", "index": st["idx"], "delta": d}, "content_block_delta")
                else:
                    close_block(); st["idx"] += 1
                    self._sse({"type": "content_block_start", "index": st["idx"],
                               "content_block": {"type": "tool_use", "id": val["id"], "name": val["name"], "input": {}}}, "content_block_start")
                    self._sse({"type": "content_block_delta", "index": st["idx"],
                               "delta": {"type": "input_json_delta", "partial_json": json.dumps(val["input"])}}, "content_block_delta")
                    self._sse({"type": "content_block_stop", "index": st["idx"]}, "content_block_stop")
            r = run_request(ids, max_new, temperature, top_p, imgs, rid, on_event=on_event, **kw)
            close_block()
            self._sse({"type": "message_delta", "delta": {"stop_reason": stop_of(r), "stop_sequence": r["stop_sequence"]},
                       "usage": {"output_tokens": r["out_len"]}}, "message_delta")
            self._sse({"type": "message_stop"}, "message_stop")
        else:
            r = run_request(ids, max_new, temperature, top_p, imgs, rid, **kw)
            content = []
            if r["reasoning"]:
                content.append({"type": "thinking", "thinking": r["reasoning"], "signature": ""})
            if r["text"].strip() or not r["calls"]:
                content.append({"type": "text", "text": r["text"]})
            content += [{"type": "tool_use", "id": c["id"], "name": c["name"], "input": c["input"]} for c in r["calls"]]
            self._json(200, {"id": rid, "type": "message", "role": "assistant", "model": MODEL, "content": content,
                             "stop_reason": stop_of(r), "stop_sequence": r["stop_sequence"],
                             "usage": {"input_tokens": len(ids), "output_tokens": r["out_len"]}})
        self._done(rid, ids, r)

    def _done(self, rid, ids, r):
        log(f"req {rid}: {len(ids)} prompt tok ({r['reused']} reused), {r['out_len']} new, prefill {r['tp']:.2f}s, "
            f"{r['out_len'] / max(r['tg'], 1e-6):.1f} tok/s, {r['stop']}")


# ----------------------------------------------------------------------------- 5. warm-up, serve, tunnel
banner(4, "Warm-up", "prefill buckets, batched decode programs, snapshot programs")
t_warm = time.time()
for b in [b for b in eng.PREFILL_BUCKETS if b <= eng.prefill_piece]:
    t = time.time()
    seq = np.concatenate([ids] * (b // ids.shape[1] + 1), 1)[:, :b]
    lg, cc, _ = eng.prefill(seq); jax.block_until_ready(lg); del cc
    log(f"   prefill bucket {b}: {time.time() - t:.0f}s")
    publish("compiling", what=f"prefill {b}", secs=int(time.time() - t))
for B in range(1, MAX_STREAMS + 1):
    t = time.time()
    _sets = [eng.alloc_caches(1) for _ in range(B)]
    _samp = DeviceSampler([1.0] * B, [0.95] * B, seed=0)
    _ids, _lg, _sets, _pos = eng.decode_rows(np.full((B,), 0, np.int32), _sets, [0] * B, _samp)
    jax.block_until_ready(_ids); del _sets, _lg, _ids, _pos
    log(f"   batched decode B={B}: {time.time() - t:.0f}s")
    publish("compiling", what=f"decode B={B}", secs=int(time.time() - t))
n_shard = max(eng.lcfg.seq_shard, 1)
for n_tok in range(SNAP_ROWS * n_shard, SNAP_WARM + 1, SNAP_ROWS * n_shard):
    cc = eng.alloc_caches(1)
    hs = eng.snapshot_to_host(eng.snapshot_prefix(cc, n_tok, rows_bucket=SNAP_ROWS)); del cc
    cc = eng.restore_prefix(eng.snapshot_from_host(hs)); jax.block_until_ready(cc[0]["state"] if "state" in cc[0] else cc[0]["c"]); del cc, hs
use, lim = hbm()
log(f"   warm-up done in {(time.time() - t_warm) / 60:.1f} min; HBM chip0 {use:.2f} / {lim:.2f} GB")
if CACHE_DIR.exists():
    n_files = sum(1 for _ in CACHE_DIR.iterdir()); size = sum(f.stat().st_size for f in CACHE_DIR.iterdir()) / 1e9
    log(f"   compile cache: {n_files} entries, {size:.2f} GB in {CACHE_DIR}")
publish("warmed", minutes=round((time.time() - t_warm) / 60, 1), hbm_gb=round(use, 2))

SCHED.start()
srv = ThreadingHTTPServer(("0.0.0.0", PORT), H)
threading.Thread(target=srv.serve_forever, daemon=True).start()
log(f"   HTTP server on :{PORT}")

banner(5, "Tunnel", "a public cloudflared URL")
url = None
TUN = [None]
TUN_ATTEMPTS = []


def stop_tunnel():
    """Best-effort cleanup for the active cloudflared child process."""
    proc = TUN[0]
    TUN[0] = None
    if proc is None or proc.poll() is not None:
        return
    proc.terminate()
    try:
        proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        proc.kill()
        proc.wait(timeout=5)


def start_tunnel(attempts=3, wait_s=60):
    """Start a quick tunnel, trying protocols that work in restricted Kaggle networks.

    A registration attempt that prints no URL within ``wait_s`` is terminated
    before the next protocol/attempt is tried.
    """
    pat = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com(?:/[^\s]*)?", re.I)
    protocols = (None, "http2", "quic")
    for i in range(attempts):
        for protocol in protocols:
            cmd = [str(CLOUDFLARED), "tunnel", "--url", f"http://localhost:{PORT}",
                   "--no-autoupdate"]
            if protocol:
                cmd += ["--protocol", protocol]
            label = protocol or "default"
            lines = []
            log(f"   starting cloudflared ({label}): {' '.join(cmd)}")
            try:
                tun = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                                       stderr=subprocess.STDOUT, text=True)
            except OSError as e:
                TUN_ATTEMPTS.append({"protocol": label, "returncode": None,
                                     "error": f"{type(e).__name__}: {e}"})
                log(f"   cloudflared ({label}) could not start: {e}")
                continue
            q = queue.Queue()
            threading.Thread(
                target=lambda proc=tun, output=q, captured=lines: [
                    (captured.append(line.rstrip()), output.put(line))
                    for line in iter(proc.stdout.readline, "")
                ],
                daemon=True,
            ).start()
            t0 = time.time()
            while time.time() - t0 < wait_s:
                try:
                    line = q.get(timeout=5)
                except queue.Empty:
                    if tun.poll() is not None:
                        break
                    continue
                m = pat.search(line)
                if m:
                    TUN[0] = tun
                    TUN_ATTEMPTS.append({"protocol": label, "returncode": tun.poll(),
                                         "output_tail": lines[-8:]})
                    return m.group(0)
            if tun.poll() is None:
                tun.terminate()
                try:
                    tun.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    tun.kill()
                    tun.wait(timeout=5)
            TUN_ATTEMPTS.append({"protocol": label, "returncode": tun.returncode,
                                 "output_tail": lines[-12:]})
            if lines:
                log(f"   cloudflared ({label}) output:")
                for line in lines[-12:]:
                    log(f"     {line}")
            log(f"   tunnel attempt {i + 1}/{attempts} ({label}): "
                f"no URL within {wait_s} s (cloudflared rc {tun.returncode}); retrying")
    return None


def tunnel_probe(u, wait_s=180):
    """Background: confirm the public URL answers (a fresh hostname can take a minute to resolve); if it never does,
    open a new tunnel once and announce the new URL."""
    t0 = time.time()
    while time.time() - t0 < wait_s:
        try:
            urllib.request.urlopen(f"{u}/health", timeout=10).read()
            log(f"   tunnel reachable from outside: {u} ({time.time() - t0:.0f} s after it opened)")
            return
        except Exception:  # noqa: BLE001
            time.sleep(10)
    log(f"   tunnel URL {u} did not answer in {wait_s} s: opening a new one")
    stop_tunnel()
    new = start_tunnel()
    if new:
        global url
        url = new
        STATE["url"] = new
        publish("tunnel-url", endpoint=new)
        log(f"#  NEW ENDPOINT: {new}   (the earlier URL never resolved; the API key is unchanged)")
    else:
        publish("tunnel-failed", note="server still reachable inside the kernel on :8000",
                attempts=TUN_ATTEMPTS)


if CFG["tunnel"]:
    url = start_tunnel()
    if url:
        publish("tunnel-url", endpoint=url)
        threading.Thread(target=tunnel_probe, args=(url,), daemon=True).start()
    else:
        publish("tunnel-failed", note="server still reachable inside the kernel on :8000",
                attempts=TUN_ATTEMPTS)
# Do not advertise the loopback fallback as a public endpoint. The launcher
# treats a null endpoint as a tunnel failure and reports it explicitly.
STATE["url"] = url

# ----------------------------------------------------------------------------- 6. ready, self-test, keep alive
banner(6, "Ready", "self-test, then serving")


def _post(path, body):
    hdr = {"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"}
    r = urllib.request.urlopen(urllib.request.Request(f"http://127.0.0.1:{PORT}{path}", data=json.dumps(body).encode(), headers=hdr), timeout=900)
    return json.loads(r.read().decode())


try:
    r = _post("/v1/chat/completions", {"messages": [{"role": "user", "content": "Say hi in five words."}], "max_tokens": 48, "temperature": 0})
    log("   self-test openai:", json.dumps(r["choices"][0]["message"]["content"])[:120], r["usage"])
    tools = [{"name": "read_file", "description": "Read a text file from disk.",
              "input_schema": {"type": "object", "properties": {"path": {"type": "string"}, "max_lines": {"type": "integer"}}, "required": ["path"]}}]
    msgs = [{"role": "user", "content": "Use the read_file tool to read /etc/hostname (at most 5 lines)."}]
    r = _post("/v1/messages", {"model": MODEL, "max_tokens": 256, "messages": msgs, "tools": tools, "temperature": 0})
    log("   self-test anthropic:", json.dumps(r["content"])[:200], r["stop_reason"])
    outs = [None, None]

    def _one(i, prompt):
        outs[i] = _post("/v1/chat/completions", {"messages": [{"role": "user", "content": prompt}], "max_tokens": 48, "temperature": 0})
    ths = [threading.Thread(target=_one, args=(0, "Count from one to ten in words.")), threading.Thread(target=_one, args=(1, "Name three primary colors."))]
    t0 = time.time(); [t.start() for t in ths]; [t.join() for t in ths]
    log("   self-test concurrent:", [o["choices"][0]["message"]["content"][:50] for o in outs], f"{time.time() - t0:.1f}s")
except Exception as e:  # noqa: BLE001
    import traceback; log("   self-test failed:", traceback.format_exc()[-800:])

endpoint = STATE["url"]
local_endpoint = f"http://127.0.0.1:{PORT}"
log("")
log("#" * 70)
log(f"#  READY — the server is live ({elapsed()} after start)")
log(f"#  ENDPOINT : {endpoint or local_endpoint + ' (public tunnel failed)'}   "
    "(OpenAI at /v1, Anthropic at /v1/messages)")
log(f"#  API KEY  : {API_KEY}")
log(f"#  MODEL    : {MODEL}   (context {CFG['max_len']}, up to {MAX_STREAMS} streams)")
log("#" * 70)
if endpoint:
    log("#  Claude Code:")
    log(f"#    ANTHROPIC_BASE_URL={endpoint} ANTHROPIC_AUTH_TOKEN={API_KEY} ANTHROPIC_MODEL={MODEL} \\")
    log(f"#    ANTHROPIC_SMALL_FAST_MODEL={MODEL} CLAUDE_CODE_MAX_CONTEXT_TOKENS={CFG['max_len']} claude")
    log("#  OpenAI-compatible clients: base URL " + endpoint + "/v1, model " + MODEL)
else:
    log("#  PUBLIC URL UNAVAILABLE — cloudflared tunnel failed")
log(f"#  Serving for up to {CFG['keepalive_min']} min, then this cell exits on its own.")
log("#" * 70)
publish("ready", endpoint=endpoint, api_key=API_KEY, model=MODEL, max_model_len=CFG["max_len"],
        keepalive_min=CFG["keepalive_min"], startup_secs=int(time.time() - T0))

if globals().get("SERVE_FOREVER", True):
    t_serve = time.time()
    while time.time() - t_serve < CFG["keepalive_min"] * 60:
        time.sleep(120)
        if SCHED.thread is not None and not SCHED.thread.is_alive():
            publish("stopped", reason="scheduler-exit")
            stop_tunnel()
            srv.shutdown()
            sys.exit(1)
        up = int((time.time() - t_serve) / 60)
        if up % 10 < 2:
            publish("heartbeat", up_min=up, endpoint=STATE["url"], requests=STATE["requests"], tokens=STATE["tokens"])
    publish("auto-shutdown", served_min=CFG["keepalive_min"])
    stop_tunnel()
    SCHED.stop(); srv.shutdown()
    sys.exit(0)


### Launch (leave this cell running: it is the server)
What you will see, step by step:
1. **Runtime** — a pinned TPU runtime and a few packages (~1 min)
2. **Weights** — the 3-bit experts and the int8 non-expert weights go straight onto the eight chips (~4 min)
3. **Vision** — the vision tower, sharded over the chips (~20 s)
4. **Warm-up** — the prefill and batched-decode programs are loaded from the compile cache and traced (~9 min; ~14 cold)
5. **Tunnel** — a public URL
6. **READY banner** with `ENDPOINT`, `API KEY` and `MODEL`, then a short self-test

Use the endpoint from anywhere:
```bash
curl <ENDPOINT>/v1/chat/completions -H "Authorization: Bearer <API_KEY>" \
  -H "Content-Type: application/json" -d '{
    "model": "glm-5.3-flash",
    "messages": [{"role": "user", "content": "Hello!"}]
  }'
```
Claude Code (the banner prints this line filled in):
```bash
ANTHROPIC_BASE_URL=<ENDPOINT> ANTHROPIC_AUTH_TOKEN=<API_KEY> ANTHROPIC_MODEL=glm-5.3-flash \
ANTHROPIC_SMALL_FAST_MODEL=glm-5.3-flash CLAUDE_CODE_MAX_CONTEXT_TOKENS=262144 claude
```


In [ ]:
!python serve_glm53.py
